# Zero-Shot Fault Diagnosis via Domain Adaptation -- Kaggle test run (CPU, offline)

Self-contained snapshot of `src/domain_adaptation/` (plus the specific `vibration_ad/` modules it imports directly -- regenerated by `scripts/build_kaggle_notebook_domain_adaptation.py`, not live-synced). Third, independent experiment in this project -- see `/DOMAIN_ADAPTATION_PLAN.md` for the full plan and its 'Revisions & Decisions' section for what changed from the original proposal and why.

Trains a supervised source-only baseline (Phase 2), then a DANN model (Phase 3: domain discriminator + gradient reversal + a healthy-anchor alignment loss), then evaluates both on a target operating condition's REAL bearing faults that neither model ever saw during training -- the zero-shot check. No Internet needed -- both `esraakhaled299/cwru-data` and `dippatel03/paderborn-db` are attached input datasets, readable regardless of the kernel's Internet setting (this account isn't phone-verified, see vibration_ad's `notes/07_dvc_and_kaggle.md`). CPU only.

## 1. Reconstruct the `vibration_ad` + `domain_adaptation` packages + config on disk

In [ ]:
import base64, pathlib

WORKDIR = pathlib.Path('/kaggle/working')
SRC_FILES = {
    'src/vibration_ad/__init__.py': (
        'IiIiVW5zdXBlcnZpc2VkIHZpYnJhdGlvbiBhbm9tYWx5IGRldGVjdGlvbiBvbiB0aGUgQ1dSVSBiZWFyaW5nIGRhdGFzZXQuIiIiCgpfX3ZlcnNpb25fXyA9ICIwLjEuMCIK'
    ),
    'src/vibration_ad/config.py': (
        'IiIiTG9hZCB0aGUgWUFNTCBjb25maWcgaW50byBhIHNtYWxsIHRyZWUgb2YgZGF0YWNsYXNzZXMuCgpFdmVyeXRoaW5nIGRvd25zdHJlYW0gKGRhdGEgcHJlcCwgdHJhaW5pbmcsIGV2YWx1YXRpb24sIGluZmVyZW5jZSkgdGFrZXMKb25lIG9mIHRoZXNlIGBDb25maWdgIG9iamVjdHMgaW5zdGVhZCBvZiBzY2F0dGVyZWQgQ0xJIGZsYWdzLCBzbyBhIHJ1biBpcwpmdWxseSBkZXNjcmliZWQgYnkgYSBzaW5nbGUgWUFNTCBmaWxlIChzZWUgY29uZmlncy9kZWZhdWx0LnlhbWwpLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB5YW1sCgoKQGRhdGFjbGFzcwpjbGFzcyBQYXRoc0NvbmZpZzoKICAgIHJhd19kaXI6IHN0cgogICAgcHJvY2Vzc2VkX2Rpcjogc3RyCiAgICBtb2RlbHNfZGlyOiBzdHIKICAgIGZpZ3VyZXNfZGlyOiBzdHIKICAgIHJlcG9ydHNfZGlyOiBzdHIKCgpAZGF0YWNsYXNzCmNsYXNzIERhdGFDb25maWc6CiAgICBzZW5zb3I6IHN0cgogICAgZnNfaHo6IGludAogICAgdHJhaW5fcnBtczogbGlzdFtpbnRdCiAgICBob2xkb3V0X3JwbTogaW50CiAgICBmYXVsdF90eXBlczogbGlzdFtzdHJdCiAgICBmYXVsdF9ycG1zOiBsaXN0W2ludF0KCgpAZGF0YWNsYXNzCmNsYXNzIFdpbmRvd2luZ0NvbmZpZzoKICAgIHNlZ21lbnRfbGVuZ3RoOiBpbnQKICAgIHRyYWluX3N0cmlkZTogaW50CiAgICBldmFsX3N0cmlkZTogaW50CgoKQGRhdGFjbGFzcwpjbGFzcyBTVEZUQ29uZmlnOgogICAgbnBlcnNlZzogaW50CiAgICBob3BfbGVuZ3RoOiBpbnQKICAgIGxvZ19lcHNpbG9uOiBmbG9hdAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG5vdmVybGFwKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5ucGVyc2VnIC0gc2VsZi5ob3BfbGVuZ3RoCgoKQGRhdGFjbGFzcwpjbGFzcyBNb2RlbENvbmZpZzoKICAgIGxhdGVudF9kaW06IGludAogICAgYmFzZV9jaGFubmVsczogaW50CgoKQGRhdGFjbGFzcwpjbGFzcyBUcmFpbkNvbmZpZzoKICAgIGJhdGNoX3NpemU6IGludAogICAgZXBvY2hzOiBpbnQKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0CiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0CiAgICBlYXJseV9zdG9wcGluZ19wYXRpZW5jZTogaW50CiAgICB2YWxfZnJhY3Rpb25fb2ZfaG9sZG91dDogZmxvYXQKICAgIGRldmljZTogc3RyCgoKQGRhdGFjbGFzcwpjbGFzcyBFdmFsdWF0aW9uQ29uZmlnOgogICAgdGhyZXNob2xkX3BlcmNlbnRpbGU6IGZsb2F0CgoKQGRhdGFjbGFzcwpjbGFzcyBDb25maWc6CiAgICBzZWVkOiBpbnQKICAgIHBhdGhzOiBQYXRoc0NvbmZpZwogICAgZGF0YTogRGF0YUNvbmZpZwogICAgd2luZG93aW5nOiBXaW5kb3dpbmdDb25maWcKICAgIHN0ZnQ6IFNURlRDb25maWcKICAgIG1vZGVsOiBNb2RlbENvbmZpZwogICAgdHJhaW46IFRyYWluQ29uZmlnCiAgICBldmFsdWF0aW9uOiBFdmFsdWF0aW9uQ29uZmlnCiAgICAjIERpcmVjdG9yeSB0aGUgY29uZmlnIGZpbGUgbGl2ZXMgaW47IHJlbGF0aXZlIHBhdGhzIGluIGBwYXRoc2AgYXJlCiAgICAjIHJlc29sdmVkIGFnYWluc3QgdGhlIHByb2plY3Qgcm9vdCAocGFyZW50IG9mIGNvbmZpZ3MvKSwgbm90IGFnYWluc3QgY3dkLgogICAgcHJvamVjdF9yb290OiBQYXRoID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PVBhdGguY3dkKQoKICAgIGRlZiByZXNvbHZlKHNlbGYsIHJlbGF0aXZlX3BhdGg6IHN0cikgLT4gUGF0aDoKICAgICAgICByZXR1cm4gKHNlbGYucHJvamVjdF9yb290IC8gcmVsYXRpdmVfcGF0aCkucmVzb2x2ZSgpCgoKZGVmIGxvYWRfY29uZmlnKGNvbmZpZ19wYXRoOiBzdHIgfCBQYXRoKSAtPiBDb25maWc6CiAgICBjb25maWdfcGF0aCA9IFBhdGgoY29uZmlnX3BhdGgpLnJlc29sdmUoKQogICAgd2l0aCBvcGVuKGNvbmZpZ19wYXRoKSBhcyBmOgogICAgICAgIHJhdyA9IHlhbWwuc2FmZV9sb2FkKGYpCgogICAgcHJvamVjdF9yb290ID0gY29uZmlnX3BhdGgucGFyZW50LnBhcmVudCAgIyBjb25maWdzL2RlZmF1bHQueWFtbCAtPiBwcm9qZWN0IHJvb3QKCiAgICByZXR1cm4gQ29uZmlnKAogICAgICAgIHNlZWQ9cmF3WyJzZWVkIl0sCiAgICAgICAgcGF0aHM9UGF0aHNDb25maWcoKipyYXdbInBhdGhzIl0pLAogICAgICAgIGRhdGE9RGF0YUNvbmZpZygqKnJhd1siZGF0YSJdKSwKICAgICAgICB3aW5kb3dpbmc9V2luZG93aW5nQ29uZmlnKCoqcmF3WyJ3aW5kb3dpbmciXSksCiAgICAgICAgc3RmdD1TVEZUQ29uZmlnKCoqcmF3WyJzdGZ0Il0pLAogICAgICAgIG1vZGVsPU1vZGVsQ29uZmlnKCoqcmF3WyJtb2RlbCJdKSwKICAgICAgICB0cmFpbj1UcmFpbkNvbmZpZygqKnJhd1sidHJhaW4iXSksCiAgICAgICAgZXZhbHVhdGlvbj1FdmFsdWF0aW9uQ29uZmlnKCoqcmF3WyJldmFsdWF0aW9uIl0pLAogICAgICAgIHByb2plY3Rfcm9vdD1wcm9qZWN0X3Jvb3QsCiAgICApCg=='
    ),
    'src/vibration_ad/data/__init__.py': (
        ''
    ),
    'src/vibration_ad/data/catalog.py': (
        'IiIiUGFyc2UgQ1dSVSAubnB6IGZpbGVuYW1lcyBpbnRvIHN0cnVjdHVyZWQgbWV0YWRhdGEuCgpDV1JVX0JlYXJpbmdfTnVtUHkgbmFtaW5nIGNvbnZlbnRpb24gKHNlZSBkYXRhL1JFQURNRS5tZCk6CiAgICB7cnBtfV9Ob3JtYWwubnB6CiAgICB7cnBtfV97ZmF1bHR9X3tzaXplX21pbHN9X3tlbmR9Lm5wegoKYGZhdWx0YCBpcyBvbmUgb2YgSVIgKGlubmVyIHJhY2UpLCBCIChiYWxsKSwgT1JAMyAvIE9SQDYgLyBPUkAxMiAob3V0ZXIKcmFjZSwgY2xvY2sgcG9zaXRpb24gcmVsYXRpdmUgdG8gdGhlIGxvYWQgem9uZSkuIGBzaXplX21pbHNgIGlzIHRoZSBmYXVsdApkaWFtZXRlciBpbiB0aG91c2FuZHRocyBvZiBhbiBpbmNoLiBgZW5kYCBpcyBERTEyL0RFNDgvRkU7IHdlIG9ubHkgdXNlCkRFMTIgKDEyIGtIeiBEcml2ZS1FbmQpIHNvIGl0J3MgY29uc3RhbnQgYWNyb3NzIG91ciByYXcgZmlsZXMuCgpOb3RlOiB0aGVzZSBmYXVsdCBsYWJlbHMgYXJlIE9OTFkgZXZlciB1c2VkIGZvciBldmFsdWF0aW9uIGluIHRoaXMgcHJvamVjdAooY29tcHV0aW5nIFJPQy1BVUMsIGNvbmZ1c2lvbiBtYXRyaWNlcywgcGVyLXNldmVyaXR5IGRldGVjdGlvbiByYXRlKS4gVGhlCmF1dG9lbmNvZGVyIGl0c2VsZiBpcyB0cmFpbmVkIGV4Y2x1c2l2ZWx5IG9uIHdpbmRvd3MgZnJvbSBOb3JtYWwubnB6IGZpbGVzCmFuZCBuZXZlciBzZWVzIGEgbGFiZWwuIFNlZSBub3Rlcy8wNF9ldmFsdWF0aW9uX25vdGVzLm1kLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKTk9STUFMX1JFID0gcmUuY29tcGlsZShyIl4oP1A8cnBtPlxkKylfTm9ybWFsXC5ucHokIikKRkFVTFRfUkUgPSByZS5jb21waWxlKAogICAgciJeKD9QPHJwbT5cZCspXyg/UDxmYXVsdD5JUnxCfE9SQDN8T1JANnxPUkAxMilfKD9QPHNpemU+XGQrKV8oP1A8ZW5kPkRFXGQrfEZFKVwubnB6JCIKKQoKCmRlZiBwYXJzZV9maWxlbmFtZShmaWxlbmFtZTogc3RyKSAtPiBkaWN0OgogICAgIiIiUmV0dXJuIG1ldGFkYXRhIGZvciBvbmUgQ1dSVSBucHogZmlsZW5hbWUsIG9yIHJhaXNlIFZhbHVlRXJyb3IuIiIiCiAgICBtID0gTk9STUFMX1JFLm1hdGNoKGZpbGVuYW1lKQogICAgaWYgbToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAicnBtIjogaW50KG0uZ3JvdXAoInJwbSIpKSwKICAgICAgICAgICAgImZhdWx0X3R5cGUiOiAiTm9ybWFsIiwKICAgICAgICAgICAgImZhdWx0X3NpemVfbWlscyI6IDAsCiAgICAgICAgICAgICJlbmQiOiBOb25lLAogICAgICAgICAgICAibGFiZWwiOiAwLAogICAgICAgIH0KCiAgICBtID0gRkFVTFRfUkUubWF0Y2goZmlsZW5hbWUpCiAgICBpZiBtOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJycG0iOiBpbnQobS5ncm91cCgicnBtIikpLAogICAgICAgICAgICAiZmF1bHRfdHlwZSI6IG0uZ3JvdXAoImZhdWx0IiksCiAgICAgICAgICAgICJmYXVsdF9zaXplX21pbHMiOiBpbnQobS5ncm91cCgic2l6ZSIpKSwKICAgICAgICAgICAgImVuZCI6IG0uZ3JvdXAoImVuZCIpLAogICAgICAgICAgICAibGFiZWwiOiAxLAogICAgICAgIH0KCiAgICByYWlzZSBWYWx1ZUVycm9yKGYiRmlsZW5hbWUgZG9lc24ndCBtYXRjaCBDV1JVIG5hbWluZyBjb252ZW50aW9uOiB7ZmlsZW5hbWV9IikKCgpkZWYgYnVpbGRfY2F0YWxvZyhyYXdfZGlyOiBQYXRoKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJTY2FuIHJhd19kaXIgZm9yIC5ucHogZmlsZXMgYW5kIHJldHVybiBvbmUgbWV0YWRhdGEgcm93IHBlciBmaWxlLiIiIgogICAgcm93cyA9IFtdCiAgICBmb3IgcGF0aCBpbiBzb3J0ZWQocmF3X2Rpci5nbG9iKCIqLm5weiIpKToKICAgICAgICBtZXRhID0gcGFyc2VfZmlsZW5hbWUocGF0aC5uYW1lKQogICAgICAgIG1ldGFbImZpbGVuYW1lIl0gPSBwYXRoLm5hbWUKICAgICAgICBtZXRhWyJmaWxlcGF0aCJdID0gc3RyKHBhdGgpCiAgICAgICAgcm93cy5hcHBlbmQobWV0YSkKCiAgICBpZiBub3Qgcm93czoKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJObyAubnB6IGZpbGVzIGZvdW5kIGluIHtyYXdfZGlyfS4gUnVuICIKICAgICAgICAgICAgImBweXRob24gLW0gdmlicmF0aW9uX2FkLmRhdGEuZG93bmxvYWRgIGZpcnN0LiIKICAgICAgICApCgogICAgZGYgPSBwZC5EYXRhRnJhbWUocm93cykKICAgIGNvbHMgPSBbImZpbGVuYW1lIiwgImZpbGVwYXRoIiwgInJwbSIsICJmYXVsdF90eXBlIiwgImZhdWx0X3NpemVfbWlscyIsICJlbmQiLCAibGFiZWwiXQogICAgcmV0dXJuIGRmW2NvbHNdLnNvcnRfdmFsdWVzKFsicnBtIiwgImZhdWx0X3R5cGUiLCAiZmF1bHRfc2l6ZV9taWxzIl0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgc2F2ZV9jYXRhbG9nKGRmOiBwZC5EYXRhRnJhbWUsIHByb2Nlc3NlZF9kaXI6IFBhdGgpIC0+IFBhdGg6CiAgICBwcm9jZXNzZWRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dF9wYXRoID0gcHJvY2Vzc2VkX2RpciAvICJjYXRhbG9nLmNzdiIKICAgIGRmLnRvX2NzdihvdXRfcGF0aCwgaW5kZXg9RmFsc2UpCiAgICByZXR1cm4gb3V0X3BhdGgK'
    ),
    'src/vibration_ad/data/dataset.py': (
        'IiIiQnVpbGQgdGhlIHRyYWluL3ZhbC90ZXN0IHNwZWN0cm9ncmFtIGFycmF5cyB1c2VkIGJ5IGV2ZXJ5IG90aGVyIHN0YWdlLgoKU3BsaXQgc3RyYXRlZ3kgKHNlZSBub3Rlcy8wMV9kYXRhc2V0X25vdGVzLm1kIGFuZCBub3Rlcy8wNF9ldmFsdWF0aW9uX25vdGVzLm1kCmZvciB0aGUgZnVsbCByZWFzb25pbmcpOgoKICBUUkFJTiAoaGVhbHRoeSBvbmx5LCBkZW5zZSB3aW5kb3cgb3ZlcmxhcCk6CiAgICBOb3JtYWwubnB6IHdpbmRvd3MgZnJvbSBjb25maWcuZGF0YS50cmFpbl9ycG1zIChkZWZhdWx0OiAxNzMwLzE3NTAvMTc3MikuCgogIFZBTCAoaGVhbHRoeSBvbmx5LCBzcGFyc2Ugb3ZlcmxhcCk6CiAgICBmaXJzdCBoYWxmIG9mIE5vcm1hbC5ucHogd2luZG93cyBmcm9tIGNvbmZpZy5kYXRhLmhvbGRvdXRfcnBtICgxNzk3KS4KICAgIFVzZWQgZm9yIGVhcmx5IHN0b3BwaW5nIGFuZCBmb3IgcGlja2luZyB0aGUgYW5vbWFseSB0aHJlc2hvbGQuCgogIFRFU1QgKGhlYWx0aHkgKyBmYXVsdHksIHNwYXJzZSBvdmVybGFwKToKICAgIHNlY29uZCBoYWxmIG9mIE5vcm1hbC5ucHogd2luZG93cyBmcm9tIGhvbGRvdXRfcnBtICh0aGUgIm5vcm1hbCIgY2xhc3MpCiAgICBQTFVTIGV2ZXJ5IGZhdWx0IGZpbGUgZnJvbSBjb25maWcuZGF0YS5mYXVsdF9ycG1zIHggZmF1bHRfdHlwZXMgKHRoZQogICAgImFub21hbHkiIGNsYXNzKS4gTm8gZmF1bHQgZGF0YSwgYW5kIG5vIGhvbGRvdXQtUlBNIGhlYWx0aHkgZGF0YSwgaXMKICAgIGV2ZXIgc2VlbiBkdXJpbmcgdHJhaW5pbmcuCgpUaGUgdmFsL3Rlc3Qgc3BsaXQgb2YgdGhlIGhvbGRvdXQgaGVhbHRoeSBmaWxlIGlzIGEgY29udGlndW91cyBzcGxpdCAoZmlyc3QKTiB3aW5kb3dzIHZzLiBsYXN0IE0gd2luZG93cyksIG5vdCBhIHJhbmRvbSBzaHVmZmxlOiBiZWNhdXNlIHdpbmRvd3Mgb3ZlcmxhcApieSBkZXNpZ24sIGEgcmFuZG9tIHNwbGl0IHdvdWxkIHB1dCBuZWFyLWR1cGxpY2F0ZSB3aW5kb3dzIG9uIGJvdGggc2lkZXMgb2YKdGhlIHNwbGl0IGFuZCBxdWlldGx5IGxlYWsgaW5mb3JtYXRpb24gZnJvbSB2YWwgaW50byB0ZXN0LgoKVGhpcyBtb2R1bGUgb25seSBkb2VzIGRldGVybWluaXN0aWMgZGF0YSBlbmdpbmVlcmluZyAobm8gbGVhcm5lZCBzdGF0ZSAtLQpub3JtYWxpemF0aW9uIGlzIGZpdCBsYXRlciwgaW4gdHJhaW4ucHkpLiBSZXN1bHRzIGFyZSBjYWNoZWQgdG8KZGF0YS9wcm9jZXNzZWQvIGFzIHBsYWluIC5ucHkvLmNzdiBzbyByZXBlYXRlZCBydW5zIHNraXAgdGhlIFNURlQgcmVjb21wdXRlLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSB2aWJyYXRpb25fYWQuY29uZmlnIGltcG9ydCBDb25maWcKZnJvbSB2aWJyYXRpb25fYWQuZGF0YS5jYXRhbG9nIGltcG9ydCBidWlsZF9jYXRhbG9nLCBzYXZlX2NhdGFsb2cKZnJvbSB2aWJyYXRpb25fYWQuZGF0YS53aW5kb3dzIGltcG9ydCBtYWtlX3dpbmRvd3MKZnJvbSB2aWJyYXRpb25fYWQuZmVhdHVyZXMuc3BlY3Ryb2dyYW0gaW1wb3J0IHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcwoKCmRlZiBsb2FkX3NpZ25hbChucHpfcGF0aDogc3RyLCBzZW5zb3Jfa2V5OiBzdHIpIC0+IG5wLm5kYXJyYXk6CiAgICB3aXRoIG5wLmxvYWQobnB6X3BhdGgpIGFzIGRhdGE6CiAgICAgICAgaWYgc2Vuc29yX2tleSBub3QgaW4gZGF0YToKICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJ7bnB6X3BhdGh9IGhhcyBubyAne3NlbnNvcl9rZXl9JyBhcnJheSAoaGFzOiB7bGlzdChkYXRhLmtleXMoKSl9KSIpCiAgICAgICAgcmV0dXJuIGRhdGFbc2Vuc29yX2tleV0uYXN0eXBlKG5wLmZsb2F0NjQpLnJhdmVsKCkKCgpkZWYgX3dpbmRvd3NfZm9yX2ZpbGUoZmlsZXBhdGg6IHN0ciwgc2Vuc29yOiBzdHIsIHNlZ21lbnRfbGVuZ3RoOiBpbnQsIHN0cmlkZTogaW50KSAtPiBucC5uZGFycmF5OgogICAgc2lnbmFsID0gbG9hZF9zaWduYWwoZmlsZXBhdGgsIHNlbnNvcikKICAgIHJldHVybiBtYWtlX3dpbmRvd3Moc2lnbmFsLCBzZWdtZW50X2xlbmd0aCwgc3RyaWRlKQoKCmRlZiBfc3BlY3Ryb2dyYW1zX2Zvcl9yb3dzKAogICAgcm93czogcGQuRGF0YUZyYW1lLCBjb25maWc6IENvbmZpZywgc3RyaWRlOiBpbnQKKSAtPiB0dXBsZVtucC5uZGFycmF5LCBwZC5EYXRhRnJhbWVdOgogICAgIiIiV2luZG93ICsgU1RGVCBldmVyeSBmaWxlIGluIGByb3dzYCwgY29uY2F0ZW5hdGUsIGFuZCByZXR1cm4gbWF0Y2hpbmcKICAgIHBlci13aW5kb3cgbWV0YWRhdGEgKG9uZSBtZXRhZGF0YSByb3cgcGVyIG91dHB1dCB3aW5kb3csIG5vdCBwZXIgZmlsZSkuCiAgICAiIiIKICAgIGFsbF93aW5kb3dzID0gW10KICAgIG1ldGFfcm93cyA9IFtdCiAgICBmb3IgXywgcm93IGluIHJvd3MuaXRlcnJvd3MoKToKICAgICAgICB3ID0gX3dpbmRvd3NfZm9yX2ZpbGUocm93WyJmaWxlcGF0aCJdLCBjb25maWcuZGF0YS5zZW5zb3IsIGNvbmZpZy53aW5kb3dpbmcuc2VnbWVudF9sZW5ndGgsIHN0cmlkZSkKICAgICAgICBpZiBsZW4odykgPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBhbGxfd2luZG93cy5hcHBlbmQodykKICAgICAgICBmb3IgXyBpbiByYW5nZShsZW4odykpOgogICAgICAgICAgICBtZXRhX3Jvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJycG0iOiByb3dbInJwbSJdLAogICAgICAgICAgICAgICAgICAgICJmYXVsdF90eXBlIjogcm93WyJmYXVsdF90eXBlIl0sCiAgICAgICAgICAgICAgICAgICAgImZhdWx0X3NpemVfbWlscyI6IHJvd1siZmF1bHRfc2l6ZV9taWxzIl0sCiAgICAgICAgICAgICAgICAgICAgImxhYmVsIjogcm93WyJsYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJzb3VyY2VfZmlsZSI6IHJvd1siZmlsZW5hbWUiXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQoKICAgIHdpbmRvd3MgPSBucC5jb25jYXRlbmF0ZShhbGxfd2luZG93cywgYXhpcz0wKQogICAgc3BlY3Ryb2dyYW1zID0gd2luZG93c190b19sb2dfc3BlY3Ryb2dyYW1zKHdpbmRvd3MsIGNvbmZpZy5kYXRhLmZzX2h6LCBjb25maWcuc3RmdCkKICAgIHJldHVybiBzcGVjdHJvZ3JhbXMsIHBkLkRhdGFGcmFtZShtZXRhX3Jvd3MpCgoKZGVmIF9jYWNoZV9wYXRocyhwcm9jZXNzZWRfZGlyOiBQYXRoKSAtPiBkaWN0W3N0ciwgUGF0aF06CiAgICByZXR1cm4gewogICAgICAgICJ0cmFpbl9YIjogcHJvY2Vzc2VkX2RpciAvICJ0cmFpbl9zcGVjdHJvZ3JhbXMubnB5IiwKICAgICAgICAidmFsX1giOiBwcm9jZXNzZWRfZGlyIC8gInZhbF9zcGVjdHJvZ3JhbXMubnB5IiwKICAgICAgICAidmFsX21ldGEiOiBwcm9jZXNzZWRfZGlyIC8gInZhbF9tZXRhLmNzdiIsCiAgICAgICAgInRlc3RfWCI6IHByb2Nlc3NlZF9kaXIgLyAidGVzdF9zcGVjdHJvZ3JhbXMubnB5IiwKICAgICAgICAidGVzdF9tZXRhIjogcHJvY2Vzc2VkX2RpciAvICJ0ZXN0X21ldGEuY3N2IiwKICAgIH0KCgpkZWYgYnVpbGRfcmF3X2RhdGFzZXRzKAogICAgY29uZmlnOiBDb25maWcsIGZvcmNlOiBib29sID0gRmFsc2UKKSAtPiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5LCBwZC5EYXRhRnJhbWUsIG5wLm5kYXJyYXksIHBkLkRhdGFGcmFtZV06CiAgICAiIiJSZXR1cm5zICh0cmFpbl9YLCB2YWxfWCwgdmFsX21ldGEsIHRlc3RfWCwgdGVzdF9tZXRhKS4KCiAgICB0cmFpbl9YOiAoTl90cmFpbiwgRiwgVCkgZmxvYXQzMiwgdW4tbm9ybWFsaXplZCBsb2ctbWFnbml0dWRlIHNwZWN0cm9ncmFtcy4KICAgIHZhbF9YOiAgIChOX3ZhbCwgRiwgVCksIGhlYWx0aHkgb25seS4KICAgIHRlc3RfWDogIChOX3Rlc3QsIEYsIFQpLCBoZWFsdGh5IChsYWJlbD0wKSArIGZhdWx0eSAobGFiZWw9MSkuCiAgICAqX21ldGE6ICBvbmUgcm93IHBlciB3aW5kb3csIGNvbHVtbnMgcnBtL2ZhdWx0X3R5cGUvZmF1bHRfc2l6ZV9taWxzL2xhYmVsL3NvdXJjZV9maWxlLgogICAgIiIiCiAgICBwcm9jZXNzZWRfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnByb2Nlc3NlZF9kaXIpCiAgICBwcm9jZXNzZWRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGhzID0gX2NhY2hlX3BhdGhzKHByb2Nlc3NlZF9kaXIpCgogICAgaWYgbm90IGZvcmNlIGFuZCBhbGwocC5leGlzdHMoKSBmb3IgcCBpbiBwYXRocy52YWx1ZXMoKSk6CiAgICAgICAgdHJhaW5fWCA9IG5wLmxvYWQocGF0aHNbInRyYWluX1giXSkKICAgICAgICB2YWxfWCA9IG5wLmxvYWQocGF0aHNbInZhbF9YIl0pCiAgICAgICAgdmFsX21ldGEgPSBwZC5yZWFkX2NzdihwYXRoc1sidmFsX21ldGEiXSkKICAgICAgICB0ZXN0X1ggPSBucC5sb2FkKHBhdGhzWyJ0ZXN0X1giXSkKICAgICAgICB0ZXN0X21ldGEgPSBwZC5yZWFkX2NzdihwYXRoc1sidGVzdF9tZXRhIl0pCiAgICAgICAgcmV0dXJuIHRyYWluX1gsIHZhbF9YLCB2YWxfbWV0YSwgdGVzdF9YLCB0ZXN0X21ldGEKCiAgICByYXdfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnJhd19kaXIpCiAgICBjYXRhbG9nID0gYnVpbGRfY2F0YWxvZyhyYXdfZGlyKQogICAgc2F2ZV9jYXRhbG9nKGNhdGFsb2csIHByb2Nlc3NlZF9kaXIpCgogICAgIyAtLS0gVFJBSU46IE5vcm1hbCBmaWxlcyBmcm9tIHRyYWluX3JwbXMsIGRlbnNlIG92ZXJsYXAgLS0tCiAgICB0cmFpbl9yb3dzID0gY2F0YWxvZ1sKICAgICAgICAoY2F0YWxvZy5mYXVsdF90eXBlID09ICJOb3JtYWwiKSAmIChjYXRhbG9nLnJwbS5pc2luKGNvbmZpZy5kYXRhLnRyYWluX3JwbXMpKQogICAgXQogICAgaWYgbGVuKHRyYWluX3Jvd3MpICE9IGxlbihjb25maWcuZGF0YS50cmFpbl9ycG1zKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiRXhwZWN0ZWQgb25lIE5vcm1hbC5ucHogcGVyIHRyYWluIFJQTSB7Y29uZmlnLmRhdGEudHJhaW5fcnBtc30sICIKICAgICAgICAgICAgZiJmb3VuZCB7bGVuKHRyYWluX3Jvd3MpfSBtYXRjaGluZyBmaWxlcy4iCiAgICAgICAgKQogICAgdHJhaW5fWCwgXyA9IF9zcGVjdHJvZ3JhbXNfZm9yX3Jvd3ModHJhaW5fcm93cywgY29uZmlnLCBjb25maWcud2luZG93aW5nLnRyYWluX3N0cmlkZSkKCiAgICAjIC0tLSBIT0xET1VUIGhlYWx0aHk6IE5vcm1hbCBmaWxlIGZyb20gaG9sZG91dF9ycG0sIHNwYXJzZSBvdmVybGFwLCBzcGxpdCBpbiB0d28gLS0tCiAgICBob2xkb3V0X3Jvd3MgPSBjYXRhbG9nWwogICAgICAgIChjYXRhbG9nLmZhdWx0X3R5cGUgPT0gIk5vcm1hbCIpICYgKGNhdGFsb2cucnBtID09IGNvbmZpZy5kYXRhLmhvbGRvdXRfcnBtKQogICAgXQogICAgaWYgbGVuKGhvbGRvdXRfcm93cykgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiRXhwZWN0ZWQgZXhhY3RseSBvbmUgTm9ybWFsLm5weiBmb3IgaG9sZG91dCBSUE0ge2NvbmZpZy5kYXRhLmhvbGRvdXRfcnBtfSwgIgogICAgICAgICAgICBmImZvdW5kIHtsZW4oaG9sZG91dF9yb3dzKX0uIgogICAgICAgICkKICAgIGhvbGRvdXRfWCwgaG9sZG91dF9tZXRhID0gX3NwZWN0cm9ncmFtc19mb3Jfcm93cyhob2xkb3V0X3Jvd3MsIGNvbmZpZywgY29uZmlnLndpbmRvd2luZy5ldmFsX3N0cmlkZSkKICAgIG5fdmFsID0gaW50KHJvdW5kKGxlbihob2xkb3V0X1gpICogY29uZmlnLnRyYWluLnZhbF9mcmFjdGlvbl9vZl9ob2xkb3V0KSkKICAgIHZhbF9YLCB0ZXN0X2hlYWx0aHlfWCA9IGhvbGRvdXRfWFs6bl92YWxdLCBob2xkb3V0X1hbbl92YWw6XQogICAgdmFsX21ldGEsIHRlc3RfaGVhbHRoeV9tZXRhID0gaG9sZG91dF9tZXRhLmlsb2NbOm5fdmFsXS5jb3B5KCksIGhvbGRvdXRfbWV0YS5pbG9jW25fdmFsOl0uY29weSgpCgogICAgIyAtLS0gVEVTVCBmYXVsdHk6IGV2ZXJ5IGZhdWx0IGZpbGUgZnJvbSBmYXVsdF9ycG1zIHggZmF1bHRfdHlwZXMgLS0tCiAgICBmYXVsdF9yb3dzID0gY2F0YWxvZ1sKICAgICAgICBjYXRhbG9nLmZhdWx0X3R5cGUuaXNpbihjb25maWcuZGF0YS5mYXVsdF90eXBlcykgJiBjYXRhbG9nLnJwbS5pc2luKGNvbmZpZy5kYXRhLmZhdWx0X3JwbXMpCiAgICBdCiAgICBpZiBsZW4oZmF1bHRfcm93cykgPT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIGZhdWx0eSBmaWxlcyBtYXRjaGVkIGNvbmZpZy5kYXRhLmZhdWx0X3R5cGVzL2ZhdWx0X3JwbXMuIikKICAgIGZhdWx0X1gsIGZhdWx0X21ldGEgPSBfc3BlY3Ryb2dyYW1zX2Zvcl9yb3dzKGZhdWx0X3Jvd3MsIGNvbmZpZywgY29uZmlnLndpbmRvd2luZy5ldmFsX3N0cmlkZSkKCiAgICB0ZXN0X1ggPSBucC5jb25jYXRlbmF0ZShbdGVzdF9oZWFsdGh5X1gsIGZhdWx0X1hdLCBheGlzPTApCiAgICB0ZXN0X21ldGEgPSBwZC5jb25jYXQoW3Rlc3RfaGVhbHRoeV9tZXRhLCBmYXVsdF9tZXRhXSwgYXhpcz0wLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKCiAgICBucC5zYXZlKHBhdGhzWyJ0cmFpbl9YIl0sIHRyYWluX1gpCiAgICBucC5zYXZlKHBhdGhzWyJ2YWxfWCJdLCB2YWxfWCkKICAgIHZhbF9tZXRhLnRvX2NzdihwYXRoc1sidmFsX21ldGEiXSwgaW5kZXg9RmFsc2UpCiAgICBucC5zYXZlKHBhdGhzWyJ0ZXN0X1giXSwgdGVzdF9YKQogICAgdGVzdF9tZXRhLnRvX2NzdihwYXRoc1sidGVzdF9tZXRhIl0sIGluZGV4PUZhbHNlKQoKICAgIHJldHVybiB0cmFpbl9YLCB2YWxfWCwgdmFsX21ldGEsIHRlc3RfWCwgdGVzdF9tZXRhCg=='
    ),
    'src/vibration_ad/data/windows.py': (
        'IiIiQ3V0IGEgbG9uZyAxRCB2aWJyYXRpb24gc2lnbmFsIGludG8gZml4ZWQtbGVuZ3RoIG92ZXJsYXBwaW5nIHdpbmRvd3MuCgpUaGlzIGlzIHRoZSBzdGVwIHRoYXQgdHVybnMgIm9uZSBsb25nIHJlY29yZGluZyBwZXIgYmVhcmluZyBjb25kaXRpb24iCmludG8gIm1hbnkgaW5kZXBlbmRlbnQgZXhhbXBsZXMiIGZvciB0aGUgYXV0b2VuY29kZXIgdG8gdHJhaW4vZXZhbHVhdGUgb24uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgbWFrZV93aW5kb3dzKHNpZ25hbDogbnAubmRhcnJheSwgc2VnbWVudF9sZW5ndGg6IGludCwgc3RyaWRlOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJTbGljZSBgc2lnbmFsYCBpbnRvIG92ZXJsYXBwaW5nIHdpbmRvd3Mgb2YgbGVuZ3RoIGBzZWdtZW50X2xlbmd0aGAuCgogICAgUmV0dXJucyBhbiBhcnJheSBvZiBzaGFwZSAobl93aW5kb3dzLCBzZWdtZW50X2xlbmd0aCkuIFVzZXMgYSBzdHJpZGVkCiAgICB2aWV3IChubyBjb3B5KSB2aWEgbnVtcHkncyBzbGlkaW5nX3dpbmRvd192aWV3IGZvciBzcGVlZCwgdGhlbiB0YWtlcwogICAgZXZlcnkgYHN0cmlkZWAtdGggd2luZG93LgoKICAgIEFueSBzYW1wbGVzIHRoYXQgZG9uJ3QgZmlsbCBhIGZpbmFsIGNvbXBsZXRlIHdpbmRvdyBhcmUgZHJvcHBlZC4KICAgICIiIgogICAgaWYgbGVuKHNpZ25hbCkgPCBzZWdtZW50X2xlbmd0aDoKICAgICAgICByZXR1cm4gbnAuZW1wdHkoKDAsIHNlZ21lbnRfbGVuZ3RoKSwgZHR5cGU9c2lnbmFsLmR0eXBlKQoKICAgIGFsbF93aW5kb3dzID0gbnAubGliLnN0cmlkZV90cmlja3Muc2xpZGluZ193aW5kb3dfdmlldyhzaWduYWwsIHNlZ21lbnRfbGVuZ3RoKQogICAgcmV0dXJuIG5wLmFzY29udGlndW91c2FycmF5KGFsbF93aW5kb3dzWzo6c3RyaWRlXSkKCgpkZWYgbl93aW5kb3dzKHNpZ25hbF9sZW5ndGg6IGludCwgc2VnbWVudF9sZW5ndGg6IGludCwgc3RyaWRlOiBpbnQpIC0+IGludDoKICAgICIiIk51bWJlciBvZiB3aW5kb3dzIG1ha2Vfd2luZG93cygpIHdvdWxkIHByb2R1Y2UsIHdpdGhvdXQgYnVpbGRpbmcgdGhlbS4iIiIKICAgIGlmIHNpZ25hbF9sZW5ndGggPCBzZWdtZW50X2xlbmd0aDoKICAgICAgICByZXR1cm4gMAogICAgcmV0dXJuIChzaWduYWxfbGVuZ3RoIC0gc2VnbWVudF9sZW5ndGgpIC8vIHN0cmlkZSArIDEK'
    ),
    'src/vibration_ad/features/__init__.py': (
        ''
    ),
    'src/vibration_ad/features/spectrogram.py': (
        'IiIiVHVybiByYXcgc2lnbmFsIHdpbmRvd3MgaW50byBub3JtYWxpemVkIGxvZy1tYWduaXR1ZGUgU1RGVCBzcGVjdHJvZ3JhbXMuCgpXZSB1c2UgYSBwbGFpbiBTVEZUIChsaW5lYXIgZnJlcXVlbmN5IGF4aXMpIHJhdGhlciB0aGFuIGEgTWVsLXNwZWN0cm9ncmFtLgpNZWwgc2NhbGluZyB3YXJwcyB0aGUgZnJlcXVlbmN5IGF4aXMgdG8gbWF0Y2ggaHVtYW4gcGl0Y2ggcGVyY2VwdGlvbiwgd2hpY2gKaXMgdGhlIHJpZ2h0IHRoaW5nIGZvciBzcGVlY2gvbXVzaWMgYnV0IGFjdGl2ZWx5IHVuaGVscGZ1bCBoZXJlOiBiZWFyaW5nCmZhdWx0IGVuZXJneSBzaG93cyB1cCBhdCBzcGVjaWZpYyBtZWNoYW5pY2FsIGZyZXF1ZW5jaWVzIChCUEZPLCBCUEZJLCBCU0YsCnNoYWZ0IHJhdGUgYW5kIHRoZWlyIGhhcm1vbmljcyAtLSBzZWUgbm90ZXMvMDVfZ2xvc3NhcnkubWQpLCBhbmQgTWVsIGJpbm5pbmcKd291bGQgYmx1ciBleGFjdGx5IHRoZSBoaWdoZXItZnJlcXVlbmN5IGRldGFpbCB0aGF0IHNlcGFyYXRlcyBvbmUgZmF1bHQKdHlwZS9zZXZlcml0eSBmcm9tIGFub3RoZXIuIExpbmVhciBTVEZUIGtlZXBzIHRoYXQgcmVzb2x1dGlvbiBpbnRhY3QgYW5kCmFsc28gZHJvcHMgdGhlIGxpYnJvc2EgZGVwZW5kZW5jeSBlbnRpcmVseSAoc2NpcHkuc2lnbmFsLnN0ZnQgaXMgZW5vdWdoKS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNjaXB5LnNpZ25hbCBpbXBvcnQgc3RmdAoKZnJvbSB2aWJyYXRpb25fYWQuY29uZmlnIGltcG9ydCBTVEZUQ29uZmlnLCBXaW5kb3dpbmdDb25maWcKCgpkZWYgZXhwZWN0ZWRfc2hhcGUod2luZG93aW5nOiBXaW5kb3dpbmdDb25maWcsIHN0ZnRfY2ZnOiBTVEZUQ29uZmlnKSAtPiB0dXBsZVtpbnQsIGludF06CiAgICAiIiIobl9mcmVxX2JpbnMsIG5fdGltZV9mcmFtZXMpIHRoYXQgd2luZG93c190b19sb2dfc3BlY3Ryb2dyYW1zKCkgd2lsbAogICAgcHJvZHVjZSBmb3IgdGhlIGdpdmVuIGNvbmZpZywgY29tcHV0ZWQgd2l0aG91dCBydW5uaW5nIGFueSBTVEZULgoKICAgIEZyZXEgYmlucyA9IG5wZXJzZWcgLy8gMiAodGhlIE55cXVpc3QgYmluIGlzIGRyb3BwZWQsIHNlZSBiZWxvdykuCiAgICBUaW1lIGZyYW1lcyBmb2xsb3dzIHNjaXB5J3Mgbm8tcGFkZGluZywgbm8tYm91bmRhcnkgU1RGVCBmcmFtZSBjb3VudDoKICAgIDEgKyAoc2VnbWVudF9sZW5ndGggLSBucGVyc2VnKSAvLyBob3BfbGVuZ3RoLgogICAgIiIiCiAgICBuX2ZyZXEgPSBzdGZ0X2NmZy5ucGVyc2VnIC8vIDIKICAgIG5fdGltZSA9ICh3aW5kb3dpbmcuc2VnbWVudF9sZW5ndGggLSBzdGZ0X2NmZy5ucGVyc2VnKSAvLyBzdGZ0X2NmZy5ob3BfbGVuZ3RoICsgMQogICAgcmV0dXJuIG5fZnJlcSwgbl90aW1lCgoKZGVmIHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcygKICAgIHdpbmRvd3M6IG5wLm5kYXJyYXksIGZzX2h6OiBpbnQsIHN0ZnRfY2ZnOiBTVEZUQ29uZmlnCikgLT4gbnAubmRhcnJheToKICAgICIiIndpbmRvd3M6IChOLCBzZWdtZW50X2xlbmd0aCkgLT4gbG9nLW1hZ25pdHVkZSBzcGVjdHJvZ3JhbXMgKE4sIEYsIFQpLgoKICAgIGJvdW5kYXJ5PU5vbmUsIHBhZGRlZD1GYWxzZSBzbyB0aGUgb3V0cHV0IHRpbWUtZnJhbWUgY291bnQgaXMgZXhhY3RseQogICAgcHJlZGljdGFibGUgZnJvbSAoc2VnbWVudF9sZW5ndGgsIG5wZXJzZWcsIGhvcF9sZW5ndGgpIC0tIHNlZQogICAgZXhwZWN0ZWRfc2hhcGUoKS4gVGhlIE55cXVpc3QgYmluIChsYXN0IGZyZXEgYmluKSBpcyBkcm9wcGVkIG9ubHkgdG8KICAgIG1ha2UgRiBhIHJvdW5kLCBwb29saW5nLWZyaWVuZGx5IG51bWJlciBmb3IgdGhlIGNvbnYgYXV0b2VuY29kZXI7IGZvcgogICAgdGhlc2Ugc2lnbmFscyBpdCBjYXJyaWVzIG5lZ2xpZ2libGUgZW5lcmd5IGFueXdheS4KICAgICIiIgogICAgXywgXywgWnh4ID0gc3RmdCgKICAgICAgICB3aW5kb3dzLAogICAgICAgIGZzPWZzX2h6LAogICAgICAgIHdpbmRvdz0iaGFubiIsCiAgICAgICAgbnBlcnNlZz1zdGZ0X2NmZy5ucGVyc2VnLAogICAgICAgIG5vdmVybGFwPXN0ZnRfY2ZnLm5vdmVybGFwLAogICAgICAgIGJvdW5kYXJ5PU5vbmUsCiAgICAgICAgcGFkZGVkPUZhbHNlLAogICAgICAgIGF4aXM9LTEsCiAgICApCiAgICBtYWduaXR1ZGUgPSBucC5hYnMoWnh4KVs6LCA6LTEsIDpdICAjIGRyb3AgTnlxdWlzdCBiaW4gLT4gKE4sIG5wZXJzZWcvMiwgVCkKICAgIHJldHVybiBucC5sb2cobWFnbml0dWRlICsgc3RmdF9jZmcubG9nX2Vwc2lsb24pLmFzdHlwZShucC5mbG9hdDMyKQoKCkBkYXRhY2xhc3MKY2xhc3MgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyOgogICAgIiIiUGVyLWZyZXF1ZW5jeS1iaW4gei1zY29yZSBub3JtYWxpemF0aW9uLCBmaXQgb24gVFJBSU4gaGVhbHRoeSBkYXRhIG9ubHkuCgogICAgUGVyLWJpbiAobm90IG9uZSBnbG9iYWwgc2NhbGFyKSBtYXR0ZXJzIGhlcmU6IGxvdyBmcmVxdWVuY2llcyBjYXJyeSBmYXIKICAgIG1vcmUgZW5lcmd5IHRoYW4gaGlnaCBmcmVxdWVuY2llcyBpbiB0aGVzZSBzaWduYWxzLCBzbyBhIHNpbmdsZSBnbG9iYWwKICAgIG1lYW4vc3RkIHdvdWxkIGxlYXZlIGhpZ2gtZnJlcXVlbmN5IGJpbnMgLS0gd2hlcmUgc3VidGxlIGZhdWx0IGhhcm1vbmljcwogICAgb2Z0ZW4gc2hvdyB1cCAtLSBhbG1vc3QgZmxhdCBhZnRlciBub3JtYWxpemF0aW9uLiBGaXR0aW5nIG9ubHkgb24KICAgIHRyYWluaW5nLXNwbGl0IGhlYWx0aHkgc3BlY3Ryb2dyYW1zIChuZXZlciB2YWwvdGVzdC9mYXVsdHkgZGF0YSkgYXZvaWRzCiAgICBsZWFraW5nIGFueSBpbmZvcm1hdGlvbiBhYm91dCB0aGUgZXZhbHVhdGlvbiBzZXQgaW50byBwcmVwcm9jZXNzaW5nLgogICAgIiIiCgogICAgbWVhbjogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lICAjIHNoYXBlIChGLCAxKQogICAgc3RkOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUgICMgc2hhcGUgKEYsIDEpCgogICAgZGVmIGZpdChzZWxmLCBzcGVjdHJvZ3JhbXM6IG5wLm5kYXJyYXkpIC0+ICJTcGVjdHJvZ3JhbU5vcm1hbGl6ZXIiOgogICAgICAgIHNlbGYubWVhbiA9IHNwZWN0cm9ncmFtcy5tZWFuKGF4aXM9KDAsIDIpLCBrZWVwZGltcz1UcnVlKVswXQogICAgICAgIHNlbGYuc3RkID0gc3BlY3Ryb2dyYW1zLnN0ZChheGlzPSgwLCAyKSwga2VlcGRpbXM9VHJ1ZSlbMF0gKyAxZS04CiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgdHJhbnNmb3JtKHNlbGYsIHNwZWN0cm9ncmFtczogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICBpZiBzZWxmLm1lYW4gaXMgTm9uZSBvciBzZWxmLnN0ZCBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlNwZWN0cm9ncmFtTm9ybWFsaXplci5maXQoKSBtdXN0IGJlIGNhbGxlZCBiZWZvcmUgdHJhbnNmb3JtKCkuIikKICAgICAgICByZXR1cm4gKChzcGVjdHJvZ3JhbXMgLSBzZWxmLm1lYW4pIC8gc2VsZi5zdGQpLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIGRlZiBzYXZlKHNlbGYsIHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgbnAuc2F2ZXoocGF0aCwgbWVhbj1zZWxmLm1lYW4sIHN0ZD1zZWxmLnN0ZCkKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBsb2FkKGNscywgcGF0aDogUGF0aCkgLT4gIlNwZWN0cm9ncmFtTm9ybWFsaXplciI6CiAgICAgICAgZGF0YSA9IG5wLmxvYWQocGF0aCkKICAgICAgICByZXR1cm4gY2xzKG1lYW49ZGF0YVsibWVhbiJdLCBzdGQ9ZGF0YVsic3RkIl0pCg=='
    ),
    'src/domain_adaptation/__init__.py': (
        'IiIiWmVyby1zaG90IGZhdWx0IGRpYWdub3NpcyB2aWEgdW5zdXBlcnZpc2VkIGRvbWFpbiBhZGFwdGF0aW9uIChEQU5OKS4KClRoaXJkLCBpbmRlcGVuZGVudCBleHBlcmltZW50IGluIHRoaXMgcmVwbyAoc2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQpOgpgdmlicmF0aW9uX2FkL2AgKHNpbmdsZS1zZW5zb3IgQ1dSVSBhdXRvZW5jb2RlcikgYW5kIGBoeWRyYXVsaWNfZnVzaW9uX2FkL2AKKG11bHRpLXNlbnNvciBaZU1BIGZ1c2lvbiBhdXRvZW5jb2RlcikgYXJlIGJvdGggdW50b3VjaGVkLiBUaGlzIHBhY2thZ2UKY29tYmluZXMgQ1dSVSAoYWxyZWFkeSBsb2NhbCwgYGRhdGEvcmF3L2ApIGFuZCBQYWRlcmJvcm4gVW5pdmVyc2l0eSdzIEtBdApiZWFyaW5nIGRhdGFzZXQgKGF0dGFjaGVkIGFzIGEgS2FnZ2xlIGlucHV0IGRhdGFzZXQsIG5ldmVyIGRvd25sb2FkZWQgaW4KZnVsbCkgdG8gdHJhaW4gYSBjbGFzc2lmaWVyIHRoYXQgdHJhbnNmZXJzIGFjcm9zcyBib3RoIGRhdGFzZXRzIGFuZCBhCmhlbGQtb3V0IG9wZXJhdGluZyBjb25kaXRpb24sIHVzaW5nIG9ubHkgdGhhdCBjb25kaXRpb24ncyBIRUFMVEhZIGRhdGEgdG8KYWRhcHQgLS0gbmV2ZXIgYSBzaW5nbGUgbGFiZWxlZCAob3IgZXZlbiB1bmxhYmVsZWQpIGZhdWx0IGV4YW1wbGUgZnJvbSB0aGUKdGFyZ2V0IC0tIHRoZW4gY2hlY2tzIHdoZXRoZXIgaXQgY2FuIGRpYWdub3NlIHJlYWwgdGFyZ2V0IGZhdWx0cyBpdCBuZXZlcgp0cmFpbmVkIG9uLgoiIiIK'
    ),
    'src/domain_adaptation/config.py': (
        'IiIiTG9hZCBjb25maWdzL2RvbWFpbl9hZGFwdGF0aW9uLnlhbWwgaW50byBhIHNtYWxsIHRyZWUgb2YgZGF0YWNsYXNzZXMuCgpTYW1lIHBhdHRlcm4gYXMgdmlicmF0aW9uX2FkLmNvbmZpZyAvIGh5ZHJhdWxpY19mdXNpb25fYWQuY29uZmlnOiBvbmUgWUFNTApmaWxlIGlzIHRoZSBzaW5nbGUgc291cmNlIG9mIHRydXRoIGZvciBhIHJ1biwgYW5kIGV2ZXJ5IHN0YWdlIChkYXRhIHByZXAsCmJhc2VsaW5lIHRyYWluaW5nLCBEQU5OIHRyYWluaW5nLCBldmFsdWF0aW9uKSB0YWtlcyBhIGBDb25maWdgIG9iamVjdAppbnN0ZWFkIG9mIHNjYXR0ZXJlZCBDTEkgZmxhZ3MuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHlhbWwKCgpAZGF0YWNsYXNzCmNsYXNzIFBhdGhzQ29uZmlnOgogICAgY3dydV9yYXdfZGlyOiBzdHIKICAgIHByb2Nlc3NlZF9kaXI6IHN0cgogICAgbW9kZWxzX2Rpcjogc3RyCiAgICBmaWd1cmVzX2Rpcjogc3RyCiAgICByZXBvcnRzX2Rpcjogc3RyCiAgICAjIExvY2FsLWRldi1vbmx5IGZhbGxiYWNrOiBhIGRpcmVjdG9yeSBsYWlkIG91dCBsaWtlIHRoZSByZWFsIFBVIGRhdGFzZXQKICAgICMgKGJlYXJpbmctY29kZSBzdWJmb2xkZXJzIG9mIC5tYXQgZmlsZXMpLCB1c2VkIHdoZW4gL2thZ2dsZS9pbnB1dCBpc24ndAogICAgIyBwcmVzZW50IGFuZCAtLXB1LXJvb3Qgd2Fzbid0IHBhc3NlZCBvbiB0aGUgQ0xJLiBOZXZlciByZXF1aXJlZCB0bwogICAgIyBleGlzdCAtLSBzZWUgZGF0YS9wdV9sb2FkZXIuZmluZF9wdV9yb290KCkuCiAgICBwdV9yb290X2xvY2FsOiBzdHIKCgpAZGF0YWNsYXNzCmNsYXNzIENvbW1vbkNvbmZpZzoKICAgICMgRXZlcnkgZG9tYWluIGlzIHJlc2FtcGxlZCB0byB0aGlzIHJhdGUgYmVmb3JlIHdpbmRvd2luZy9TVEZUIC0tIHNlZQogICAgIyBkYXRhL3RyYW5zZm9ybXMucHkgYW5kIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncyAiUmV2aXNpb25zIiBzZWN0aW9uCiAgICAjIGZvciB3aHkgMTJrSHogKENXUlUncyBuYXRpdmUgcmF0ZSkgcmF0aGVyIHRoYW4gdXAgc2FtcGxpbmcgZXZlcnl0aGluZy4KICAgIGZzX2h6OiBpbnQKICAgIHNlZ21lbnRfbGVuZ3RoOiBpbnQKICAgIHRyYWluX3N0cmlkZTogaW50CiAgICBldmFsX3N0cmlkZTogaW50CgoKQGRhdGFjbGFzcwpjbGFzcyBTVEZUQ29uZmlnOgogICAgbnBlcnNlZzogaW50CiAgICBob3BfbGVuZ3RoOiBpbnQKICAgIGxvZ19lcHNpbG9uOiBmbG9hdAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG5vdmVybGFwKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5ucGVyc2VnIC0gc2VsZi5ob3BfbGVuZ3RoCgoKQGRhdGFjbGFzcwpjbGFzcyBDV1JVQ29uZmlnOgogICAgc2Vuc29yOiBzdHIKICAgIHRyYWluX3JwbXM6IGxpc3RbaW50XQogICAgdmFsX3JwbTogaW50CiAgICAjIEZhbHNlIC0+IENXUlUgaXMgZHJvcHBlZCBmcm9tIHRoZSBzb3VyY2UgcG9vbCBlbnRpcmVseSBhbmQgU09VUkNFX1RSQUlOIC8KICAgICMgU09VUkNFX1ZBTCBjb21lIG9ubHkgZnJvbSBQYWRlcmJvcm4uIElzb2xhdGVzIHRoZSBjcm9zcy1EQVRBU0VUIGdhcCBmcm9tCiAgICAjIHRoZSBjcm9zcy1DT05ESVRJT04gb25lOiB3aXRoIENXUlUgb3V0LCB0aGUgb25seSBzaGlmdCBsZWZ0IGJldHdlZW4KICAgICMgc291cmNlIGFuZCB0YXJnZXQgaXMgUFUncyBvd24gTjE1IC0+IE4wOSBzcGVlZCBjaGFuZ2UgKHBsdXMgYXJ0aWZpY2lhbAogICAgIyAtPiByZWFsIGRhbWFnZSkuIFNlZSBET01BSU5fQURBUFRBVElPTl9QTEFOLm1kJ3Mga2VybmVsIHY0IG5vdGVzLgogICAgaW5jbHVkZV9pbl9zb3VyY2U6IGJvb2wgPSBUcnVlCgoKQGRhdGFjbGFzcwpjbGFzcyBQVUNvbmZpZzoKICAgIGRhdGFzZXRfc2x1Zzogc3RyCiAgICBuYXRpdmVfZnNfaHo6IGludAogICAgdmlicmF0aW9uX2NoYW5uZWw6IHN0cgogICAgaGVhbHRoeV9jb2RlczogbGlzdFtzdHJdCiAgICBhcnRpZmljaWFsX291dGVyX2NvZGVzOiBsaXN0W3N0cl0KICAgIGFydGlmaWNpYWxfaW5uZXJfY29kZXM6IGxpc3Rbc3RyXQogICAgcmVhbF9vdXRlcl9jb2RlczogbGlzdFtzdHJdCiAgICByZWFsX2lubmVyX2NvZGVzOiBsaXN0W3N0cl0KICAgIHNvdXJjZV9jb25kaXRpb25zOiBsaXN0W3N0cl0KICAgIHRhcmdldF9jb25kaXRpb246IHN0cgogICAgIyBGcmFjdGlvbiBvZiBlYWNoIHNvdXJjZS1jb25kaXRpb24gYmVhcmluZydzIHJ1bnMgKG9mIH4yMCkgdXNlZCBmb3IKICAgICMgU09VUkNFIFRSQUlOOyB0aGUgcmVzdCAtPiBzb3VyY2UgdmFsLiBDb250aWd1b3VzIHNwbGl0IGJ5IHJ1biBpbmRleCwKICAgICMgbm90IHJhbmRvbSAtLSBzYW1lIGFudGktbGVha2FnZSBzcGlyaXQgYXMgQ1dSVSdzIFJQTSBob2xkb3V0LCB0aG91Z2gKICAgICMgaGVyZSBpdCdzIHdob2xlIDRzIHJ1bnMsIG5vdCBvdmVybGFwcGluZyB3aW5kb3dzLCB0aGF0IGdldCBzcGxpdC4KICAgIHNvdXJjZV90cmFpbl9mcmFjdGlvbjogZmxvYXQKICAgICMgRnJhY3Rpb24gb2YgdGhlIFRBUkdFVCBjb25kaXRpb24ncyBoZWFsdGh5IHJ1bnMgdXNlZCBmb3IgdGhlCiAgICAjIGFkYXB0YXRpb24gc3RlcCAoZG9tYWluIGRpc2NyaW1pbmF0b3IgKyBoZWFsdGh5LWFuY2hvciBsb3NzKTsgdGhlCiAgICAjIHJlc3QgYXJlIGhlbGQgb3V0IGFzIGhlYWx0aHkgcm93cyBpbiB0aGUgZmluYWwgemVyby1zaG90IHRlc3Qgc2V0LgogICAgdGFyZ2V0X25vcm1hbF9hZGFwdF9mcmFjdGlvbjogZmxvYXQKCiAgICBAcHJvcGVydHkKICAgIGRlZiBzb3VyY2VfY29kZXMoc2VsZikgLT4gbGlzdFtzdHJdOgogICAgICAgIHJldHVybiBzZWxmLmhlYWx0aHlfY29kZXMgKyBzZWxmLmFydGlmaWNpYWxfb3V0ZXJfY29kZXMgKyBzZWxmLmFydGlmaWNpYWxfaW5uZXJfY29kZXMKCiAgICBAcHJvcGVydHkKICAgIGRlZiByZWFsX2NvZGVzKHNlbGYpIC0+IGxpc3Rbc3RyXToKICAgICAgICByZXR1cm4gc2VsZi5yZWFsX291dGVyX2NvZGVzICsgc2VsZi5yZWFsX2lubmVyX2NvZGVzCgoKQGRhdGFjbGFzcwpjbGFzcyBNb2RlbENvbmZpZzoKICAgIGJhc2VfY2hhbm5lbHM6IGludAogICAgZmVhdHVyZV9kaW06IGludAogICAgY2xhc3NpZmllcl9oaWRkZW46IGludAogICAgZG9tYWluX2hpZGRlbjogaW50CiAgICBkcm9wb3V0OiBmbG9hdAoKCkBkYXRhY2xhc3MKY2xhc3MgQmFzZWxpbmVUcmFpbkNvbmZpZzoKICAgIGJhdGNoX3NpemU6IGludAogICAgZXBvY2hzOiBpbnQKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0CiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0CiAgICBlYXJseV9zdG9wcGluZ19wYXRpZW5jZTogaW50CiAgICBkZXZpY2U6IHN0cgoKCkBkYXRhY2xhc3MKY2xhc3MgREFOTlRyYWluQ29uZmlnOgogICAgYmF0Y2hfc2l6ZTogaW50CiAgICBlcG9jaHM6IGludAogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQKICAgIHdlaWdodF9kZWNheTogZmxvYXQKICAgIGVhcmx5X3N0b3BwaW5nX3BhdGllbmNlOiBpbnQKICAgIGRvbWFpbl9sb3NzX3dlaWdodDogZmxvYXQKICAgIGFuY2hvcl9sb3NzX3dlaWdodDogZmxvYXQKICAgIGdybF9nYW1tYTogZmxvYXQgICMgc3RlZXBuZXNzIG9mIHRoZSBHUkwgbGFtYmRhIHdhcm0tdXAgc2NoZWR1bGUgKHNlZSBtb2RlbHMvZ3JsLnB5KQogICAgIyBNYXggZ2xvYmFsIGdyYWRpZW50IG5vcm0sIGNsaXBwZWQgZXZlcnkgc3RlcC4gUHVyZSBzYWZldHkgbmV0OiBpdCBpcwogICAgIyBpbmVydCBvbiBhIGhlYWx0aHkgcnVuIGFuZCBvbmx5IGJpdGVzIG9uIGEgc3Bpa2UgbGlrZSBrZXJuZWwgdjMncwogICAgIyBlcG9jaCA0IChkb21haW5fbG9zcyAxNS40LCBhbmNob3JfbG9zcyA2Ny44KSwgd2hpY2ggYmxldyB0aGUgZmVhdHVyZQogICAgIyBleHRyYWN0b3Igb3V0IGFuZCBsZWZ0IGJvdGggaGVhZHMgZW1pdHRpbmcgY29uc3RhbnRzIGZvciB0aGUgcmVzdCBvZgogICAgIyB0aGUgcnVuLiBTZXQgZ2VuZXJvdXNseSBzbyBpdCBkb2VzIG5vdCBhY3QgYXMgYSByZWd1bGFyaXplci4KICAgIGdyYWRfY2xpcF9ub3JtOiBmbG9hdAogICAgIyBMZWFybmluZy1yYXRlIG11bHRpcGxpZXIgZm9yIHRoZSBkb21haW4gZGlzY3JpbWluYXRvcidzIG93biBwYXJhbWV0ZXIKICAgICMgZ3JvdXAgKDEuMCA9IHNhbWUgYXMgZXZlcnl0aGluZyBlbHNlKS4gTGV0cyB0aGUgZGlzY3JpbWluYXRvciBzdGF5IGEKICAgICMgc3RlcCBhaGVhZCBpbiB0aGUgbWluaW1heCBnYW1lIGlmIGl0J3MgbG9zaW5nIHRoZSByYWNlIC0tIHNlZQogICAgIyB0cmFpbl9kYW5uLnB5J3Mgb3B0aW1pemVyIHNldHVwLgogICAgZGlzY3JpbWluYXRvcl9scl9tdWx0aXBsaWVyOiBmbG9hdAogICAgZGV2aWNlOiBzdHIKCgpAZGF0YWNsYXNzCmNsYXNzIEV2YWx1YXRpb25Db25maWc6CiAgICB0c25lX3BlcnBsZXhpdHk6IGZsb2F0CgoKQGRhdGFjbGFzcwpjbGFzcyBDb25maWc6CiAgICBzZWVkOiBpbnQKICAgIHBhdGhzOiBQYXRoc0NvbmZpZwogICAgY29tbW9uOiBDb21tb25Db25maWcKICAgIHN0ZnQ6IFNURlRDb25maWcKICAgIGN3cnU6IENXUlVDb25maWcKICAgIHB1OiBQVUNvbmZpZwogICAgbW9kZWw6IE1vZGVsQ29uZmlnCiAgICB0cmFpbl9iYXNlbGluZTogQmFzZWxpbmVUcmFpbkNvbmZpZwogICAgdHJhaW5fZGFubjogREFOTlRyYWluQ29uZmlnCiAgICBldmFsdWF0aW9uOiBFdmFsdWF0aW9uQ29uZmlnCiAgICBwcm9qZWN0X3Jvb3Q6IFBhdGggPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9UGF0aC5jd2QpCgogICAgZGVmIHJlc29sdmUoc2VsZiwgcmVsYXRpdmVfcGF0aDogc3RyKSAtPiBQYXRoOgogICAgICAgIHJldHVybiAoc2VsZi5wcm9qZWN0X3Jvb3QgLyByZWxhdGl2ZV9wYXRoKS5yZXNvbHZlKCkKCgpkZWYgbG9hZF9jb25maWcoY29uZmlnX3BhdGg6IHN0ciB8IFBhdGgpIC0+IENvbmZpZzoKICAgIGNvbmZpZ19wYXRoID0gUGF0aChjb25maWdfcGF0aCkucmVzb2x2ZSgpCiAgICB3aXRoIG9wZW4oY29uZmlnX3BhdGgpIGFzIGY6CiAgICAgICAgcmF3ID0geWFtbC5zYWZlX2xvYWQoZikKCiAgICBwcm9qZWN0X3Jvb3QgPSBjb25maWdfcGF0aC5wYXJlbnQucGFyZW50ICAjIGNvbmZpZ3MvZG9tYWluX2FkYXB0YXRpb24ueWFtbCAtPiBwcm9qZWN0IHJvb3QKCiAgICByZXR1cm4gQ29uZmlnKAogICAgICAgIHNlZWQ9cmF3WyJzZWVkIl0sCiAgICAgICAgcGF0aHM9UGF0aHNDb25maWcoKipyYXdbInBhdGhzIl0pLAogICAgICAgIGNvbW1vbj1Db21tb25Db25maWcoKipyYXdbImNvbW1vbiJdKSwKICAgICAgICBzdGZ0PVNURlRDb25maWcoKipyYXdbInN0ZnQiXSksCiAgICAgICAgY3dydT1DV1JVQ29uZmlnKCoqcmF3WyJjd3J1Il0pLAogICAgICAgIHB1PVBVQ29uZmlnKCoqcmF3WyJwdSJdKSwKICAgICAgICBtb2RlbD1Nb2RlbENvbmZpZygqKnJhd1sibW9kZWwiXSksCiAgICAgICAgdHJhaW5fYmFzZWxpbmU9QmFzZWxpbmVUcmFpbkNvbmZpZygqKnJhd1sidHJhaW5fYmFzZWxpbmUiXSksCiAgICAgICAgdHJhaW5fZGFubj1EQU5OVHJhaW5Db25maWcoKipyYXdbInRyYWluX2Rhbm4iXSksCiAgICAgICAgZXZhbHVhdGlvbj1FdmFsdWF0aW9uQ29uZmlnKCoqcmF3WyJldmFsdWF0aW9uIl0pLAogICAgICAgIHByb2plY3Rfcm9vdD1wcm9qZWN0X3Jvb3QsCiAgICApCg=='
    ),
    'src/domain_adaptation/data/__init__.py': (
        ''
    ),
    'src/domain_adaptation/data/pu_bearing_codes.py': (
        'IiIiQ2Fub25pY2FsIGJlYXJpbmctY29kZSAtPiBmYXVsdCBtZXRhZGF0YSBmb3IgdGhlIFBhZGVyYm9ybiBVbml2ZXJzaXR5IC8KS0F0LURhdGFDZW50ZXIgYmVhcmluZyBkYXRhc2V0IChMZXNzbWVpZXIgZXQgYWwuIDIwMTYsICJDb25kaXRpb24gTW9uaXRvcmluZwpvZiBCZWFyaW5nIERhbWFnZSBpbiBFbGVjdHJvbWVjaGFuaWNhbCBEcml2ZSBTeXN0ZW1zIikuCgpWZXJpZmllZCBhZ2FpbnN0IHRoZSByZWFsIGRhdGFzZXQgb24gMjAyNi0wOS0xMCAoc2VlCkRPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncyAiUmV2aXNpb25zIiBzZWN0aW9uKTogZG93bmxvYWRlZCAzIHJlYWwgZmlsZXMKb2ZmIHRoZSBhdHRhY2hlZCBLYWdnbGUgbWlycm9yIChgZGlwcGF0ZWwwMy9wYWRlcmJvcm4tZGJgKSBhbmQgcGFnZWQKdGhyb3VnaCBpdHMgZnVsbCBmaWxlIGxpc3RpbmcgdmlhIHRoZSBLYWdnbGUgQVBJIHRvIGNvbmZpcm0gZXZlcnkgY29kZQpiZWxvdyBleGlzdHMsIHdpdGggMjAgcnVucyB1bmRlciBlYWNoIG9mIHRoZSBkYXRhc2V0J3MgNCBvcGVyYXRpbmcKY29uZGl0aW9ucy4KCmB1bml2ZXJzYWxfbGFiZWxgIG1hcHMgaW50byB0aGUgU0FNRSAzLWNsYXNzIHNwYWNlIENXUlUgdXNlcyAoc2VlCmN3cnVfbG9hZGVyLlVOSVZFUlNBTF9MQUJFTFMpOiAwPWhlYWx0aHksIDE9aW5uZXJfcmFjZSwgMj1vdXRlcl9yYWNlLgpDb21iaW5lZC1kYW1hZ2UgY29kZXMgKEtCKiwgYm90aCByYWNlcyBkYW1hZ2VkIG9uIG9uZSBwaHlzaWNhbCBiZWFyaW5nKQpnZXQgdW5pdmVyc2FsX2xhYmVsPU5vbmUgLS0gdGhleSBkb24ndCBmaXQgYSBzaW5nbGUtZmF1bHQtdHlwZSBsYWJlbCBhbmQKYXJlIGV4Y2x1ZGVkIGZyb20gdGhpcyBwcm9qZWN0J3MgY2xhc3NpZmljYXRpb24gdGFzazsga2VwdCBoZXJlIG9ubHkgZm9yCnJlZmVyZW5jZS9jb21wbGV0ZW5lc3MuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpIRUFMVEhZID0gWyJLMDAxIiwgIkswMDIiLCAiSzAwMyIsICJLMDA0IiwgIkswMDUiLCAiSzAwNiJdCkFSVElGSUNJQUxfT1VURVIgPSBbIktBMDEiLCAiS0EwMyIsICJLQTA1IiwgIktBMDYiLCAiS0EwNyIsICJLQTA4IiwgIktBMDkiXQpBUlRJRklDSUFMX0lOTkVSID0gWyJLSTAxIiwgIktJMDMiLCAiS0kwNSIsICJLSTA3IiwgIktJMDgiXQpSRUFMX09VVEVSID0gWyJLQTA0IiwgIktBMTUiLCAiS0ExNiIsICJLQTIyIiwgIktBMzAiXQpSRUFMX0lOTkVSID0gWyJLSTA0IiwgIktJMTQiLCAiS0kxNiIsICJLSTE3IiwgIktJMTgiLCAiS0kyMSJdClJFQUxfQ09NQklORUQgPSBbIktCMjMiLCAiS0IyNCIsICJLQjI3Il0KCgpkZWYgX2J1aWxkX3RhYmxlKCkgLT4gZGljdFtzdHIsIGRpY3RdOgogICAgdGFibGU6IGRpY3Rbc3RyLCBkaWN0XSA9IHt9CiAgICBmb3IgY29kZSBpbiBIRUFMVEhZOgogICAgICAgIHRhYmxlW2NvZGVdID0geyJmYXVsdF90eXBlIjogImhlYWx0aHkiLCAiZGFtYWdlX21vZGUiOiAibm9uZSIsICJ1bml2ZXJzYWxfbGFiZWwiOiAwfQogICAgZm9yIGNvZGUgaW4gQVJUSUZJQ0lBTF9PVVRFUjoKICAgICAgICB0YWJsZVtjb2RlXSA9IHsiZmF1bHRfdHlwZSI6ICJvdXRlcl9yYWNlIiwgImRhbWFnZV9tb2RlIjogImFydGlmaWNpYWwiLCAidW5pdmVyc2FsX2xhYmVsIjogMn0KICAgIGZvciBjb2RlIGluIEFSVElGSUNJQUxfSU5ORVI6CiAgICAgICAgdGFibGVbY29kZV0gPSB7ImZhdWx0X3R5cGUiOiAiaW5uZXJfcmFjZSIsICJkYW1hZ2VfbW9kZSI6ICJhcnRpZmljaWFsIiwgInVuaXZlcnNhbF9sYWJlbCI6IDF9CiAgICBmb3IgY29kZSBpbiBSRUFMX09VVEVSOgogICAgICAgIHRhYmxlW2NvZGVdID0geyJmYXVsdF90eXBlIjogIm91dGVyX3JhY2UiLCAiZGFtYWdlX21vZGUiOiAicmVhbCIsICJ1bml2ZXJzYWxfbGFiZWwiOiAyfQogICAgZm9yIGNvZGUgaW4gUkVBTF9JTk5FUjoKICAgICAgICB0YWJsZVtjb2RlXSA9IHsiZmF1bHRfdHlwZSI6ICJpbm5lcl9yYWNlIiwgImRhbWFnZV9tb2RlIjogInJlYWwiLCAidW5pdmVyc2FsX2xhYmVsIjogMX0KICAgIGZvciBjb2RlIGluIFJFQUxfQ09NQklORUQ6CiAgICAgICAgdGFibGVbY29kZV0gPSB7ImZhdWx0X3R5cGUiOiAiY29tYmluZWQiLCAiZGFtYWdlX21vZGUiOiAicmVhbCIsICJ1bml2ZXJzYWxfbGFiZWwiOiBOb25lfQogICAgcmV0dXJuIHRhYmxlCgoKUFVfQkVBUklOR19DT0RFUyA9IF9idWlsZF90YWJsZSgpCg=='
    ),
    'src/domain_adaptation/data/cwru_loader.py': (
        'IiIiQ1dSVSBjYXRhbG9nLCByZW1hcHBlZCBvbnRvIHRoZSB1bml2ZXJzYWwgMy1jbGFzcyBsYWJlbCBzcGFjZS4KClJldXNlcyB2aWJyYXRpb25fYWQuZGF0YS5jYXRhbG9nLmJ1aWxkX2NhdGFsb2coKSBmb3IgdGhlIGFjdHVhbCBmaWxlbmFtZQpwYXJzaW5nIChycG0gLyBmYXVsdF90eXBlIC8gZmF1bHRfc2l6ZV9taWxzKSAtLSBubyBuZWVkIHRvIHJlcGFyc2UgQ1dSVSdzCm5hbWluZyBjb252ZW50aW9uIGEgc2Vjb25kIHRpbWUsIHNlZSB2aWJyYXRpb25fYWQncyBub3Rlcy8wMV9kYXRhc2V0X25vdGVzLm1kCmZvciB0aGF0IGxvZ2ljLiBUaGlzIG1vZHVsZSBvbmx5IGFkZHMgd2hhdCdzIHNwZWNpZmljIHRvIHRoZQpkb21haW4tYWRhcHRhdGlvbiB0YXNrOiBtYXBwaW5nIENXUlUncyA1IGZhdWx0IGxhYmVscyBvbnRvIHRoZSB1bml2ZXJzYWwKMy1jbGFzcyBzcGFjZSBzaGFyZWQgd2l0aCBQYWRlcmJvcm4sIGFuZCB0YWdnaW5nIGVhY2ggZmlsZSB3aXRoIGEKdHJhaW4vdmFsIHNwbGl0IHJvbGUuCgpVbml2ZXJzYWwgbGFiZWwgc3BhY2UgKHNoYXJlZCB3aXRoIHB1X2xvYWRlci5weSk6CiAgICAwID0gaGVhbHRoeSwgMSA9IGlubmVyIHJhY2UgZmF1bHQsIDIgPSBvdXRlciByYWNlIGZhdWx0LgpCYWxsICgiQiIpIGZhdWx0cyBhcmUgRFJPUFBFRCBoZXJlIC0tIFBhZGVyYm9ybidzIHJlYWwtZGFtYWdlIGJlYXJpbmdzIGFyZQpvbmx5IGV2ZXIgaW5uZXItIG9yIG91dGVyLXJhY2UgKHNlZSBwdV9iZWFyaW5nX2NvZGVzLnB5KSwgc28gdGhlcmUncyBubwpjcm9zcy1kb21haW4gYmFsbC1mYXVsdCBjbGFzcyB0byBhbGlnbiBhZ2FpbnN0OyBrZWVwaW5nIENXUlUncyBiYWxsIGZhdWx0cwp3b3VsZCBqdXN0IGJlIHNvdXJjZS1vbmx5IG5vaXNlIHRoZSBjbGFzc2lmaWVyIGhhcyBub3doZXJlIHRvIHRyYW5zZmVyLgpTZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzICJSZXZpc2lvbnMiIHNlY3Rpb24uIENXUlUncyBvd24KYHZpYnJhdGlvbl9hZGAgZXhwZXJpbWVudCBpcyB1bmFmZmVjdGVkIC0tIGl0IHN0aWxsIHVzZXMgYWxsIGZhdWx0IHR5cGVzLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLmNhdGFsb2cgaW1wb3J0IGJ1aWxkX2NhdGFsb2cKClVOSVZFUlNBTF9MQUJFTFMgPSB7ImhlYWx0aHkiOiAwLCAiaW5uZXJfcmFjZSI6IDEsICJvdXRlcl9yYWNlIjogMn0KCl9GQVVMVF9UWVBFX1RPX1VOSVZFUlNBTCA9IHsKICAgICJOb3JtYWwiOiAiaGVhbHRoeSIsCiAgICAiSVIiOiAiaW5uZXJfcmFjZSIsCiAgICAiT1JAMyI6ICJvdXRlcl9yYWNlIiwKICAgICJPUkA2IjogIm91dGVyX3JhY2UiLAogICAgIk9SQDEyIjogIm91dGVyX3JhY2UiLAogICAgIyAiQiIgKGJhbGwpIGhhcyBubyBlbnRyeSBvbiBwdXJwb3NlIC0+IGRyb3BwZWQgYnkgdGhlIGlzaW4oKSBmaWx0ZXIgYmVsb3cuCn0KCgpkZWYgYnVpbGRfY3dydV9jYXRhbG9nKHJhd19kaXI6IFBhdGgsIHRyYWluX3JwbXM6IGxpc3RbaW50XSwgdmFsX3JwbTogaW50KSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJPbmUgcm93IHBlciBDV1JVIGZpbGUgd2l0aCBhIHVuaXZlcnNhbC1sYWJlbCBtYXBwaW5nIChiYWxsIGZhdWx0cwogICAgZHJvcHBlZCksIHNwbGl0ICd0cmFpbicvJ3ZhbCcgYnkgUlBNIGZhbWlseSAtLSBzYW1lIGFudGktbGVha2FnZQogICAgcmVhc29uaW5nIGFzIHZpYnJhdGlvbl9hZCdzIG93biBSUE0gaG9sZG91dDogYSByYW5kb20gc3BsaXQgd291bGQgbWl4CiAgICBuZWFyLWR1cGxpY2F0ZSBvdmVybGFwcGluZyB3aW5kb3dzIGZyb20gdGhlIHNhbWUgbG9uZyByZWNvcmRpbmcgYWNyb3NzCiAgICB0cmFpbiBhbmQgdmFsLgoKICAgIFJldHVybnMgY29sdW1uczogZmlsZXBhdGgsIGRvbWFpbiwgY29uZGl0aW9uLCBmc19uYXRpdmVfaHosIGZhdWx0X3R5cGUsCiAgICB1bml2ZXJzYWxfbGFiZWwsIHNwbGl0LgogICAgIiIiCiAgICBjYXRhbG9nID0gYnVpbGRfY2F0YWxvZyhQYXRoKHJhd19kaXIpKQogICAgY2F0YWxvZyA9IGNhdGFsb2dbY2F0YWxvZy5mYXVsdF90eXBlLmlzaW4oX0ZBVUxUX1RZUEVfVE9fVU5JVkVSU0FMKV0uY29weSgpCiAgICBpZiBjYXRhbG9nLmVtcHR5OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIk5vIENXUlUgZmlsZXMgd2l0aCBhIHVuaXZlcnNhbC1sYWJlbCBtYXBwaW5nIGZvdW5kIGluIHtyYXdfZGlyfSIpCgogICAgY2F0YWxvZ1sidW5pdmVyc2FsX2xhYmVsIl0gPSBjYXRhbG9nLmZhdWx0X3R5cGUubWFwKF9GQVVMVF9UWVBFX1RPX1VOSVZFUlNBTCkubWFwKFVOSVZFUlNBTF9MQUJFTFMpCiAgICBjYXRhbG9nWyJkb21haW4iXSA9ICJjd3J1IgogICAgY2F0YWxvZ1siZnNfbmF0aXZlX2h6Il0gPSAxMjAwMAogICAgY2F0YWxvZ1siY29uZGl0aW9uIl0gPSBjYXRhbG9nLnJwbS5hc3R5cGUoc3RyKQoKICAgIGtub3duX3JwbXMgPSBzZXQodHJhaW5fcnBtcykgfCB7dmFsX3JwbX0KICAgIHByZXNlbnRfcnBtcyA9IHNldChjYXRhbG9nLnJwbS51bmlxdWUoKS50b2xpc3QoKSkKICAgIHVua25vd24gPSBwcmVzZW50X3JwbXMgLSBrbm93bl9ycG1zCiAgICBpZiB1bmtub3duOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiQ1dSVSBjYXRhbG9nIGhhcyBSUE1zIG5vdCBjb3ZlcmVkIGJ5IHRyYWluX3JwbXMvdmFsX3JwbToge3NvcnRlZCh1bmtub3duKX0iCiAgICAgICAgKQogICAgaWYgdmFsX3JwbSBub3QgaW4gcHJlc2VudF9ycG1zOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ2YWxfcnBtPXt2YWxfcnBtfSBub3QgcHJlc2VudCBpbiBDV1JVIGNhdGFsb2cgUlBNcyB7c29ydGVkKHByZXNlbnRfcnBtcyl9IikKCiAgICBjYXRhbG9nWyJzcGxpdCJdID0gY2F0YWxvZy5ycG0uYXBwbHkobGFtYmRhIHI6ICJ2YWwiIGlmIHIgPT0gdmFsX3JwbSBlbHNlICJ0cmFpbiIpCiAgICBjb2xzID0gWyJmaWxlcGF0aCIsICJkb21haW4iLCAiY29uZGl0aW9uIiwgImZzX25hdGl2ZV9oeiIsICJmYXVsdF90eXBlIiwgInVuaXZlcnNhbF9sYWJlbCIsICJzcGxpdCJdCiAgICByZXR1cm4gY2F0YWxvZ1tjb2xzXS5yZXNldF9pbmRleChkcm9wPVRydWUpCg=='
    ),
    'src/domain_adaptation/data/pu_loader.py': (
        'IiIiUGFyc2UgdGhlIFBhZGVyYm9ybiBVbml2ZXJzaXR5IChQVSAvIEtBdC1EYXRhQ2VudGVyKSBiZWFyaW5nIGRhdGFzZXQuCgpMYXlvdXQgKGNvbmZpcm1lZCAyMDI2LTA5LTEwIGJ5IGRvd25sb2FkaW5nIDMgcmVhbCBmaWxlcyBvZmYgdGhlIGF0dGFjaGVkCkthZ2dsZSBtaXJyb3IsIGBkaXBwYXRlbDAzL3BhZGVyYm9ybi1kYmAgLS0gc2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQncwoiUmV2aXNpb25zIiBzZWN0aW9uKToKCiAgICA8cm9vdD4vPGJlYXJpbmdfY29kZT4vPGNvbmRpdGlvbj5fPGJlYXJpbmdfY29kZT5fPHJ1bj4ubWF0CgplLmcuIGBLMDAxL04wOV9NMDdfRjEwX0swMDFfMS5tYXRgLiBgY29uZGl0aW9uYCBlbmNvZGVzIHJvdGF0aW9uYWwgc3BlZWQKKE4wOT05MDBycG0sIE4xNT0xNTAwcnBtKSwgbG9hZCB0b3JxdWUgKE0wMT0wLjFObSwgTTA3PTAuN05tKSBhbmQgcmFkaWFsCmZvcmNlIChGMDQ9NDAwTiwgRjEwPTEwMDBOKSAtLSBzZWUgcHVfYmVhcmluZ19jb2Rlcy5weSBmb3Igd2hhdCBlYWNoCmBiZWFyaW5nX2NvZGVgIG1lYW5zIChoZWFsdGh5L2FydGlmaWNpYWwvcmVhbCwgaW5uZXIvb3V0ZXIgcmFjZSkuCgpFYWNoIGAubWF0YCBmaWxlIGlzIGEgTUFUTEFCIHN0cnVjdCAodG9wLWxldmVsIGtleSA9IHRoZSBmaWxlbmFtZSB3aXRoIHRoZQpleHRlbnNpb24gc3RyaXBwZWQpIHdpdGggYSBgWWAgZmllbGQ6IGEgbGVuZ3RoLTcgbGlzdCBvZiBwZXItY2hhbm5lbApkaWN0cywgZWFjaCBgeydOYW1lJzogc3RyLCAnRGF0YSc6IDFEIGFycmF5fWAuIENoYW5uZWxzIGFuZCB0aGVpcgphcHByb3hpbWF0ZSBuYXRpdmUgcmF0ZXMgKGZyb20gdGhlIHNhbWUgaW5zcGVjdGlvbik6IGZvcmNlL3NwZWVkL3RvcnF1ZQpAIH40a0h6LCBwaGFzZV9jdXJyZW50XzEvMiBAIH42NGtIeiwgdGVtcF8yX2JlYXJpbmdfbW9kdWxlIEAgfjEuMjVIeiAoNQpzYW1wbGVzIHBlciA0cyBydW4pLCB2aWJyYXRpb25fMSBAIH42NGtIeiAtLSBvbmx5IGB2aWJyYXRpb25fMWAgaXMgdXNlZApoZXJlLCBtYXRjaGluZyBDV1JVJ3Mgc2luZ2xlLWFjY2VsZXJvbWV0ZXItY2hhbm5lbCBzY29wZS4gUmF3IHJ1biBsZW5ndGhzCnZhcnkgYnkgYSBmZXcgaHVuZHJlZCBzYW1wbGVzIGFyb3VuZCB0aGUgbm9taW5hbCA0cy8yNTYwMDAgc2FtcGxlcyAoYSBEQVEKcXVpcmssIG5vdCBhIGJ1ZykgLS0gbm90aGluZyBkb3duc3RyZWFtIGFzc3VtZXMgYW4gZXhhY3QgbGVuZ3RoLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHNjaXB5LmlvCgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEucHVfYmVhcmluZ19jb2RlcyBpbXBvcnQgUFVfQkVBUklOR19DT0RFUwoKRklMRU5BTUVfUkUgPSByZS5jb21waWxlKHIiXig/UDxjb25kaXRpb24+TlxkezJ9X01cZHsyfV9GXGR7Mn0pXyg/UDxjb2RlPltBLVpdK1xkKylfKD9QPHJ1bj5cZCspXC5tYXQkIikKCgpkZWYgZmluZF9wdV9yb290KGxvY2FsX2hpbnQ6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gUGF0aDoKICAgICIiIkxvY2F0ZSB0aGUgUFUgZGF0YXNldCdzIHJvb3QgZGlyZWN0b3J5ICh0aGUgZm9sZGVyIHRoYXQgZGlyZWN0bHkKICAgIGNvbnRhaW5zIEswMDEvLCBLMDAyLywgLi4uIHN1YmZvbGRlcnMpLgoKICAgIE9uIEthZ2dsZSwgc2VhcmNoZXMgL2thZ2dsZS9pbnB1dCBmb3IgYSBmb2xkZXIgY29udGFpbmluZyBhIEswMDEvCiAgICBzdWJmb2xkZXIgLS0gYnkgY29udGVudCwgbm90IGEgZml4ZWQgcGF0aCwgYmVjYXVzZSBhdHRhY2hlZC1kYXRhc2V0CiAgICBtb3VudCBwYXRocyBoYXZlIHByb3ZlbiBpbmNvbnNpc3RlbnQgaW4gdGhpcyBwcm9qZWN0IGJlZm9yZSAoc2VlCiAgICB2aWJyYXRpb25fYWQncyBub3Rlcy8wN19kdmNfYW5kX2thZ2dsZS5tZCkuIEZhbGxzIGJhY2sgdG8KICAgIGBsb2NhbF9oaW50YCAoY29uZmlncy9kb21haW5fYWRhcHRhdGlvbi55YW1sJ3MgcGF0aHMucHVfcm9vdF9sb2NhbCkgZm9yCiAgICBsb2NhbCBkZXZlbG9wbWVudCBhZ2FpbnN0IGEgc21hbGwgaGFuZC1idWlsdCBzYW1wbGUuCiAgICAiIiIKICAgIGthZ2dsZV9pbnB1dCA9IFBhdGgoIi9rYWdnbGUvaW5wdXQiKQogICAgaWYga2FnZ2xlX2lucHV0LmlzX2RpcigpOgogICAgICAgIGNhbmRpZGF0ZXMgPSBzb3J0ZWQocC5wYXJlbnQgZm9yIHAgaW4ga2FnZ2xlX2lucHV0LnJnbG9iKCJLMDAxIikgaWYgcC5pc19kaXIoKSkKICAgICAgICBpZiBjYW5kaWRhdGVzOgogICAgICAgICAgICByZXR1cm4gY2FuZGlkYXRlc1swXQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIk5vIEswMDEvIGZvbGRlciBmb3VuZCB1bmRlciB7a2FnZ2xlX2lucHV0fSAtLSBpcyB0aGUgUFUgZGF0YXNldCAiCiAgICAgICAgICAgICJhdHRhY2hlZD8gKHNlZSBub3RlYm9va3MvZG9tYWluX2FkYXB0YXRpb24va2VybmVsLW1ldGFkYXRhLmpzb24ncyBkYXRhc2V0X3NvdXJjZXMpIgogICAgICAgICkKICAgIGlmIGxvY2FsX2hpbnQgaXMgbm90IE5vbmUgYW5kIFBhdGgobG9jYWxfaGludCkuaXNfZGlyKCk6CiAgICAgICAgcmV0dXJuIFBhdGgobG9jYWxfaGludCkKICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICJOb3QgcnVubmluZyBvbiBLYWdnbGUgKC9rYWdnbGUvaW5wdXQgbWlzc2luZykgYW5kIG5vIHVzYWJsZSBsb2NhbCAiCiAgICAgICAgZiJwdV9yb290ICh7bG9jYWxfaGludCFyfSkgLS0gcGFzcyAtLXB1LXJvb3QgZXhwbGljaXRseS4iCiAgICApCgoKY2xhc3MgVW5yZWFkYWJsZU1hdEZpbGUoUnVudGltZUVycm9yKToKICAgICIiIkEgUFUgLm1hdCBmaWxlIHNjaXB5IGNhbm5vdCBwYXJzZS4gUmFpc2VkIGluc3RlYWQgb2YgbGV0dGluZyBzY2lweSdzCiAgICBiYXJlIGBUeXBlRXJyb3I6IEV4cGVjdGluZyBtYXRyaXggaGVyZWAgZXNjYXBlIHdpdGggbm8gaW5kaWNhdGlvbiBvZgogICAgV0hJQ0ggb2YgfjE0MDAgZmlsZXMgd2FzIGF0IGZhdWx0IC0tIHdoaWNoIGlzIGV4YWN0bHkgaG93IGtlcm5lbCB2NQogICAgZGllZCBhZnRlciAzIG1pbnV0ZXMgd2l0aCBhbiB1bmF0dHJpYnV0YWJsZSB0cmFjZWJhY2suCiAgICAiIiIKCgpkZWYgbG9hZF92aWJyYXRpb25fc2lnbmFsKG1hdF9wYXRoOiBzdHIgfCBQYXRoLCBjaGFubmVsOiBzdHIgPSAidmlicmF0aW9uXzEiKSAtPiBucC5uZGFycmF5OgogICAgIiIiRXh0cmFjdCBvbmUgY2hhbm5lbCdzIDFEIGFycmF5IGZyb20gYSBQVSAubWF0IGZpbGUuIiIiCiAgICB0cnk6CiAgICAgICAgZGF0YSA9IHNjaXB5LmlvLmxvYWRtYXQoc3RyKG1hdF9wYXRoKSwgc2ltcGxpZnlfY2VsbHM9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBzY2lweSByYWlzZXMgYmFyZSBUeXBlRXJyb3IvVmFsdWVFcnJvciB3aXRoIG5vIGZpbGUgY29udGV4dAogICAgICAgIHJhaXNlIFVucmVhZGFibGVNYXRGaWxlKGYie21hdF9wYXRofTogc2NpcHkgY291bGQgbm90IHBhcnNlIHRoaXMgLm1hdCAoe3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9KSIpIGZyb20gZXhjCiAgICBrZXkgPSBuZXh0KGsgZm9yIGsgaW4gZGF0YSBpZiBub3Qgay5zdGFydHN3aXRoKCJfXyIpKQogICAgY2hhbm5lbHMgPSBkYXRhW2tleV1bIlkiXQogICAgZm9yIGNoIGluIGNoYW5uZWxzOgogICAgICAgIGlmIGNoWyJOYW1lIl0gPT0gY2hhbm5lbDoKICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkoY2hbIkRhdGEiXSwgZHR5cGU9bnAuZmxvYXQ2NCkucmF2ZWwoKQogICAgYXZhaWxhYmxlID0gW2NoWyJOYW1lIl0gZm9yIGNoIGluIGNoYW5uZWxzXQogICAgcmFpc2UgS2V5RXJyb3IoZiJ7bWF0X3BhdGh9OiBjaGFubmVsIHtjaGFubmVsIXJ9IG5vdCBmb3VuZCAoaGFzOiB7YXZhaWxhYmxlfSkiKQoKCmRlZiBidWlsZF9wdV9jYXRhbG9nKHJvb3RfZGlyOiBzdHIgfCBQYXRoLCBjb2RlczogbGlzdFtzdHJdLCBuYXRpdmVfZnNfaHo6IGludCA9IDY0MDAwKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJPbmUgcm93IHBlciBgPGNvZGU+Lzxjb25kaXRpb24+Xzxjb2RlPl88cnVuPi5tYXRgIGZpbGUgZm91bmQgdW5kZXIKICAgIGByb290X2RpcmAsIGZvciBldmVyeSBgY29kZWAgaW4gYGNvZGVzYCAoc2tpcHMgdGhlIHBlci1jb2RlIC5wZGYgZG9jcwogICAgYW5kIGFueXRoaW5nIHRoYXQgZG9lc24ndCBtYXRjaCB0aGUgbmFtaW5nIGNvbnZlbnRpb24gb3IgaGFzIG5vCiAgICB1bml2ZXJzYWwtbGFiZWwgbWFwcGluZywgZS5nLiBjb21iaW5lZC1kYW1hZ2UgY29kZXMgLS0gc2VlCiAgICBwdV9iZWFyaW5nX2NvZGVzLnB5KS4gUm93IG9yZGVyIGlzIGRldGVybWluaXN0aWMgKHNvcnRlZCBnbG9iKSwgbm90CiAgICBkaXNrIG9yZGVyLgoKICAgIGBuYXRpdmVfZnNfaHpgIGlzIGEgcGFyYW1ldGVyIChub3QgaGFyZGNvZGVkKSBzbyB0ZXN0cyBjYW4gZXhlcmNpc2UKICAgIHRoaXMgYWdhaW5zdCBzbWFsbCBzeW50aGV0aWMgc2lnbmFscyBhdCBhIGxvd2VyIHJhdGUgLS0gcHJvZHVjdGlvbgogICAgY2FsbGVycyBwYXNzIGNvbmZpZy5wdS5uYXRpdmVfZnNfaHogKDY0MDAwIGZvciB0aGUgcmVhbCBkYXRhc2V0KS4KCiAgICBSZXR1cm5zIGNvbHVtbnM6IGZpbGVwYXRoLCBjb2RlLCBjb25kaXRpb24sIHJ1biwgZmF1bHRfdHlwZSwKICAgIGRhbWFnZV9tb2RlLCB1bml2ZXJzYWxfbGFiZWwsIGZzX25hdGl2ZV9oeiwgZG9tYWluLgogICAgIiIiCiAgICByb290X2RpciA9IFBhdGgocm9vdF9kaXIpCiAgICByb3dzID0gW10KICAgIGZvciBjb2RlIGluIGNvZGVzOgogICAgICAgIGNvZGVfZGlyID0gcm9vdF9kaXIgLyBjb2RlCiAgICAgICAgaWYgbm90IGNvZGVfZGlyLmlzX2RpcigpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIlBVIGJlYXJpbmcgY29kZSBmb2xkZXIgbm90IGZvdW5kOiB7Y29kZV9kaXJ9IikKICAgICAgICBtZXRhID0gUFVfQkVBUklOR19DT0RFUy5nZXQoY29kZSkKICAgICAgICBpZiBtZXRhIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIEtleUVycm9yKGYiVW5rbm93biBQVSBiZWFyaW5nIGNvZGUge2NvZGUhcn0gLS0gbm90IGluIHB1X2JlYXJpbmdfY29kZXMuUFVfQkVBUklOR19DT0RFUyIpCiAgICAgICAgaWYgbWV0YVsidW5pdmVyc2FsX2xhYmVsIl0gaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUgICMgY29tYmluZWQtZGFtYWdlIGNvZGUsIGV4Y2x1ZGVkIGZyb20gdGhpcyB0YXNrIChzZWUgbW9kdWxlIGRvY3N0cmluZykKCiAgICAgICAgZm9yIG1hdF9wYXRoIGluIHNvcnRlZChjb2RlX2Rpci5nbG9iKCIqLm1hdCIpKToKICAgICAgICAgICAgbSA9IEZJTEVOQU1FX1JFLm1hdGNoKG1hdF9wYXRoLm5hbWUpCiAgICAgICAgICAgIGlmIG5vdCBtIG9yIG0uZ3JvdXAoImNvZGUiKSAhPSBjb2RlOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImZpbGVwYXRoIjogc3RyKG1hdF9wYXRoKSwKICAgICAgICAgICAgICAgICAgICAiY29kZSI6IGNvZGUsCiAgICAgICAgICAgICAgICAgICAgImNvbmRpdGlvbiI6IG0uZ3JvdXAoImNvbmRpdGlvbiIpLAogICAgICAgICAgICAgICAgICAgICJydW4iOiBpbnQobS5ncm91cCgicnVuIikpLAogICAgICAgICAgICAgICAgICAgICJmYXVsdF90eXBlIjogbWV0YVsiZmF1bHRfdHlwZSJdLAogICAgICAgICAgICAgICAgICAgICJkYW1hZ2VfbW9kZSI6IG1ldGFbImRhbWFnZV9tb2RlIl0sCiAgICAgICAgICAgICAgICAgICAgInVuaXZlcnNhbF9sYWJlbCI6IG1ldGFbInVuaXZlcnNhbF9sYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJmc19uYXRpdmVfaHoiOiBuYXRpdmVfZnNfaHosCiAgICAgICAgICAgICAgICAgICAgImRvbWFpbiI6ICJwdSIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIGlmIG5vdCByb3dzOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiTm8gbWF0Y2hpbmcgUFUgLm1hdCBmaWxlcyBmb3VuZCB1bmRlciB7cm9vdF9kaXJ9IGZvciBjb2RlcyB7Y29kZXN9IikKICAgIHJldHVybiBwZC5EYXRhRnJhbWUocm93cykK'
    ),
    'src/domain_adaptation/data/transforms.py': (
        'IiIiQ3Jvc3MtZG9tYWluIHNhbXBsZS1yYXRlIGFsaWdubWVudC4KCkNXUlUncyBEcml2ZS1FbmQgc2lnbmFsIGlzIGFscmVhZHkgMTJrSHo7IFBhZGVyYm9ybidzIHZpYnJhdGlvbl8xIGNoYW5uZWwKaXMgNjRrSHouIEEgQ05OIHRyYWluZWQgb24gbG9nLVNURlQtbWFnbml0dWRlIGltYWdlcyBuZWVkcyBwaXhlbHMgdGhhdAptZWFuIHRoZSBzYW1lIHBoeXNpY2FsIGZyZXF1ZW5jeSByYW5nZSBpbiBldmVyeSBkb21haW4sIHNvIGV2ZXJ5IHNpZ25hbCBpcwpyZXNhbXBsZWQgdG8gb25lIGNvbW1vbiByYXRlIChgY29uZmlnLmNvbW1vbi5mc19oemAsIGRlZmF1bHQgMTIwMDApIEJFRk9SRQp3aW5kb3dpbmcvU1RGVCAtLSBzZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzICJSZXZpc2lvbnMiIHNlY3Rpb24gZm9yCndoeSAxMmtIeiAoQ1dSVSdzIG5hdGl2ZSByYXRlKSB3YXMgY2hvc2VuIG92ZXIgdXAgc2FtcGxpbmcgZXZlcnl0aGluZyB0bwo0OGtIeiBhcyB0aGUgb3JpZ2luYWwgcGxhbiBwcm9wb3NlZC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZnJhY3Rpb25zIGltcG9ydCBGcmFjdGlvbgoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuc2lnbmFsIGltcG9ydCByZXNhbXBsZV9wb2x5CgoKZGVmIHJlc2FtcGxlX3RvX3JhdGUoc2lnbmFsOiBucC5uZGFycmF5LCBuYXRpdmVfZnNfaHo6IGludCwgdGFyZ2V0X2ZzX2h6OiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJQb2x5cGhhc2UgcmVzYW1wbGUgKHByb3BlciBhbnRpLWFsaWFzaW5nIGZpbHRlciwgdW5saWtlIGEgbmFpdmUKICAgIEZGVC1iaW4gcmVzYW1wbGUpIGZyb20gbmF0aXZlX2ZzX2h6IHRvIHRhcmdldF9mc19oei4KCiAgICBOby1vcHMgKHJldHVybnMgYHNpZ25hbGAgdW5jaGFuZ2VkLCBub3QgZXZlbiBjb3BpZWQpIHdoZW4gdGhlIHJhdGVzCiAgICBhbHJlYWR5IG1hdGNoIC0tIGUuZy4gQ1dSVSBhdCAxMmtIeiB3aGVuIHRhcmdldF9mc19oej0xMjAwMCAtLSBzbyBkYXRhCiAgICB0aGF0IGRvZXNuJ3QgbmVlZCByZXNhbXBsaW5nIG5ldmVyIHBpY2tzIHVwIGZpbHRlciBhcnRpZmFjdHMgZnJvbSBvbmUuCiAgICAiIiIKICAgIGlmIG5hdGl2ZV9mc19oeiA9PSB0YXJnZXRfZnNfaHo6CiAgICAgICAgcmV0dXJuIHNpZ25hbAogICAgcmF0aW8gPSBGcmFjdGlvbih0YXJnZXRfZnNfaHosIG5hdGl2ZV9mc19oeikubGltaXRfZGVub21pbmF0b3IoMTAwMCkKICAgIHJldHVybiByZXNhbXBsZV9wb2x5KHNpZ25hbCwgdXA9cmF0aW8ubnVtZXJhdG9yLCBkb3duPXJhdGlvLmRlbm9taW5hdG9yKS5hc3R5cGUoc2lnbmFsLmR0eXBlKQo='
    ),
    'src/domain_adaptation/data/dataset.py': (
        'IiIiQnVpbGQgdGhlIGZvdXIgZGF0YSBzcGxpdHMgZXZlcnkgZG93bnN0cmVhbSBzdGFnZSByZWFkczogU09VUkNFX1RSQUlOLApTT1VSQ0VfVkFMLCBUQVJHRVRfQURBUFQsIFRBUkdFVF9URVNULgoKRG9tYWluIGlkICh1c2VkIGJ5IHRoZSBEQU5OIGRpc2NyaW1pbmF0b3I7IHNvdXJjZV92YWwvdGFyZ2V0X3Rlc3Qgcm93cwpzdGlsbCBnZXQgb25lIGZvciBwbG90dGluZywganVzdCBuZXZlciBzZWUgdGhlIGRpc2NyaW1pbmF0b3IgbG9zcyk6CiAgICAwID0gY3dydSAgICAgICAgICAgIDEgPSBwdV9hcnRpZmljaWFsICAgICAgICAgICAgMiA9IHB1X3RhcmdldAoocHVfdGFyZ2V0IGNvdmVycyBCT1RIIHRoZSBoZWxkLW91dCBjb25kaXRpb24ncyBoZWFsdGh5IHJvd3MgYW5kIGl0cyByZWFsCmZhdWx0IHJvd3MgLS0gcGh5c2ljYWxseSB0aGUgc2FtZSByaWcvY29uZGl0aW9uLCBqdXN0IGRpZmZlcmVudCBiZWFyaW5nCmhlYWx0aCwgc28gdGhleSdyZSBvbmUgZG9tYWluIGVpdGhlciB3YXkuKQoKUm9sZSAtPiB3aGF0IGl0J3MgdXNlZCBmb3IgYW5kIHdoZXJlIGl0cyByb3dzIGNvbWUgZnJvbToKICAgIFNPVVJDRV9UUkFJTiAgc3VwZXJ2aXNlZCBjbGFzc2lmaWNhdGlvbiBsb3NzICsgZG9tYWluIGxvc3MgKERBTk4pLgogICAgICAgICAgICAgICAgICBDV1JVIHJvd3MgZnJvbSBjd3J1LnRyYWluX3JwbXMgKEFMTCBmYXVsdCB0eXBlcywgbGFiZWxzCiAgICAgICAgICAgICAgICAgIHVzZWQgLS0gdGhpcyBpcyBzdXBlcnZpc2VkLCB1bmxpa2UgdmlicmF0aW9uX2FkJ3MKICAgICAgICAgICAgICAgICAgaGVhbHRoeS1vbmx5IGF1dG9lbmNvZGVyKSArIFBVIHJvd3MgZnJvbSBwdS5zb3VyY2VfY29kZXMKICAgICAgICAgICAgICAgICAgdW5kZXIgcHUuc291cmNlX2NvbmRpdGlvbnMsIGZpcnN0CiAgICAgICAgICAgICAgICAgIHB1LnNvdXJjZV90cmFpbl9mcmFjdGlvbiBvZiBlYWNoIGJlYXJpbmcncyB+MjAgcnVucy4KICAgIFNPVVJDRV9WQUwgICAgc3VwZXJ2aXNlZCBsb3NzIG9ubHksIGZvciBlYXJseSBzdG9wcGluZyAvIGJhc2VsaW5lCiAgICAgICAgICAgICAgICAgIG1ldHJpYyAtLSBORVZFUiBzZWVuIGJ5IHRoZSBkb21haW4vYW5jaG9yIGxvc3MuIENXUlUncwogICAgICAgICAgICAgICAgICBoZWxkLW91dCBSUE0gKGN3cnUudmFsX3JwbSkgKyBQVSdzIHJlbWFpbmluZyBzb3VyY2UgcnVucy4KICAgIFRBUkdFVF9BREFQVCAgZG9tYWluIGxvc3MgKyBoZWFsdGh5LWFuY2hvciBsb3NzIE9OTFkgLS0gY2xhc3NpZmljYXRpb24KICAgICAgICAgICAgICAgICAgbGFiZWxzIGFyZSBuZXZlciB1c2VkIGhlcmUgZXZlbiB0aG91Z2ggdGhleSdyZSBrbm93bgogICAgICAgICAgICAgICAgICAoYWxsIGhlYWx0aHkpLiBQVSBoZWFsdGh5LWNvZGUgcm93cyB1bmRlcgogICAgICAgICAgICAgICAgICBwdS50YXJnZXRfY29uZGl0aW9uLCBmaXJzdAogICAgICAgICAgICAgICAgICBwdS50YXJnZXRfbm9ybWFsX2FkYXB0X2ZyYWN0aW9uIG9mIHJ1bnMuIFRoaXMgaXMgdGhlCiAgICAgICAgICAgICAgICAgIEVOVElSRSBhZGFwdGF0aW9uIHNpZ25hbDogbm8gZmF1bHQgZXhhbXBsZSwgbGFiZWxlZCBvcgogICAgICAgICAgICAgICAgICBub3QsIGZyb20gdGhlIHRhcmdldCBldmVyIGFwcGVhcnMgZHVyaW5nIHRyYWluaW5nLgogICAgVEFSR0VUX1RFU1QgICBldmFsdWF0ZS5weSBPTkxZIC0tIHRoZSB6ZXJvLXNob3QgY2hlY2suIFBVIGhlYWx0aHktY29kZQogICAgICAgICAgICAgICAgICByb3dzIHVuZGVyIHB1LnRhcmdldF9jb25kaXRpb24gTk9UIHVzZWQgaW4gVEFSR0VUX0FEQVBUCiAgICAgICAgICAgICAgICAgIChsYWJlbD0wKSwgcGx1cyBBTEwgb2YgcHUucmVhbF9jb2RlcyB1bmRlcgogICAgICAgICAgICAgICAgICBwdS50YXJnZXRfY29uZGl0aW9uIChsYWJlbD0xIG9yIDIsIHBlciBiZWFyaW5nIGNvZGUpLgoKU3BsaXQtYnktcnVuIChub3QgYnkgd2luZG93KSBmb3IgUFUsIHNhbWUgYW50aS1sZWFrYWdlIHNwaXJpdCBhcyBDV1JVJ3MKUlBNIGhvbGRvdXQgYW5kIHZpYnJhdGlvbl9hZCdzIGNvbnRpZ3VvdXMgdmFsL3Rlc3Qgc3BsaXQgb2Ygb3ZlcmxhcHBpbmcKd2luZG93czogYSBydW4gaXMgb25lIGNvbnRpbnVvdXMgfjRzIHJlY29yZGluZywgc28gc3BsaXR0aW5nIHdob2xlIHJ1bnMKKHJhdGhlciB0aGFuIHNodWZmbGluZyB3aW5kb3dzIGFjcm9zcyB0aGUgc3BsaXQpIGtlZXBzIG5lYXItZHVwbGljYXRlCm92ZXJsYXBwaW5nIHdpbmRvd3MgZnJvbSBhIHNpbmdsZSBydW4gb24gb25seSBvbmUgc2lkZS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCByZXNvdXJjZQppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gdmlicmF0aW9uX2FkLmRhdGEuZGF0YXNldCBpbXBvcnQgbG9hZF9zaWduYWwgYXMgbG9hZF9jd3J1X3NpZ25hbApmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLndpbmRvd3MgaW1wb3J0IG1ha2Vfd2luZG93cwpmcm9tIHZpYnJhdGlvbl9hZC5mZWF0dXJlcy5zcGVjdHJvZ3JhbSBpbXBvcnQgZXhwZWN0ZWRfc2hhcGUsIHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcwoKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5jb25maWcgaW1wb3J0IENvbmZpZwpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEuY3dydV9sb2FkZXIgaW1wb3J0IGJ1aWxkX2N3cnVfY2F0YWxvZwpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEucHVfbG9hZGVyIGltcG9ydCBVbnJlYWRhYmxlTWF0RmlsZSwgYnVpbGRfcHVfY2F0YWxvZywgbG9hZF92aWJyYXRpb25fc2lnbmFsCmZyb20gZG9tYWluX2FkYXB0YXRpb24uZGF0YS50cmFuc2Zvcm1zIGltcG9ydCByZXNhbXBsZV90b19yYXRlCgpET01BSU5fSURTID0geyJjd3J1IjogMCwgInB1X2FydGlmaWNpYWwiOiAxLCAicHVfdGFyZ2V0IjogMn0KUk9MRVMgPSAoInNvdXJjZV90cmFpbiIsICJzb3VyY2VfdmFsIiwgInRhcmdldF9hZGFwdCIsICJ0YXJnZXRfdGVzdCIpCgoKZGVmIF9zcGxpdF9wdV9ydW5zX2J5X2ZyYWN0aW9uKGRmOiBwZC5EYXRhRnJhbWUsIGZyYWN0aW9uOiBmbG9hdCkgLT4gdHVwbGVbcGQuRGF0YUZyYW1lLCBwZC5EYXRhRnJhbWVdOgogICAgIiIiU3BsaXQgZWFjaCBiZWFyaW5nIGNvZGUncyBydW5zIGludG8gYSBjb250aWd1b3VzIChmaXJzdF9wYXJ0LAogICAgc2Vjb25kX3BhcnQpIHBhaXIgYnkgcnVuIGluZGV4LCBpbmRlcGVuZGVudGx5IHBlciBjb2RlIChzbyBlLmcuIG9uZQogICAgY29kZSBoYXZpbmcgMTggcnVucyBhbmQgYW5vdGhlciAyMCBib3RoIHNwbGl0IHByb3BvcnRpb25hbGx5KS4KICAgICIiIgogICAgbl9ydW5zID0gZGYuZ3JvdXBieSgiY29kZSIpWyJydW4iXS50cmFuc2Zvcm0oIm51bmlxdWUiKQogICAgY3V0b2ZmID0gKG5fcnVucyAqIGZyYWN0aW9uKS5yb3VuZCgpCiAgICBmaXJzdCA9IGRmW2RmWyJydW4iXSA8PSBjdXRvZmZdCiAgICBzZWNvbmQgPSBkZltkZlsicnVuIl0gPiBjdXRvZmZdCiAgICByZXR1cm4gZmlyc3QuY29weSgpLCBzZWNvbmQuY29weSgpCgoKZGVmIF9hc3NpZ25fcHVfcm9sZXMocHVfY2F0YWxvZzogcGQuRGF0YUZyYW1lLCBjb25maWc6IENvbmZpZykgLT4gcGQuRGF0YUZyYW1lOgogICAgcHVfY2ZnID0gY29uZmlnLnB1CiAgICBzb3VyY2VfbWFzayA9IHB1X2NhdGFsb2cuY29kZS5pc2luKHB1X2NmZy5zb3VyY2VfY29kZXMpICYgcHVfY2F0YWxvZy5jb25kaXRpb24uaXNpbihwdV9jZmcuc291cmNlX2NvbmRpdGlvbnMpCiAgICB0YXJnZXRfaGVhbHRoeV9tYXNrID0gcHVfY2F0YWxvZy5jb2RlLmlzaW4ocHVfY2ZnLmhlYWx0aHlfY29kZXMpICYgKHB1X2NhdGFsb2cuY29uZGl0aW9uID09IHB1X2NmZy50YXJnZXRfY29uZGl0aW9uKQogICAgdGFyZ2V0X3JlYWxfbWFzayA9IHB1X2NhdGFsb2cuY29kZS5pc2luKHB1X2NmZy5yZWFsX2NvZGVzKSAmIChwdV9jYXRhbG9nLmNvbmRpdGlvbiA9PSBwdV9jZmcudGFyZ2V0X2NvbmRpdGlvbikKCiAgICBwYXJ0cyA9IFtdCgogICAgc291cmNlX3RyYWluLCBzb3VyY2VfdmFsID0gX3NwbGl0X3B1X3J1bnNfYnlfZnJhY3Rpb24ocHVfY2F0YWxvZ1tzb3VyY2VfbWFza10sIHB1X2NmZy5zb3VyY2VfdHJhaW5fZnJhY3Rpb24pCiAgICBzb3VyY2VfdHJhaW5bInJvbGUiXSwgc291cmNlX3RyYWluWyJkb21haW4iXSA9ICJzb3VyY2VfdHJhaW4iLCAicHVfYXJ0aWZpY2lhbCIKICAgIHNvdXJjZV92YWxbInJvbGUiXSwgc291cmNlX3ZhbFsiZG9tYWluIl0gPSAic291cmNlX3ZhbCIsICJwdV9hcnRpZmljaWFsIgogICAgcGFydHMgKz0gW3NvdXJjZV90cmFpbiwgc291cmNlX3ZhbF0KCiAgICB0YXJnZXRfYWRhcHQsIHRhcmdldF9oZWFsdGh5X3Rlc3QgPSBfc3BsaXRfcHVfcnVuc19ieV9mcmFjdGlvbigKICAgICAgICBwdV9jYXRhbG9nW3RhcmdldF9oZWFsdGh5X21hc2tdLCBwdV9jZmcudGFyZ2V0X25vcm1hbF9hZGFwdF9mcmFjdGlvbgogICAgKQogICAgdGFyZ2V0X2FkYXB0WyJyb2xlIl0sIHRhcmdldF9hZGFwdFsiZG9tYWluIl0gPSAidGFyZ2V0X2FkYXB0IiwgInB1X3RhcmdldCIKICAgIHRhcmdldF9oZWFsdGh5X3Rlc3RbInJvbGUiXSwgdGFyZ2V0X2hlYWx0aHlfdGVzdFsiZG9tYWluIl0gPSAidGFyZ2V0X3Rlc3QiLCAicHVfdGFyZ2V0IgogICAgcGFydHMgKz0gW3RhcmdldF9hZGFwdCwgdGFyZ2V0X2hlYWx0aHlfdGVzdF0KCiAgICB0YXJnZXRfcmVhbF90ZXN0ID0gcHVfY2F0YWxvZ1t0YXJnZXRfcmVhbF9tYXNrXS5jb3B5KCkKICAgIHRhcmdldF9yZWFsX3Rlc3RbInJvbGUiXSwgdGFyZ2V0X3JlYWxfdGVzdFsiZG9tYWluIl0gPSAidGFyZ2V0X3Rlc3QiLCAicHVfdGFyZ2V0IgogICAgcGFydHMuYXBwZW5kKHRhcmdldF9yZWFsX3Rlc3QpCgogICAgYXNzaWduZWQgPSBwZC5jb25jYXQocGFydHMsIGlnbm9yZV9pbmRleD1UcnVlKQogICAgZm9yIHJvbGUgaW4gUk9MRVM6CiAgICAgICAgaWYgYXNzaWduZWRbYXNzaWduZWQucm9sZSA9PSByb2xlXS5lbXB0eToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiUFUgcm9sZSBhc3NpZ25tZW50IHByb2R1Y2VkIHplcm8gcm93cyBmb3Ige3JvbGUhcn0gLS0gY2hlY2sgY29uZmlnLnB1J3MgY29kZS9jb25kaXRpb24gbGlzdHMuIikKICAgIHJldHVybiBhc3NpZ25lZAoKCmRlZiBfc3BlY3Ryb2dyYW1zX2FuZF9tZXRhX2Zvcl9jd3J1X3Jvd3Mocm93czogcGQuRGF0YUZyYW1lLCBjb25maWc6IENvbmZpZywgc3RyaWRlOiBpbnQpIC0+IHR1cGxlW25wLm5kYXJyYXksIHBkLkRhdGFGcmFtZV06CiAgICAiIiJTVEZUIGlzIGFwcGxpZWQgUEVSIEZJTEUgYW5kIHRoZSByYXcgd2luZG93cyBkcm9wcGVkIGltbWVkaWF0ZWx5LgoKICAgIERvaW5nIGl0IG9uY2Ugb3ZlciB0aGUgd2hvbGUgc3BsaXQgaW5zdGVhZCAtLSB3aGljaCBpcyB3aGF0IHRoaXMgZGlkCiAgICB1bnRpbCBrZXJuZWwgdjYgLS0gbWFrZXMgc2NpcHkgaG9sZCBhIGNvbXBsZXgxMjggaW50ZXJtZWRpYXRlIHNpemVkCiAgICAobl93aW5kb3dzLCBucGVyc2VnLzIrMSwgbl9mcmFtZXMpIHggMTYgYnl0ZXMuIEF0IDY5LDI4MCB3aW5kb3dzIHRoYXQgaXMKICAgIDE3IEdCIG9uIGl0cyBvd24sIGFuZCB0aGUgd2hvbGUgYnVpbGQgcGVha2VkIG5lYXIgMzQgR0IgYWdhaW5zdCBLYWdnbGUncwogICAgfjMwIEdCLCBzbyB0aGUgcHJvY2VzcyB3YXMgU0lHS0lMTGVkLiBQZXIgZmlsZSBpdCBpcyBhIGZldyBodW5kcmVkIEtCIGF0CiAgICBhIHRpbWUgYW5kIHBlYWsgbWVtb3J5IGJlY29tZXMganVzdCB0aGUgZmxvYXQzMiBvdXRwdXQuCiAgICAiIiIKICAgIGNodW5rcywgbWV0YV9yb3dzID0gW10sIFtdCiAgICBmb3IgXywgcm93IGluIHJvd3MuaXRlcnJvd3MoKToKICAgICAgICBzaWduYWwgPSBsb2FkX2N3cnVfc2lnbmFsKHJvd1siZmlsZXBhdGgiXSwgY29uZmlnLmN3cnUuc2Vuc29yKSAgIyBhbHJlYWR5IDEya0h6ID09IGNvbW1vbi5mc19oegogICAgICAgIHcgPSBtYWtlX3dpbmRvd3Moc2lnbmFsLCBjb25maWcuY29tbW9uLnNlZ21lbnRfbGVuZ3RoLCBzdHJpZGUpCiAgICAgICAgaWYgbGVuKHcpID09IDA6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY2h1bmtzLmFwcGVuZCh3aW5kb3dzX3RvX2xvZ19zcGVjdHJvZ3JhbXModywgY29uZmlnLmNvbW1vbi5mc19oeiwgY29uZmlnLnN0ZnQpKQogICAgICAgIGZvciBfIGluIHJhbmdlKGxlbih3KSk6CiAgICAgICAgICAgIG1ldGFfcm93cy5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImRvbWFpbiI6IHJvd1siZG9tYWluIl0sCiAgICAgICAgICAgICAgICAgICAgImRvbWFpbl9pZCI6IERPTUFJTl9JRFNbImN3cnUiXSwKICAgICAgICAgICAgICAgICAgICAicm9sZSI6IHJvd1sicm9sZSJdLAogICAgICAgICAgICAgICAgICAgICJ1bml2ZXJzYWxfbGFiZWwiOiByb3dbInVuaXZlcnNhbF9sYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJjb25kaXRpb24iOiByb3dbImNvbmRpdGlvbiJdLAogICAgICAgICAgICAgICAgICAgICJjb2RlIjogIkNXUlUiLAogICAgICAgICAgICAgICAgICAgICJzb3VyY2VfZmlsZSI6IFBhdGgocm93WyJmaWxlcGF0aCJdKS5uYW1lLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICBpZiBub3QgY2h1bmtzOgogICAgICAgIHJldHVybiBucC5lbXB0eSgoMCwgKmV4cGVjdGVkX3NoYXBlKGNvbmZpZy5jb21tb24sIGNvbmZpZy5zdGZ0KSksIGR0eXBlPW5wLmZsb2F0MzIpLCBwZC5EYXRhRnJhbWUobWV0YV9yb3dzKQogICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKGNodW5rcywgYXhpcz0wKSwgcGQuRGF0YUZyYW1lKG1ldGFfcm93cykKCgojIEEgZmV3IHVucmVhZGFibGUgZmlsZXMgb3V0IG9mIH4xNDAwIGlzIGEgZmxhdyBpbiB0aGUgS2FnZ2xlIG1pcnJvciwgbm90IGEKIyByZWFzb24gdG8gdGhyb3cgYXdheSBhIDEwMC1taW51dGUgcnVuOyBhIExPVCBvZiB0aGVtIG1lYW5zIHNvbWV0aGluZwojIHN5c3RlbWljICh3cm9uZyBkYXRhc2V0IGF0dGFjaGVkLCB0cnVuY2F0ZWQgbW91bnQpIHRoYXQgbXVzdCBub3QgYmUKIyBzaWxlbnRseSB0cmFpbmVkIHRocm91Z2guIFNhbWUgcG9zdHVyZSB0aGUgQ1dSVSBkb3dubG9hZGVyIGxhbmRlZCBvbiAtLQojIHNlZSB2aWJyYXRpb25fYWQncyBub3Rlcy8wN19kdmNfYW5kX2thZ2dsZS5tZCwgd2hlcmUgdGhlIG9wcG9zaXRlIGJ1ZwojIChzd2FsbG93aW5nIGEgdG90YWwgZmFpbHVyZSkgaGlkIGEgYnJva2VuIHJ1biBmb3IgMzUgbWludXRlcy4KIyBDYWxpYnJhdGlvbiBtYXR0ZXJzIGhlcmUuIEF0IH4xMDAwIGZpbGVzIHBlciBzcGxpdCwgb25lIGVudGlyZWx5LWNvcnJ1cHQKIyBiZWFyaW5nIGNvZGUgaXMgMyBjb25kaXRpb25zIHggMjAgcnVucyA9IDYwIGZpbGVzID0gfjYlLCB3aGljaCBpcyBOT1QKIyBzeXN0ZW1pYyAtLSBpdCBjb3N0cyAxIG9mIDE4IGNvZGVzJyBkaXZlcnNpdHkgYW5kIHNob3VsZCBwcm9jZWVkIGxvdWRseSwKIyBub3QgYWJvcnQuIEEgYnJva2VuIG1vdW50IG9yIHdyb25nIGRhdGFzZXQgaXMgZmFyIGFib3ZlIHRoYXQuIDEwJSBzaXRzCiMgYmV0d2VlbiB0aGUgdHdvLgpNQVhfVU5SRUFEQUJMRV9GUkFDVElPTiA9IDAuMTAKCgpkZWYgX3NwZWN0cm9ncmFtc19hbmRfbWV0YV9mb3JfcHVfcm93cyhyb3dzOiBwZC5EYXRhRnJhbWUsIGNvbmZpZzogQ29uZmlnLCBzdHJpZGU6IGludCkgLT4gdHVwbGVbbnAubmRhcnJheSwgcGQuRGF0YUZyYW1lXToKICAgICIiIlBlci1maWxlIFNURlQsIHNhbWUgbWVtb3J5IHJlYXNvbmluZyBhcyB0aGUgQ1dSVSBoZWxwZXIgYWJvdmUuIiIiCiAgICBjaHVua3MsIG1ldGFfcm93cyA9IFtdLCBbXQogICAgdW5yZWFkYWJsZTogbGlzdFtzdHJdID0gW10KICAgIGZvciBfLCByb3cgaW4gcm93cy5pdGVycm93cygpOgogICAgICAgIHRyeToKICAgICAgICAgICAgc2lnbmFsID0gbG9hZF92aWJyYXRpb25fc2lnbmFsKHJvd1siZmlsZXBhdGgiXSwgY29uZmlnLnB1LnZpYnJhdGlvbl9jaGFubmVsKQogICAgICAgIGV4Y2VwdCBVbnJlYWRhYmxlTWF0RmlsZSBhcyBleGM6CiAgICAgICAgICAgIHVucmVhZGFibGUuYXBwZW5kKHN0cihyb3dbImZpbGVwYXRoIl0pKQogICAgICAgICAgICBwcmludChmIiAgU0tJUFBJTkcgdW5yZWFkYWJsZSBmaWxlOiB7ZXhjfSIpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgc2lnbmFsID0gcmVzYW1wbGVfdG9fcmF0ZShzaWduYWwsIHJvd1siZnNfbmF0aXZlX2h6Il0sIGNvbmZpZy5jb21tb24uZnNfaHopCiAgICAgICAgdyA9IG1ha2Vfd2luZG93cyhzaWduYWwsIGNvbmZpZy5jb21tb24uc2VnbWVudF9sZW5ndGgsIHN0cmlkZSkKICAgICAgICBpZiBsZW4odykgPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjaHVua3MuYXBwZW5kKHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcyh3LCBjb25maWcuY29tbW9uLmZzX2h6LCBjb25maWcuc3RmdCkpCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UobGVuKHcpKToKICAgICAgICAgICAgbWV0YV9yb3dzLmFwcGVuZCgKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAiZG9tYWluIjogcm93WyJkb21haW4iXSwKICAgICAgICAgICAgICAgICAgICAiZG9tYWluX2lkIjogRE9NQUlOX0lEU1tyb3dbImRvbWFpbiJdXSwKICAgICAgICAgICAgICAgICAgICAicm9sZSI6IHJvd1sicm9sZSJdLAogICAgICAgICAgICAgICAgICAgICJ1bml2ZXJzYWxfbGFiZWwiOiByb3dbInVuaXZlcnNhbF9sYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJjb25kaXRpb24iOiByb3dbImNvbmRpdGlvbiJdLAogICAgICAgICAgICAgICAgICAgICJjb2RlIjogcm93WyJjb2RlIl0sCiAgICAgICAgICAgICAgICAgICAgInNvdXJjZV9maWxlIjogUGF0aChyb3dbImZpbGVwYXRoIl0pLm5hbWUsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgIGlmIHVucmVhZGFibGU6CiAgICAgICAgZnJhY3Rpb24gPSBsZW4odW5yZWFkYWJsZSkgLyBtYXgobGVuKHJvd3MpLCAxKQogICAgICAgICMgUGVyLWNvZGUgYnJlYWtkb3duOiAiNjAgZmlsZXMgZmFpbGVkIiBpcyBhbWJpZ3VvdXMgYmV0d2VlbiBvbmUgZGVhZAogICAgICAgICMgYmVhcmluZyBjb2RlIGFuZCBzY2F0dGVyZWQgcm90LCBhbmQgdGhvc2UgbmVlZCBkaWZmZXJlbnQgcmVzcG9uc2VzLgogICAgICAgIGJ5X2NvZGU6IGRpY3Rbc3RyLCBpbnRdID0ge30KICAgICAgICBmb3IgcGF0aCBpbiB1bnJlYWRhYmxlOgogICAgICAgICAgICBieV9jb2RlW1BhdGgocGF0aCkucGFyZW50Lm5hbWVdID0gYnlfY29kZS5nZXQoUGF0aChwYXRoKS5wYXJlbnQubmFtZSwgMCkgKyAxCiAgICAgICAgcGVyX2NvZGUgPSAiLCAiLmpvaW4oZiJ7Y29kZX09e259IiBmb3IgY29kZSwgbiBpbiBzb3J0ZWQoYnlfY29kZS5pdGVtcygpKSkKICAgICAgICBwcmludChmIiAge2xlbih1bnJlYWRhYmxlKX0ve2xlbihyb3dzKX0gUFUgZmlsZXMgdW5yZWFkYWJsZSAoe2ZyYWN0aW9uOi4xJX0pIGluIHRoaXMgc3BsaXQgLS0gYnkgY29kZToge3Blcl9jb2RlfSIpCiAgICAgICAgaWYgZnJhY3Rpb24gPiBNQVhfVU5SRUFEQUJMRV9GUkFDVElPTjoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKAogICAgICAgICAgICAgICAgZiJ7bGVuKHVucmVhZGFibGUpfSBvZiB7bGVuKHJvd3MpfSBQVSAubWF0IGZpbGVzIHVucmVhZGFibGUgKHtmcmFjdGlvbjouMSV9LCAiCiAgICAgICAgICAgICAgICBmImxpbWl0IHtNQVhfVU5SRUFEQUJMRV9GUkFDVElPTjouMCV9KSAtLSB0aGF0IGlzIHN5c3RlbWljLCBub3QgYSBmZXcgYmFkIGZpbGVzLiAiCiAgICAgICAgICAgICAgICBmIkZpcnN0IGZldzoge3VucmVhZGFibGVbOjVdfSIKICAgICAgICAgICAgKQogICAgaWYgbm90IGNodW5rczoKICAgICAgICByZXR1cm4gbnAuZW1wdHkoKDAsICpleHBlY3RlZF9zaGFwZShjb25maWcuY29tbW9uLCBjb25maWcuc3RmdCkpLCBkdHlwZT1ucC5mbG9hdDMyKSwgcGQuRGF0YUZyYW1lKG1ldGFfcm93cykKICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShjaHVua3MsIGF4aXM9MCksIHBkLkRhdGFGcmFtZShtZXRhX3Jvd3MpCgoKZGVmIF9wZWFrX3Jzc19nYigpIC0+IGZsb2F0OgogICAgIiIiUGVhayByZXNpZGVudCBtZW1vcnkgc28gZmFyLCBpbiBHQi4KCiAgICBydV9tYXhyc3MgdW5pdHMgYXJlIHBsYXRmb3JtLWRlcGVuZGVudDogS0Igb24gTGludXggKHdoaWNoIGlzIHdoYXQKICAgIEthZ2dsZSBydW5zKSwgQllURVMgb24gbWFjT1MuIEdldHRpbmcgdGhpcyB3cm9uZyBtYWtlcyB0aGUgbG9jYWwKICAgIHJlYWRvdXQgb2ZmIGJ5IDEwMjR4IC0tIGEgbWlzbGVhZGluZyBnYXVnZSBpcyB3b3JzZSB0aGFuIG5vbmUgZHVyaW5nIGEKICAgIG1lbW9yeSBpbnZlc3RpZ2F0aW9uLCB3aGljaCBpcyBleGFjdGx5IHdoYXQgdGhpcyB3YXMgYWRkZWQgZm9yLgogICAgIiIiCiAgICBtYXhyc3MgPSByZXNvdXJjZS5nZXRydXNhZ2UocmVzb3VyY2UuUlVTQUdFX1NFTEYpLnJ1X21heHJzcwogICAgZGl2aXNvciA9IDEwMjQgKiogMyBpZiBzeXMucGxhdGZvcm0gPT0gImRhcndpbiIgZWxzZSAxMDI0ICoqIDIKICAgIHJldHVybiBtYXhyc3MgLyBkaXZpc29yCgoKZGVmIF9jYWNoZV9wYXRocyhwcm9jZXNzZWRfZGlyOiBQYXRoKSAtPiBkaWN0W3N0ciwgUGF0aF06CiAgICBwYXRocyA9IHt9CiAgICBmb3Igcm9sZSBpbiBST0xFUzoKICAgICAgICBwYXRoc1tmIntyb2xlfV9YIl0gPSBwcm9jZXNzZWRfZGlyIC8gZiJ7cm9sZX1fc3BlY3Ryb2dyYW1zLm5weSIKICAgICAgICBwYXRoc1tmIntyb2xlfV9tZXRhIl0gPSBwcm9jZXNzZWRfZGlyIC8gZiJ7cm9sZX1fbWV0YS5jc3YiCiAgICByZXR1cm4gcGF0aHMKCgpkZWYgYnVpbGRfZG9tYWluX2FkYXB0YXRpb25fZGF0YXNldHMoCiAgICBjb25maWc6IENvbmZpZywgcHVfcm9vdDogc3RyIHwgUGF0aCwgZm9yY2U6IGJvb2wgPSBGYWxzZQopIC0+IGRpY3Rbc3RyLCB0dXBsZVtucC5uZGFycmF5LCBwZC5EYXRhRnJhbWVdXToKICAgICIiIlJldHVybnMge3JvbGU6IChYLCBtZXRhKX0gZm9yIHJvbGUgaW4gUk9MRVMuCgogICAgWDogKE4sIEYsIFQpIGZsb2F0MzIgdW4tbm9ybWFsaXplZCBsb2ctbWFnbml0dWRlIHNwZWN0cm9ncmFtcyAoc2VlCiAgICBmZWF0dXJlcy9zcGVjdHJvZ3JhbS5leHBlY3RlZF9zaGFwZSB2aWEgY29uZmlnLmNvbW1vbi9jb25maWcuc3RmdCkuCiAgICBtZXRhOiBvbmUgcm93IHBlciB3aW5kb3cgLS0gZG9tYWluLCBkb21haW5faWQsIHJvbGUsIHVuaXZlcnNhbF9sYWJlbCwKICAgIGNvbmRpdGlvbiwgY29kZSwgc291cmNlX2ZpbGUuCgogICAgYHB1X3Jvb3RgIGlzIHBhc3NlZCBleHBsaWNpdGx5IChub3QgcmVhZCBmcm9tIGNvbmZpZykgYmVjYXVzZSBpdCdzIGEKICAgIHJ1bnRpbWUgZmFjdCAtLSB0aGUgS2FnZ2xlLW1vdW50ZWQgZGF0YXNldCBwYXRoIG9yIGEgbG9jYWwgZGV2CiAgICBkaXJlY3RvcnkgLS0gc2VlIGRhdGEvcHVfbG9hZGVyLmZpbmRfcHVfcm9vdCgpLgogICAgIiIiCiAgICBwcm9jZXNzZWRfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnByb2Nlc3NlZF9kaXIpCiAgICBwcm9jZXNzZWRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGhzID0gX2NhY2hlX3BhdGhzKHByb2Nlc3NlZF9kaXIpCgogICAgaWYgbm90IGZvcmNlIGFuZCBhbGwocC5leGlzdHMoKSBmb3IgcCBpbiBwYXRocy52YWx1ZXMoKSk6CiAgICAgICAgcmV0dXJuIHtyb2xlOiAobnAubG9hZChwYXRoc1tmIntyb2xlfV9YIl0pLCBwZC5yZWFkX2NzdihwYXRoc1tmIntyb2xlfV9tZXRhIl0pKSBmb3Igcm9sZSBpbiBST0xFU30KCiAgICBpbmNsdWRlX2N3cnUgPSBjb25maWcuY3dydS5pbmNsdWRlX2luX3NvdXJjZQogICAgY3dydV9jYXRhbG9nID0gKAogICAgICAgIGJ1aWxkX2N3cnVfY2F0YWxvZyhjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMuY3dydV9yYXdfZGlyKSwgY29uZmlnLmN3cnUudHJhaW5fcnBtcywgY29uZmlnLmN3cnUudmFsX3JwbSkKICAgICAgICBpZiBpbmNsdWRlX2N3cnUKICAgICAgICBlbHNlIE5vbmUKICAgICkKICAgIGlmIG5vdCBpbmNsdWRlX2N3cnU6CiAgICAgICAgcHJpbnQoIkNXUlUgZXhjbHVkZWQgZnJvbSB0aGUgc291cmNlIHBvb2wgKGNvbmZpZy5jd3J1LmluY2x1ZGVfaW5fc291cmNlPWZhbHNlKSAtLSBQVS1vbmx5IHJ1bi4iKQogICAgcHVfY2F0YWxvZyA9IF9hc3NpZ25fcHVfcm9sZXMoCiAgICAgICAgYnVpbGRfcHVfY2F0YWxvZyhwdV9yb290LCBjb25maWcucHUuc291cmNlX2NvZGVzICsgY29uZmlnLnB1LnJlYWxfY29kZXMsIGNvbmZpZy5wdS5uYXRpdmVfZnNfaHopLCBjb25maWcKICAgICkKCiAgICAjIENXUlUgb25seSBldmVyIGNvbnRyaWJ1dGVzIHRvIHNvdXJjZV90cmFpbi9zb3VyY2VfdmFsIC0tIGl0IGlzbid0IHRoZQogICAgIyB0YXJnZXQgZG9tYWluLCBzbyB0YXJnZXRfYWRhcHQvdGFyZ2V0X3Rlc3QgYXJlIFBVLW9ubHkgYmVsb3cuCiAgICBjd3J1X3NwbGl0X2Zvcl9yb2xlID0geyJzb3VyY2VfdHJhaW4iOiAidHJhaW4iLCAic291cmNlX3ZhbCI6ICJ2YWwifQogICAgZW1wdHlfc3BlY3Ryb2dyYW1zID0gbnAuZW1wdHkoKDAsICpleHBlY3RlZF9zaGFwZShjb25maWcuY29tbW9uLCBjb25maWcuc3RmdCkpLCBkdHlwZT1ucC5mbG9hdDMyKQoKICAgIHJlc3VsdHM6IGRpY3Rbc3RyLCB0dXBsZVtucC5uZGFycmF5LCBwZC5EYXRhRnJhbWVdXSA9IHt9CiAgICBmb3Igcm9sZSBpbiBST0xFUzoKICAgICAgICBzdHJpZGUgPSBjb25maWcuY29tbW9uLnRyYWluX3N0cmlkZSBpZiByb2xlIGluICgic291cmNlX3RyYWluIiwgInRhcmdldF9hZGFwdCIpIGVsc2UgY29uZmlnLmNvbW1vbi5ldmFsX3N0cmlkZQoKICAgICAgICBpZiBpbmNsdWRlX2N3cnUgYW5kIHJvbGUgaW4gY3dydV9zcGxpdF9mb3Jfcm9sZToKICAgICAgICAgICAgY3dydV9yb3dzID0gY3dydV9jYXRhbG9nW2N3cnVfY2F0YWxvZy5zcGxpdCA9PSBjd3J1X3NwbGl0X2Zvcl9yb2xlW3JvbGVdXS5jb3B5KCkKICAgICAgICAgICAgY3dydV9yb3dzWyJyb2xlIl0gPSByb2xlCiAgICAgICAgICAgIGN3cnVfWCwgY3dydV9tZXRhID0gX3NwZWN0cm9ncmFtc19hbmRfbWV0YV9mb3JfY3dydV9yb3dzKGN3cnVfcm93cywgY29uZmlnLCBzdHJpZGUpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY3dydV9YLCBjd3J1X21ldGEgPSBlbXB0eV9zcGVjdHJvZ3JhbXMsIHBkLkRhdGFGcmFtZSgpCgogICAgICAgIHB1X1gsIHB1X21ldGEgPSBfc3BlY3Ryb2dyYW1zX2FuZF9tZXRhX2Zvcl9wdV9yb3dzKHB1X2NhdGFsb2dbcHVfY2F0YWxvZy5yb2xlID09IHJvbGVdLCBjb25maWcsIHN0cmlkZSkKCiAgICAgICAgc3BlY3Ryb2dyYW1zID0gbnAuY29uY2F0ZW5hdGUoW2N3cnVfWCwgcHVfWF0sIGF4aXM9MCkKICAgICAgICBkZWwgY3dydV9YLCBwdV9YICAjIHJlbGVhc2UgcGVyLWRvbWFpbiBjb3BpZXMgYmVmb3JlIHRoZSBuZXh0IHJvbGUKICAgICAgICBtZXRhID0gcGQuY29uY2F0KFtjd3J1X21ldGEsIHB1X21ldGFdLCBheGlzPTAsIGlnbm9yZV9pbmRleD1UcnVlKQogICAgICAgIGlmIGxlbihzcGVjdHJvZ3JhbXMpID09IDA6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJvbGUge3JvbGUhcn0gcHJvZHVjZWQgemVybyB3aW5kb3dzIC0tIGNoZWNrIGNvbmZpZy4iKQoKICAgICAgICBwcmludChmIiAge3JvbGV9OiB7c3BlY3Ryb2dyYW1zLnNoYXBlfSAoe3NwZWN0cm9ncmFtcy5uYnl0ZXMgLyAxMDI0KiozOi4xZn0gR0IpICBwZWFrIFJTUyB7X3BlYWtfcnNzX2diKCk6LjFmfSBHQiIpCiAgICAgICAgcmVzdWx0c1tyb2xlXSA9IChzcGVjdHJvZ3JhbXMsIG1ldGEpCgogICAgICAgIG5wLnNhdmUocGF0aHNbZiJ7cm9sZX1fWCJdLCBzcGVjdHJvZ3JhbXMpCiAgICAgICAgbWV0YS50b19jc3YocGF0aHNbZiJ7cm9sZX1fbWV0YSJdLCBpbmRleD1GYWxzZSkKCiAgICByZXR1cm4gcmVzdWx0cwo='
    ),
    'src/domain_adaptation/data/torch_dataset.py': (
        'IiIiVGhpbiB0b3JjaC51dGlscy5kYXRhLkRhdGFzZXQgd3JhcHBlciBhcm91bmQgYSBub3JtYWxpemVkIHNwZWN0cm9ncmFtCmFycmF5IHBsdXMgaXRzIHBlci13aW5kb3cgZG9tYWluIGlkIC8gbGFiZWwuIEtlcHQgc2VwYXJhdGUgZnJvbSBkYXRhc2V0LnB5CnNvIHRoZSBhY3R1YWwgZGF0YSBlbmdpbmVlcmluZyAod2luZG93aW5nLCBTVEZULCByZXNhbXBsaW5nLCByb2xlCnNwbGl0dGluZywgY2FjaGluZykgaGFzIG5vIHRvcmNoIGRlcGVuZGVuY3kgLS0gc2FtZSByZWFzb25pbmcgYXMKdmlicmF0aW9uX2FkLmRhdGEudG9yY2hfZGF0YXNldC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0CgoKY2xhc3MgTGFiZWxlZERvbWFpbkRhdGFzZXQoRGF0YXNldCk6CiAgICAiIiJZaWVsZHMgKHNwZWN0cm9ncmFtLCBkb21haW5faWQsIHVuaXZlcnNhbF9sYWJlbCkgdHJpcGxlcy4gVXNlZCBmb3IKICAgIGV2ZXJ5IHJvbGUgKFNPVVJDRV9UUkFJTi9WQUwsIFRBUkdFVF9URVNUKSBleGNlcHQgVEFSR0VUX0FEQVBULCB3aG9zZQogICAgbGFiZWxzIGFyZSBuZXZlciB1c2VkIGZvciB0cmFpbmluZyAtLSBzZWUgVW5sYWJlbGVkRG9tYWluRGF0YXNldC4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzcGVjdHJvZ3JhbXM6IG5wLm5kYXJyYXksIGRvbWFpbl9pZHM6IG5wLm5kYXJyYXksIGxhYmVsczogbnAubmRhcnJheSk6CiAgICAgICAgIyAoTiwgRiwgVCkgLT4gKE4sIDEsIEYsIFQpOiBhZGQgdGhlIHNpbmdsZS1jaGFubmVsIGRpbSBjb252MmQgZXhwZWN0cy4KICAgICAgICBzZWxmLmRhdGEgPSB0b3JjaC5mcm9tX251bXB5KHNwZWN0cm9ncmFtcykudW5zcXVlZXplKDEpLmZsb2F0KCkKICAgICAgICBzZWxmLmRvbWFpbl9pZHMgPSB0b3JjaC5mcm9tX251bXB5KG5wLmFzYXJyYXkoZG9tYWluX2lkcykpLmxvbmcoKQogICAgICAgIHNlbGYubGFiZWxzID0gdG9yY2guZnJvbV9udW1weShucC5hc2FycmF5KGxhYmVscykpLmxvbmcoKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZGF0YSkKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4OiBpbnQpIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdOgogICAgICAgIHJldHVybiBzZWxmLmRhdGFbaWR4XSwgc2VsZi5kb21haW5faWRzW2lkeF0sIHNlbGYubGFiZWxzW2lkeF0KCgpjbGFzcyBVbmxhYmVsZWREb21haW5EYXRhc2V0KERhdGFzZXQpOgogICAgIiIiWWllbGRzIChzcGVjdHJvZ3JhbSwgZG9tYWluX2lkKSBwYWlycyAtLSBmb3IgVEFSR0VUX0FEQVBULCB3aG9zZQogICAgcm93cyBhcmUgYWxsIGhlYWx0aHkgYnkgY29uc3RydWN0aW9uIChkYXRhL2RhdGFzZXQucHkpIGJ1dCB3aG9zZQogICAgbGFiZWxzIGFyZSBkZWxpYmVyYXRlbHkgbmV2ZXIgcmVhZCBkdXJpbmcgdHJhaW5pbmcsIG9ubHkgdGhlIGRvbWFpbiBpZAogICAgKGZvciB0aGUgZGlzY3JpbWluYXRvcikgYW5kIHRoZSBmZWF0dXJlcyB0aGVtc2VsdmVzIChmb3IgdGhlCiAgICBoZWFsdGh5LWFuY2hvciBsb3NzKS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzcGVjdHJvZ3JhbXM6IG5wLm5kYXJyYXksIGRvbWFpbl9pZHM6IG5wLm5kYXJyYXkpOgogICAgICAgIHNlbGYuZGF0YSA9IHRvcmNoLmZyb21fbnVtcHkoc3BlY3Ryb2dyYW1zKS51bnNxdWVlemUoMSkuZmxvYXQoKQogICAgICAgIHNlbGYuZG9tYWluX2lkcyA9IHRvcmNoLmZyb21fbnVtcHkobnAuYXNhcnJheShkb21haW5faWRzKSkubG9uZygpCgogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5kYXRhKQoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHg6IGludCkgLT4gdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3JdOgogICAgICAgIHJldHVybiBzZWxmLmRhdGFbaWR4XSwgc2VsZi5kb21haW5faWRzW2lkeF0K'
    ),
    'src/domain_adaptation/models/__init__.py': (
        ''
    ),
    'src/domain_adaptation/models/grl.py': (
        'IiIiR3JhZGllbnQgUmV2ZXJzYWwgTGF5ZXIgKEdSTCkgLS0gdGhlIG1lY2hhbmlzbSB0aGF0IG1ha2VzIERBTk4gYWR2ZXJzYXJpYWwuCgpGb3J3YXJkIHBhc3M6IGlkZW50aXR5IChjaGFuZ2VzIG5vdGhpbmcpLiBCYWNrd2FyZCBwYXNzOiBtdWx0aXBsaWVzIHRoZQppbmNvbWluZyBncmFkaWVudCBieSAtbGFtYmRhXy4gU28gZnJvbSB0aGUgZmVhdHVyZSBleHRyYWN0b3IncyBwb2ludCBvZgp2aWV3LCBtaW5pbWl6aW5nIHRoZSBkb21haW4gZGlzY3JpbWluYXRvcidzIGxvc3MgdGhyb3VnaCB0aGlzIGxheWVyIGlzIHRoZQpzYW1lIGFzICptYXhpbWl6aW5nKiBpdCBkaXJlY3RseSAtLSB0aGUgc3RhbmRhcmQgdHJpY2sgZnJvbSBHYW5pbiAmIExlbXBpdHNreQoyMDE1ICgiVW5zdXBlcnZpc2VkIERvbWFpbiBBZGFwdGF0aW9uIGJ5IEJhY2twcm9wYWdhdGlvbiIpIHRoYXQgbGV0cyBvbmUKb3B0aW1pemVyLCBvbmUgYmFja3dhcmQoKSBjYWxsLCB0cmFpbiB0aGUgd2hvbGUgYWR2ZXJzYXJpYWwgZ2FtZSB3aXRob3V0CmFsdGVybmF0aW5nIG1pbi9tYXggc3RlcHMgYnkgaGFuZC4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KZnJvbSB0b3JjaC5hdXRvZ3JhZCBpbXBvcnQgRnVuY3Rpb24KCgpjbGFzcyBfR3JhZGllbnRSZXZlcnNhbEZ1bmN0aW9uKEZ1bmN0aW9uKToKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBmb3J3YXJkKGN0eCwgeDogdG9yY2guVGVuc29yLCBsYW1iZGFfOiBmbG9hdCkgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGN0eC5sYW1iZGFfID0gbGFtYmRhXwogICAgICAgIHJldHVybiB4LnZpZXdfYXMoeCkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgYmFja3dhcmQoY3R4LCBncmFkX291dHB1dDogdG9yY2guVGVuc29yKToKICAgICAgICByZXR1cm4gLWN0eC5sYW1iZGFfICogZ3JhZF9vdXRwdXQsIE5vbmUKCgpjbGFzcyBHcmFkaWVudFJldmVyc2FsTGF5ZXIobm4uTW9kdWxlKToKICAgICIiImBsYW1iZGFfYCBpcyBtdXRhYmxlIChzZXQgaXQgYmVmb3JlIGVhY2ggZm9yd2FyZCBwYXNzIGZyb20gdGhlCiAgICB0cmFpbmluZyBsb29wKSBzbyBpdCBjYW4gZm9sbG93IHRoZSBzdGFuZGFyZCBEQU5OIHdhcm0tdXAgc2NoZWR1bGUgLS0KICAgIHNlZSBgZ3JsX2xhbWJkYV9zY2hlZHVsZSgpYCBiZWxvdyAtLSByYXRoZXIgdGhhbiBiZWluZyBmaXhlZCBmb3IgdGhlCiAgICB3aG9sZSBydW4uCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbGFtYmRhXzogZmxvYXQgPSAxLjApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubGFtYmRhXyA9IGxhbWJkYV8KCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gX0dyYWRpZW50UmV2ZXJzYWxGdW5jdGlvbi5hcHBseSh4LCBzZWxmLmxhbWJkYV8pCgoKZGVmIGdybF9sYW1iZGFfc2NoZWR1bGUocHJvZ3Jlc3M6IGZsb2F0LCBnYW1tYTogZmxvYXQgPSAxMC4wKSAtPiBmbG9hdDoKICAgICIiInByb2dyZXNzOiB0cmFpbmluZyBwcm9ncmVzcyBpbiBbMCwgMV0gKGUuZy4gZXBvY2ggLyB0b3RhbF9lcG9jaHMpLgoKICAgIFJldHVybnMgbGFtYmRhID0gMi8oMStleHAoLWdhbW1hKnByb2dyZXNzKSkgLSAxLCB0aGUgc2NoZWR1bGUgZnJvbSB0aGUKICAgIG9yaWdpbmFsIERBTk4gcGFwZXI6IHN0YXJ0cyBhdCAwIChlYXJseSB0cmFpbmluZyBkb2Vzbid0IGZpZ2h0IHRoZQogICAgc3RpbGwtcmFuZG9tIGZlYXR1cmUgZXh0cmFjdG9yIHdpdGggYSBmdWxsLXN0cmVuZ3RoIGFkdmVyc2FyaWFsCiAgICBzaWduYWwpIGFuZCBzYXR1cmF0ZXMgdG93YXJkIDEgYXMgdHJhaW5pbmcgcHJvZ3Jlc3Nlcy4KICAgICIiIgogICAgcHJvZ3Jlc3MgPSBtaW4obWF4KHByb2dyZXNzLCAwLjApLCAxLjApCiAgICByZXR1cm4gZmxvYXQoMi4wIC8gKDEuMCArIG5wLmV4cCgtZ2FtbWEgKiBwcm9ncmVzcykpIC0gMS4wKQo='
    ),
    'src/domain_adaptation/models/feature_extractor.py': (
        'IiIiU2hhcmVkIENOTiBmZWF0dXJlIGV4dHJhY3RvciAoR19mKSAtLSB0aGUgcGFydCBvZiB0aGUgbmV0d29yayBldmVyeQpkb21haW4ncyBzcGVjdHJvZ3JhbXMgcGFzcyB0aHJvdWdoLCBhbmQgdGhlIHBhcnQgdGhlIGRvbWFpbi1hZHZlcnNhcmlhbApsb3NzIHNoYXBlcyB0byBiZSBkb21haW4taW52YXJpYW50LgoKU2FtZSA0LWxheWVyIHN0cmlkZS0yIGNvbnYgc2hhcGUgYXMgdmlicmF0aW9uX2FkLm1vZGVscy5hdXRvZW5jb2RlcidzCmVuY29kZXIgKHJldXNlZCBpZGVhLCBub3QgcmV1c2VkIGNvZGUgLS0gdGhpcyBvbmUgZmVlZHMgYSBjbGFzc2lmaWVyICsKZG9tYWluIGhlYWQsIG5vdCBhIGRlY29kZXIpLiBEZWxpYmVyYXRlbHkgTk9UIFJlc05ldC0xOCBhcyB0aGUgb3JpZ2luYWwKcGxhbiBwcm9wb3NlZCAtLSBzZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzICJSZXZpc2lvbnMiIHNlY3Rpb246IGEKQ1BVLW9ubHkgS2FnZ2xlIGtlcm5lbCB0cmFpbmluZyBhY3Jvc3MgdHdvIGNvbWJpbmVkIGRhdGFzZXRzIG5lZWRzIGEgbW9kZWwKc21hbGwgZW5vdWdoIHRvIGl0ZXJhdGUgb24gcXVpY2tseS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKCmNsYXNzIENOTkZlYXR1cmVFeHRyYWN0b3Iobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dF9zaGFwZTogdHVwbGVbaW50LCBpbnRdLCBiYXNlX2NoYW5uZWxzOiBpbnQgPSAxNiwgZmVhdHVyZV9kaW06IGludCA9IDI1Nik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgZnJlcV9iaW5zLCB0aW1lX2ZyYW1lcyA9IGlucHV0X3NoYXBlCiAgICAgICAgaWYgZnJlcV9iaW5zICUgMTYgIT0gMCBvciB0aW1lX2ZyYW1lcyAlIDE2ICE9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmImlucHV0X3NoYXBlIHtpbnB1dF9zaGFwZX0gbXVzdCBiZSBkaXZpc2libGUgYnkgMTYgKGZvdXIgc3RyaWRlLTIgbGF5ZXJzKTsgIgogICAgICAgICAgICAgICAgImFkanVzdCBjb25maWdzL2RvbWFpbl9hZGFwdGF0aW9uLnlhbWwncyBjb21tb24vc3RmdCBwYXJhbXMuIgogICAgICAgICAgICApCiAgICAgICAgYyA9IGJhc2VfY2hhbm5lbHMKICAgICAgICBzZWxmLmNvbnYgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoMSwgYywga2VybmVsX3NpemU9Mywgc3RyaWRlPTIsIHBhZGRpbmc9MSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKGMpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYyZChjLCBjICogMiwga2VybmVsX3NpemU9Mywgc3RyaWRlPTIsIHBhZGRpbmc9MSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKGMgKiAyKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQoYyAqIDIsIGMgKiA0LCBrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9MiwgcGFkZGluZz0xKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoYyAqIDQpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYyZChjICogNCwgYyAqIDgsIGtlcm5lbF9zaXplPTMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChjICogOCksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICApCiAgICAgICAgZmxhdF9kaW0gPSBjICogOCAqIChmcmVxX2JpbnMgLy8gMTYpICogKHRpbWVfZnJhbWVzIC8vIDE2KQogICAgICAgIHNlbGYuZmMgPSBubi5MaW5lYXIoZmxhdF9kaW0sIGZlYXR1cmVfZGltKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHogPSBzZWxmLmNvbnYoeCkKICAgICAgICB6ID0gei5mbGF0dGVuKDEpCiAgICAgICAgcmV0dXJuIHNlbGYuZmMoeikKCgpkZWYgc2VsZWN0X2RldmljZShyZXF1ZXN0ZWQ6IHN0ciA9ICJhdXRvIikgLT4gdG9yY2guZGV2aWNlOgogICAgaWYgcmVxdWVzdGVkICE9ICJhdXRvIjoKICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNlKHJlcXVlc3RlZCkKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZSgiY3VkYSIpCiAgICBpZiB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZSgibXBzIikKICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImNwdSIpCg=='
    ),
    'src/domain_adaptation/models/heads.py': (
        'IiIiU21hbGwgTUxQIGhlYWRzIHNpdHRpbmcgb24gdG9wIG9mIHRoZSBzaGFyZWQgZmVhdHVyZSBleHRyYWN0b3I6IHRoZQpmYXVsdCBjbGFzc2lmaWVyIChHX3kpIGFuZCB0aGUgZG9tYWluIGRpc2NyaW1pbmF0b3IgKEdfZCkuIFN0cnVjdHVyYWxseQppZGVudGljYWwgKExpbmVhciAtPiBSZUxVIC0+IERyb3BvdXQgLT4gTGluZWFyKSAtLSB0aGV5IGRpZmZlciBvbmx5IGluCm91dHB1dCB3aWR0aCAobl9jbGFzc2VzIHZzLiBuX2RvbWFpbnMpIGFuZCBpbiB3aGF0IGZlZWRzIHRoZW0gKHRoZQpkaXNjcmltaW5hdG9yIHNpdHMgYmVoaW5kIGEgR3JhZGllbnRSZXZlcnNhbExheWVyLCB3aXJlZCB1cCBpbiBkYW5uLnB5KS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgoKCmNsYXNzIE1MUEhlYWQobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBmZWF0dXJlX2RpbTogaW50LCBoaWRkZW5fZGltOiBpbnQsIG5fb3V0cHV0czogaW50LCBkcm9wb3V0OiBmbG9hdCA9IDAuMyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5uZXQgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5MaW5lYXIoZmVhdHVyZV9kaW0sIGhpZGRlbl9kaW0pLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkRyb3BvdXQoZHJvcG91dCksCiAgICAgICAgICAgIG5uLkxpbmVhcihoaWRkZW5fZGltLCBuX291dHB1dHMpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBmZWF0dXJlczogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHNlbGYubmV0KGZlYXR1cmVzKQo='
    ),
    'src/domain_adaptation/models/dann.py': (
        'IiIiQXNzZW1ibGVzIHRoZSBzaGFyZWQgYnVpbGRpbmcgYmxvY2tzIChmZWF0dXJlX2V4dHJhY3Rvci5weSwgaGVhZHMucHksCmdybC5weSkgaW50byB0aGUgdHdvIG1vZGVscyB0aGlzIGV4cGVyaW1lbnQgdHJhaW5zIGFuZCBjb21wYXJlczoKCmBGZWF0dXJlQ2xhc3NpZmllcmAgLS0gUGhhc2UgMidzIHN1cGVydmlzZWQgYmFzZWxpbmUuIEZlYXR1cmUgZXh0cmFjdG9yICsKICAgIGZhdWx0IGNsYXNzaWZpZXIsIHRyYWluZWQgb24gbGFiZWxlZCBzb3VyY2UgZGF0YSBvbmx5LCBOTyBkb21haW4KICAgIGRpc2NyaW1pbmF0b3IuIFJ1biBvbiB0aGUgdGFyZ2V0IHRlc3Qgc2V0IHdpdGggemVybyBhZGFwdGF0aW9uIHRvCiAgICBtZWFzdXJlIHRoZSBkb21haW4tc2hpZnQgcGVyZm9ybWFuY2UgZHJvcCB0aGUgcGxhbiBhc2tzIGZvci4KCmBEQU5OTW9kZWxgIC0tIFBoYXNlIDMuIEFkZHMgYSBkb21haW4gZGlzY3JpbWluYXRvciBiZWhpbmQgYQogICAgR3JhZGllbnRSZXZlcnNhbExheWVyLCB0cmFpbmVkIGpvaW50bHkgd2l0aCB0aGUgY2xhc3NpZmllciBzbyB0aGUKICAgIGZlYXR1cmUgZXh0cmFjdG9yIGlzIHB1c2hlZCB0b3dhcmQgZG9tYWluLWludmFyaWFudCBmZWF0dXJlcy4KCmBoZWFsdGh5X2FuY2hvcl9sb3NzKClgIC0tIHRoZSBzaW1wbGlmaWVkIHN0YW5kLWluIGZvciB0aGUgcGxhbidzCkNvbmRpdGlvbmFsLU1NRCB0ZXJtIChzZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzICJSZXZpc2lvbnMiKTogcHVsbHMKdGhlIG1lYW4gZmVhdHVyZSB2ZWN0b3Igb2YgbGFiZWxlZC1oZWFsdGh5IHNvdXJjZSBleGFtcGxlcyB0b3dhcmQgdGhlIG1lYW4KZmVhdHVyZSB2ZWN0b3Igb2YgdGhlIChhbGwtaGVhbHRoeSwgYnkgY29uc3RydWN0aW9uKSB0YXJnZXQtYWRhcHRhdGlvbgpiYXRjaCwgZWFjaCBtaW5pLWJhdGNoLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmNvbmZpZyBpbXBvcnQgQ29uZmlnCmZyb20gZG9tYWluX2FkYXB0YXRpb24ubW9kZWxzLmZlYXR1cmVfZXh0cmFjdG9yIGltcG9ydCBDTk5GZWF0dXJlRXh0cmFjdG9yCmZyb20gZG9tYWluX2FkYXB0YXRpb24ubW9kZWxzLmdybCBpbXBvcnQgR3JhZGllbnRSZXZlcnNhbExheWVyCmZyb20gZG9tYWluX2FkYXB0YXRpb24ubW9kZWxzLmhlYWRzIGltcG9ydCBNTFBIZWFkCgpOX0NMQVNTRVMgPSAzICAjIGhlYWx0aHksIGlubmVyX3JhY2UsIG91dGVyX3JhY2UgLS0gc2VlIGRhdGEvY3dydV9sb2FkZXIuVU5JVkVSU0FMX0xBQkVMUwpOX0RPTUFJTlMgPSAzICAjIGN3cnUsIHB1X2FydGlmaWNpYWwsIHB1X3RhcmdldCAtLSBzZWUgZGF0YS9kYXRhc2V0LkRPTUFJTl9JRFMKCgpjbGFzcyBGZWF0dXJlQ2xhc3NpZmllcihubi5Nb2R1bGUpOgogICAgIiIiRmVhdHVyZSBleHRyYWN0b3IgKyBmYXVsdCBjbGFzc2lmaWVyLCBubyBkb21haW4gYWRhcHRhdGlvbi4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRfc2hhcGU6IHR1cGxlW2ludCwgaW50XSwgY29uZmlnOiBDb25maWcpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZmVhdHVyZV9leHRyYWN0b3IgPSBDTk5GZWF0dXJlRXh0cmFjdG9yKGlucHV0X3NoYXBlLCBjb25maWcubW9kZWwuYmFzZV9jaGFubmVscywgY29uZmlnLm1vZGVsLmZlYXR1cmVfZGltKQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IE1MUEhlYWQoY29uZmlnLm1vZGVsLmZlYXR1cmVfZGltLCBjb25maWcubW9kZWwuY2xhc3NpZmllcl9oaWRkZW4sIE5fQ0xBU1NFUywgY29uZmlnLm1vZGVsLmRyb3BvdXQpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgZmVhdHVyZXMgPSBzZWxmLmZlYXR1cmVfZXh0cmFjdG9yKHgpCiAgICAgICAgcmV0dXJuIGZlYXR1cmVzLCBzZWxmLmNsYXNzaWZpZXIoZmVhdHVyZXMpCgoKY2xhc3MgREFOTk1vZGVsKG5uLk1vZHVsZSk6CiAgICAiIiJGZWF0dXJlIGV4dHJhY3RvciArIGZhdWx0IGNsYXNzaWZpZXIgKyBkb21haW4gZGlzY3JpbWluYXRvciAoYmVoaW5kCiAgICBhIEdSTCkuIGBzZXRfZ3JsX2xhbWJkYSgpYCBpcyBjYWxsZWQgb25jZSBwZXIgdHJhaW5pbmcgc3RlcCBmcm9tCiAgICB0cmFpbl9kYW5uLnB5LCBmb2xsb3dpbmcgdGhlIHdhcm0tdXAgc2NoZWR1bGUgaW4gbW9kZWxzL2dybC5weS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dF9zaGFwZTogdHVwbGVbaW50LCBpbnRdLCBjb25maWc6IENvbmZpZyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5mZWF0dXJlX2V4dHJhY3RvciA9IENOTkZlYXR1cmVFeHRyYWN0b3IoaW5wdXRfc2hhcGUsIGNvbmZpZy5tb2RlbC5iYXNlX2NoYW5uZWxzLCBjb25maWcubW9kZWwuZmVhdHVyZV9kaW0pCiAgICAgICAgc2VsZi5jbGFzc2lmaWVyID0gTUxQSGVhZChjb25maWcubW9kZWwuZmVhdHVyZV9kaW0sIGNvbmZpZy5tb2RlbC5jbGFzc2lmaWVyX2hpZGRlbiwgTl9DTEFTU0VTLCBjb25maWcubW9kZWwuZHJvcG91dCkKICAgICAgICBzZWxmLmRvbWFpbl9kaXNjcmltaW5hdG9yID0gTUxQSGVhZChjb25maWcubW9kZWwuZmVhdHVyZV9kaW0sIGNvbmZpZy5tb2RlbC5kb21haW5faGlkZGVuLCBOX0RPTUFJTlMsIGNvbmZpZy5tb2RlbC5kcm9wb3V0KQogICAgICAgIHNlbGYuZ3JsID0gR3JhZGllbnRSZXZlcnNhbExheWVyKGxhbWJkYV89MC4wKQoKICAgIGRlZiBzZXRfZ3JsX2xhbWJkYShzZWxmLCBsYW1iZGFfOiBmbG9hdCkgLT4gTm9uZToKICAgICAgICBzZWxmLmdybC5sYW1iZGFfID0gbGFtYmRhXwoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3IsIHRvcmNoLlRlbnNvcl06CiAgICAgICAgZmVhdHVyZXMgPSBzZWxmLmZlYXR1cmVfZXh0cmFjdG9yKHgpCiAgICAgICAgY2xhc3NfbG9naXRzID0gc2VsZi5jbGFzc2lmaWVyKGZlYXR1cmVzKQogICAgICAgIGRvbWFpbl9sb2dpdHMgPSBzZWxmLmRvbWFpbl9kaXNjcmltaW5hdG9yKHNlbGYuZ3JsKGZlYXR1cmVzKSkKICAgICAgICByZXR1cm4gZmVhdHVyZXMsIGNsYXNzX2xvZ2l0cywgZG9tYWluX2xvZ2l0cwoKCmRlZiBoZWFsdGh5X2FuY2hvcl9sb3NzKAogICAgc291cmNlX2ZlYXR1cmVzOiB0b3JjaC5UZW5zb3IsIHNvdXJjZV9sYWJlbHM6IHRvcmNoLlRlbnNvciwgdGFyZ2V0X2ZlYXR1cmVzOiB0b3JjaC5UZW5zb3IKKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiJTcXVhcmVkIGRpc3RhbmNlIGJldHdlZW4gdGhlIG1lYW4gZmVhdHVyZSB2ZWN0b3Igb2YgbGFiZWxlZC1oZWFsdGh5CiAgICBzb3VyY2UgZXhhbXBsZXMgaW4gdGhpcyBiYXRjaCBhbmQgdGhlIG1lYW4gZmVhdHVyZSB2ZWN0b3Igb2YgdGhlCiAgICB0YXJnZXQtYWRhcHRhdGlvbiBiYXRjaCAoYWxsIGhlYWx0aHksIGJ5IGNvbnN0cnVjdGlvbiBvZiBUQVJHRVRfQURBUFQKICAgIC0tIHNlZSBkYXRhL2RhdGFzZXQucHkpLiBSZXR1cm5zIDAgKG5vdCBOYU4pIGlmIHRoZSBiYXRjaCBoYXBwZW5zIHRvCiAgICBjb250YWluIG5vIGhlYWx0aHkgc291cmNlIGV4YW1wbGVzLCBzbyBhbiB1bmx1Y2t5IGJhdGNoIGNvbXBvc2l0aW9uCiAgICBkb2Vzbid0IGNyYXNoIHRyYWluaW5nLgogICAgIiIiCiAgICBoZWFsdGh5X21hc2sgPSBzb3VyY2VfbGFiZWxzID09IDAKICAgIGlmIGhlYWx0aHlfbWFzay5zdW0oKSA9PSAwOgogICAgICAgIHJldHVybiB0b3JjaC56ZXJvcygoKSwgZGV2aWNlPXNvdXJjZV9mZWF0dXJlcy5kZXZpY2UsIGR0eXBlPXNvdXJjZV9mZWF0dXJlcy5kdHlwZSkKICAgIHNvdXJjZV9oZWFsdGh5X21lYW4gPSBzb3VyY2VfZmVhdHVyZXNbaGVhbHRoeV9tYXNrXS5tZWFuKGRpbT0wKQogICAgdGFyZ2V0X21lYW4gPSB0YXJnZXRfZmVhdHVyZXMubWVhbihkaW09MCkKICAgIHJldHVybiAoKHNvdXJjZV9oZWFsdGh5X21lYW4gLSB0YXJnZXRfbWVhbikgKiogMikuc3VtKCkK'
    ),
    'src/domain_adaptation/train_baseline.py': (
        'IiIiUGhhc2UgMjogc3VwZXJ2aXNlZCBzb3VyY2Utb25seSBiYXNlbGluZSAtLSBOTyBkb21haW4gYWRhcHRhdGlvbi4KClRyYWlucyBGZWF0dXJlQ2xhc3NpZmllciBvbiBTT1VSQ0VfVFJBSU4gKENXUlUgKyBQVV9BcnRpZmljaWFsLCBmdWxseQpsYWJlbGVkKSwgZWFybHktc3RvcHMgb24gU09VUkNFX1ZBTC4gTmV2ZXIgc2VlcyBUQVJHRVRfQURBUFQgb3IKVEFSR0VUX1RFU1QgZHVyaW5nIHRyYWluaW5nLiBldmFsdWF0ZS5weSB0aGVuIHJ1bnMgdGhpcyBleGFjdCBtb2RlbCBvbgpUQVJHRVRfVEVTVCAodGhlIGhlbGQtb3V0IGNvbmRpdGlvbidzIHJlYWwgZmF1bHRzKSB0byBtZWFzdXJlIHRoZQpkb21haW4tc2hpZnQgcGVyZm9ybWFuY2UgZHJvcCBCRUZPUkUgYWRhcHRhdGlvbiAtLSB0aGUgbnVtYmVyIFBoYXNlIDMKKHRyYWluX2Rhbm4ucHkpIGlzIHRyeWluZyB0byBpbXByb3ZlIG9uLgoKVGhpcyBzY3JpcHQgaXMgYWxzbyB0aGUgY2Fub25pY2FsIHBsYWNlIHRoZSBwZXItZnJlcXVlbmN5LWJpbiBub3JtYWxpemVyCmlzIGZpdCAob24gU09VUkNFX1RSQUlOIG9ubHkpIGFuZCBzYXZlZCAtLSB0cmFpbl9kYW5uLnB5IGxvYWRzIHRoZSBTQU1FCnNhdmVkIG5vcm1hbGl6ZXIgcmF0aGVyIHRoYW4gcmUtZml0dGluZywgc28gYmFzZWxpbmUgYW5kIERBTk4gYXJlCmV2YWx1YXRlZCBvbiBpZGVudGljYWxseS1ub3JtYWxpemVkIGlucHV0cyBhbmQgdGhlaXIgbnVtYmVycyBhcmUgZGlyZWN0bHkKY29tcGFyYWJsZS4KClVzYWdlOgogICAgcHl0aG9uIC1tIGRvbWFpbl9hZGFwdGF0aW9uLnRyYWluX2Jhc2VsaW5lIC0tY29uZmlnIGNvbmZpZ3MvZG9tYWluX2FkYXB0YXRpb24ueWFtbCBbLS1wdS1yb290IERJUl0KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgZ2MKaW1wb3J0IHJhbmRvbQppbXBvcnQgc2h1dGlsCmltcG9ydCB0aW1lCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCgpmcm9tIHZpYnJhdGlvbl9hZC5mZWF0dXJlcy5zcGVjdHJvZ3JhbSBpbXBvcnQgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyLCBleHBlY3RlZF9zaGFwZQoKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5jb25maWcgaW1wb3J0IENvbmZpZywgbG9hZF9jb25maWcKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLmRhdGFzZXQgaW1wb3J0IGJ1aWxkX2RvbWFpbl9hZGFwdGF0aW9uX2RhdGFzZXRzCmZyb20gZG9tYWluX2FkYXB0YXRpb24uZGF0YS5wdV9sb2FkZXIgaW1wb3J0IGZpbmRfcHVfcm9vdApmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEudG9yY2hfZGF0YXNldCBpbXBvcnQgTGFiZWxlZERvbWFpbkRhdGFzZXQKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5tb2RlbHMuZGFubiBpbXBvcnQgRmVhdHVyZUNsYXNzaWZpZXIKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5tb2RlbHMuZmVhdHVyZV9leHRyYWN0b3IgaW1wb3J0IHNlbGVjdF9kZXZpY2UKCgpkZWYgc2V0X3NlZWQoc2VlZDogaW50KSAtPiBOb25lOgogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG5wLnJhbmRvbS5zZWVkKHNlZWQpCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQoKCmRlZiBydW5fZXBvY2gobW9kZWwsIGxvYWRlciwgb3B0aW1pemVyLCBkZXZpY2UsIHRyYWluOiBib29sKSAtPiB0dXBsZVtmbG9hdCwgZmxvYXRdOgogICAgbW9kZWwudHJhaW4odHJhaW4pCiAgICB0b3RhbF9sb3NzLCB0b3RhbF9jb3JyZWN0LCBuID0gMC4wLCAwLCAwCiAgICBjcml0ZXJpb24gPSB0b3JjaC5ubi5Dcm9zc0VudHJvcHlMb3NzKCkKICAgIGNvbnRleHQgPSB0b3JjaC5lbmFibGVfZ3JhZCgpIGlmIHRyYWluIGVsc2UgdG9yY2gubm9fZ3JhZCgpCiAgICB3aXRoIGNvbnRleHQ6CiAgICAgICAgZm9yIFgsIF9kb21haW5faWQsIGxhYmVsIGluIGxvYWRlcjoKICAgICAgICAgICAgWCwgbGFiZWwgPSBYLnRvKGRldmljZSksIGxhYmVsLnRvKGRldmljZSkKICAgICAgICAgICAgX2ZlYXR1cmVzLCBsb2dpdHMgPSBtb2RlbChYKQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKGxvZ2l0cywgbGFiZWwpCgogICAgICAgICAgICBpZiB0cmFpbjoKICAgICAgICAgICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCgogICAgICAgICAgICB0b3RhbF9sb3NzICs9IGxvc3MuaXRlbSgpICogbGVuKFgpCiAgICAgICAgICAgIHRvdGFsX2NvcnJlY3QgKz0gKGxvZ2l0cy5hcmdtYXgoZGltPTEpID09IGxhYmVsKS5zdW0oKS5pdGVtKCkKICAgICAgICAgICAgbiArPSBsZW4oWCkKICAgIHJldHVybiB0b3RhbF9sb3NzIC8gbiwgdG90YWxfY29ycmVjdCAvIG4KCgpkZWYgdHJhaW4oY29uZmlnOiBDb25maWcsIGNvbmZpZ19wYXRoOiBzdHIsIHB1X3Jvb3QsIGZvcmNlX2RhdGE6IGJvb2wgPSBGYWxzZSkgLT4gTm9uZToKICAgIHNldF9zZWVkKGNvbmZpZy5zZWVkKQoKICAgIG1vZGVsc19kaXIgPSBjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMubW9kZWxzX2RpcikKICAgIHJlcG9ydHNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnJlcG9ydHNfZGlyKQogICAgbW9kZWxzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByZXBvcnRzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgcHJpbnQoIkJ1aWxkaW5nIGRhdGFzZXRzIChjYWNoZWQgYWZ0ZXIgZmlyc3QgcnVuKS4uLiIpCiAgICBkYXRhc2V0cyA9IGJ1aWxkX2RvbWFpbl9hZGFwdGF0aW9uX2RhdGFzZXRzKGNvbmZpZywgcHVfcm9vdCwgZm9yY2U9Zm9yY2VfZGF0YSkKICAgIHRyYWluX1gsIHRyYWluX21ldGEgPSBkYXRhc2V0c1sic291cmNlX3RyYWluIl0KICAgIHZhbF9YLCB2YWxfbWV0YSA9IGRhdGFzZXRzWyJzb3VyY2VfdmFsIl0KICAgIHByaW50KGYiICBzb3VyY2VfdHJhaW4gd2luZG93czoge3RyYWluX1guc2hhcGV9ICBzb3VyY2VfdmFsIHdpbmRvd3M6IHt2YWxfWC5zaGFwZX0iKQoKICAgIGV4cGVjdGVkID0gZXhwZWN0ZWRfc2hhcGUoY29uZmlnLmNvbW1vbiwgY29uZmlnLnN0ZnQpCiAgICBpZiB0cmFpbl9YLnNoYXBlWzE6XSAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJTcGVjdHJvZ3JhbSBzaGFwZSB7dHJhaW5fWC5zaGFwZVsxOl19ICE9IGV4cGVjdGVkIHtleHBlY3RlZH0iKQoKICAgIG5vcm1hbGl6ZXIgPSBTcGVjdHJvZ3JhbU5vcm1hbGl6ZXIoKS5maXQodHJhaW5fWCkKICAgIHRyYWluX1hfbm9ybSA9IG5vcm1hbGl6ZXIudHJhbnNmb3JtKHRyYWluX1gpCiAgICB2YWxfWF9ub3JtID0gbm9ybWFsaXplci50cmFuc2Zvcm0odmFsX1gpCiAgICAjIFRoZSB1bi1ub3JtYWxpemVkIGNvcGllcyBhcmUgdGhlIHNhbWUgc2l6ZSBhcyB0aGUgbm9ybWFsaXplZCBvbmVzCiAgICAjICh+NC4yIEdCIGVhY2ggYXQgZnVsbCBzb3VyY2UgYnJlYWR0aCkgYW5kIGFyZSBkZWFkIGZyb20gaGVyZSBvbi4KICAgICMgVGhleSBzdGF5IGNhY2hlZCBvbiBkaXNrLCBzbyBub3RoaW5nIGlzIGxvc3QgYnkgZHJvcHBpbmcgdGhlbS4KICAgIGRlbCB0cmFpbl9YLCB2YWxfWCwgZGF0YXNldHMKICAgIGdjLmNvbGxlY3QoKQoKICAgIGRldmljZSA9IHNlbGVjdF9kZXZpY2UoY29uZmlnLnRyYWluX2Jhc2VsaW5lLmRldmljZSkKICAgIHByaW50KGYiVXNpbmcgZGV2aWNlOiB7ZGV2aWNlfSIpCgogICAgdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcigKICAgICAgICBMYWJlbGVkRG9tYWluRGF0YXNldCh0cmFpbl9YX25vcm0sIHRyYWluX21ldGEuZG9tYWluX2lkLnRvX251bXB5KCksIHRyYWluX21ldGEudW5pdmVyc2FsX2xhYmVsLnRvX251bXB5KCkpLAogICAgICAgIGJhdGNoX3NpemU9Y29uZmlnLnRyYWluX2Jhc2VsaW5lLmJhdGNoX3NpemUsCiAgICAgICAgc2h1ZmZsZT1UcnVlLAogICAgKQogICAgdmFsX2xvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgTGFiZWxlZERvbWFpbkRhdGFzZXQodmFsX1hfbm9ybSwgdmFsX21ldGEuZG9tYWluX2lkLnRvX251bXB5KCksIHZhbF9tZXRhLnVuaXZlcnNhbF9sYWJlbC50b19udW1weSgpKSwKICAgICAgICBiYXRjaF9zaXplPWNvbmZpZy50cmFpbl9iYXNlbGluZS5iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9RmFsc2UsCiAgICApCgogICAgbW9kZWwgPSBGZWF0dXJlQ2xhc3NpZmllcihleHBlY3RlZCwgY29uZmlnKS50byhkZXZpY2UpCiAgICBuX3BhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQogICAgcHJpbnQoZiJNb2RlbCBwYXJhbWV0ZXJzOiB7bl9wYXJhbXM6LH0iKQoKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0oCiAgICAgICAgbW9kZWwucGFyYW1ldGVycygpLCBscj1jb25maWcudHJhaW5fYmFzZWxpbmUubGVhcm5pbmdfcmF0ZSwgd2VpZ2h0X2RlY2F5PWNvbmZpZy50cmFpbl9iYXNlbGluZS53ZWlnaHRfZGVjYXkKICAgICkKCiAgICBiZXN0X3ZhbF9sb3NzID0gZmxvYXQoImluZiIpCiAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA9IDAKICAgIGhpc3RvcnkgPSBbXQogICAgYmVzdF9ja3B0X3BhdGggPSBtb2RlbHNfZGlyIC8gImJhc2VsaW5lX2Jlc3QucHQiCgogICAgZm9yIGVwb2NoIGluIHJhbmdlKDEsIGNvbmZpZy50cmFpbl9iYXNlbGluZS5lcG9jaHMgKyAxKToKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgdHJhaW5fbG9zcywgdHJhaW5fYWNjID0gcnVuX2Vwb2NoKG1vZGVsLCB0cmFpbl9sb2FkZXIsIG9wdGltaXplciwgZGV2aWNlLCB0cmFpbj1UcnVlKQogICAgICAgIHZhbF9sb3NzLCB2YWxfYWNjID0gcnVuX2Vwb2NoKG1vZGVsLCB2YWxfbG9hZGVyLCBvcHRpbWl6ZXIsIGRldmljZSwgdHJhaW49RmFsc2UpCiAgICAgICAgZHQgPSB0aW1lLnRpbWUoKSAtIHQwCgogICAgICAgIGltcHJvdmVkID0gdmFsX2xvc3MgPCBiZXN0X3ZhbF9sb3NzCiAgICAgICAgaWYgaW1wcm92ZWQ6CiAgICAgICAgICAgIGJlc3RfdmFsX2xvc3MgPSB2YWxfbG9zcwogICAgICAgICAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA9IDAKICAgICAgICAgICAgdG9yY2guc2F2ZShtb2RlbC5zdGF0ZV9kaWN0KCksIGJlc3RfY2twdF9wYXRoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGVwb2Noc193aXRob3V0X2ltcHJvdmVtZW50ICs9IDEKCiAgICAgICAgbWFya2VyID0gIiAqIiBpZiBpbXByb3ZlZCBlbHNlICIiCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiZXBvY2gge2Vwb2NoOjNkfS97Y29uZmlnLnRyYWluX2Jhc2VsaW5lLmVwb2Noc30gICIKICAgICAgICAgICAgZiJ0cmFpbl9sb3NzPXt0cmFpbl9sb3NzOi40Zn0gdHJhaW5fYWNjPXt0cmFpbl9hY2M6LjNmfSAgIgogICAgICAgICAgICBmInZhbF9sb3NzPXt2YWxfbG9zczouNGZ9IHZhbF9hY2M9e3ZhbF9hY2M6LjNmfSAgKHtkdDouMWZ9cyl7bWFya2VyfSIKICAgICAgICApCiAgICAgICAgaGlzdG9yeS5hcHBlbmQoCiAgICAgICAgICAgIHsiZXBvY2giOiBlcG9jaCwgInRyYWluX2xvc3MiOiB0cmFpbl9sb3NzLCAidHJhaW5fYWNjIjogdHJhaW5fYWNjLCAidmFsX2xvc3MiOiB2YWxfbG9zcywgInZhbF9hY2MiOiB2YWxfYWNjfQogICAgICAgICkKCiAgICAgICAgaWYgZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgPj0gY29uZmlnLnRyYWluX2Jhc2VsaW5lLmVhcmx5X3N0b3BwaW5nX3BhdGllbmNlOgogICAgICAgICAgICBwcmludChmIkVhcmx5IHN0b3BwaW5nOiBubyB2YWwgaW1wcm92ZW1lbnQgaW4ge2NvbmZpZy50cmFpbl9iYXNlbGluZS5lYXJseV9zdG9wcGluZ19wYXRpZW5jZX0gZXBvY2hzLiIpCiAgICAgICAgICAgIGJyZWFrCgogICAgcGQuRGF0YUZyYW1lKGhpc3RvcnkpLnRvX2NzdihyZXBvcnRzX2RpciAvICJiYXNlbGluZV90cmFpbmluZ19sb2cuY3N2IiwgaW5kZXg9RmFsc2UpCiAgICBub3JtYWxpemVyLnNhdmUobW9kZWxzX2RpciAvICJub3JtYWxpemVyLm5weiIpCiAgICBzaHV0aWwuY29weShjb25maWdfcGF0aCwgbW9kZWxzX2RpciAvICJydW5fY29uZmlnLnlhbWwiKQoKICAgIHByaW50KGYiXG5CZXN0IHZhbCBsb3NzOiB7YmVzdF92YWxfbG9zczouNGZ9IikKICAgIHByaW50KGYiQ2hlY2twb2ludCBzYXZlZCB0bzoge2Jlc3RfY2twdF9wYXRofSIpCiAgICBwcmludCgiTmV4dDogcHl0aG9uIC1tIGRvbWFpbl9hZGFwdGF0aW9uLnRyYWluX2Rhbm4gLS1jb25maWcgPHNhbWUgY29uZmlnPiIpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD0iY29uZmlncy9kb21haW5fYWRhcHRhdGlvbi55YW1sIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHUtcm9vdCIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iUm9vdCBkaXIgb2YgdGhlIFBVIGRhdGFzZXQgKGRlZmF1bHQ6IGF1dG8tZGV0ZWN0IC9rYWdnbGUvaW5wdXQsIGVsc2UgY29uZmlnJ3MgcHVfcm9vdF9sb2NhbCkuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZm9yY2UtZGF0YSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9IlJlY29tcHV0ZSBjYWNoZWQgc3BlY3Ryb2dyYW1zIGV2ZW4gaWYgcHJlc2VudC4iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBjb25maWcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIHB1X3Jvb3QgPSBhcmdzLnB1X3Jvb3Qgb3IgZmluZF9wdV9yb290KGNvbmZpZy5yZXNvbHZlKGNvbmZpZy5wYXRocy5wdV9yb290X2xvY2FsKSkKICAgIHRyYWluKGNvbmZpZywgY29uZmlnX3BhdGg9YXJncy5jb25maWcsIHB1X3Jvb3Q9cHVfcm9vdCwgZm9yY2VfZGF0YT1hcmdzLmZvcmNlX2RhdGEpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
    ),
    'src/domain_adaptation/train_dann.py': (
        'IiIiUGhhc2UgMzogREFOTiBhZGFwdGF0aW9uIC0tIGFkZHMgYSBkb21haW4gZGlzY3JpbWluYXRvciAoYmVoaW5kIGEKR3JhZGllbnQgUmV2ZXJzYWwgTGF5ZXIpIGFuZCBhIGhlYWx0aHktYW5jaG9yIGFsaWdubWVudCBsb3NzIG9uIHRvcCBvZgpQaGFzZSAyJ3MgY2xhc3NpZmllciwgc28gdGhlIGZlYXR1cmUgZXh0cmFjdG9yIGlzIHB1c2hlZCB0b3dhcmQgZmVhdHVyZXMKdGhhdCAoYSkgc3RpbGwgc3VwcG9ydCBmYXVsdCBjbGFzc2lmaWNhdGlvbiBvbiBsYWJlbGVkIHNvdXJjZSBkYXRhIGFuZAooYikgbG9vayBkb21haW4taW52YXJpYW50IGFjcm9zcyBDV1JVIC8gUFVfQXJ0aWZpY2lhbCAvIFBVX1RhcmdldC4KClBlciB0cmFpbmluZyBzdGVwOiBvbmUgYmF0Y2ggb2YgbGFiZWxlZCBTT1VSQ0VfVFJBSU4gKENXUlUgKyBQVV9BcnRpZmljaWFsCnBvb2xlZCkgKyBvbmUgYmF0Y2ggb2YgdW5sYWJlbGVkIFRBUkdFVF9BREFQVCAodGhlIGhlbGQtb3V0IGNvbmRpdGlvbidzCmhlYWx0aHkgcnVucyBPTkxZIC0tIG5vIGZhdWx0IGV4YW1wbGUsIGxhYmVsZWQgb3Igbm90LCBmcm9tIHRoZSB0YXJnZXQKZXZlciBhcHBlYXJzIGhlcmUsIG9yIGFueXdoZXJlIGluIHRoaXMgc2NyaXB0KS4gIk9uZSBlcG9jaCIgPSBvbmUgZnVsbApwYXNzIG92ZXIgU09VUkNFX1RSQUlOLCBjeWNsaW5nIFRBUkdFVF9BREFQVCAodXN1YWxseSB0aGUgc21hbGxlciBvZiB0aGUKdHdvKSB0byBmaWxsIGluIGJhdGNoZXMgLS0gc2VlIGBfY3ljbGUoKWAuCgpMb3NzID0gY2xhc3NpZmljYXRpb24oc291cmNlKSArIGRvbWFpbl9sb3NzX3dlaWdodCAqIGRvbWFpbihzb3VyY2UrdGFyZ2V0LApHUkwpICsgYW5jaG9yX2xvc3Nfd2VpZ2h0ICogaGVhbHRoeV9hbmNob3Ioc291cmNlX2hlYWx0aHksIHRhcmdldCkuCgpMb2FkcyB0aGUgbm9ybWFsaXplciB0cmFpbl9iYXNlbGluZS5weSBhbHJlYWR5IGZpdCBhbmQgc2F2ZWQgKE5PVApyZS1maXR0aW5nIG9uZSBoZXJlKSBzbyBiYXNlbGluZSBhbmQgREFOTiBzZWUgaWRlbnRpY2FsbHktbm9ybWFsaXplZAppbnB1dHMgLS0gcnVuIHRyYWluX2Jhc2VsaW5lLnB5IGZpcnN0LgoKVXNhZ2U6CiAgICBweXRob24gLW0gZG9tYWluX2FkYXB0YXRpb24udHJhaW5fZGFubiAtLWNvbmZpZyBjb25maWdzL2RvbWFpbl9hZGFwdGF0aW9uLnlhbWwgWy0tcHUtcm9vdCBESVJdCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGdjCmltcG9ydCByYW5kb20KaW1wb3J0IHRpbWUKZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhdG9yCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCgpmcm9tIHZpYnJhdGlvbl9hZC5mZWF0dXJlcy5zcGVjdHJvZ3JhbSBpbXBvcnQgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyLCBleHBlY3RlZF9zaGFwZQoKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5jb25maWcgaW1wb3J0IENvbmZpZywgbG9hZF9jb25maWcKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLmRhdGFzZXQgaW1wb3J0IGJ1aWxkX2RvbWFpbl9hZGFwdGF0aW9uX2RhdGFzZXRzCmZyb20gZG9tYWluX2FkYXB0YXRpb24uZGF0YS5wdV9sb2FkZXIgaW1wb3J0IGZpbmRfcHVfcm9vdApmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEudG9yY2hfZGF0YXNldCBpbXBvcnQgTGFiZWxlZERvbWFpbkRhdGFzZXQsIFVubGFiZWxlZERvbWFpbkRhdGFzZXQKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5tb2RlbHMuZGFubiBpbXBvcnQgTl9ET01BSU5TLCBEQU5OTW9kZWwsIGhlYWx0aHlfYW5jaG9yX2xvc3MKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5tb2RlbHMuZmVhdHVyZV9leHRyYWN0b3IgaW1wb3J0IHNlbGVjdF9kZXZpY2UKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5tb2RlbHMuZ3JsIGltcG9ydCBncmxfbGFtYmRhX3NjaGVkdWxlCgoKZGVmIHNldF9zZWVkKHNlZWQ6IGludCkgLT4gTm9uZToKICAgIHJhbmRvbS5zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdG9yY2gubWFudWFsX3NlZWQoc2VlZCkKCgpkZWYgX2N5Y2xlKGxvYWRlcjogRGF0YUxvYWRlcikgLT4gSXRlcmF0b3I6CiAgICB3aGlsZSBUcnVlOgogICAgICAgIHlpZWxkIGZyb20gbG9hZGVyCgoKZGVmIHJ1bl90cmFpbl9lcG9jaCgKICAgIG1vZGVsLCBzb3VyY2VfbG9hZGVyLCB0YXJnZXRfbG9hZGVyLCBvcHRpbWl6ZXIsIGRldmljZSwKICAgIGRvbWFpbl93ZWlnaHQsIGFuY2hvcl93ZWlnaHQsIGVwb2NoLCB0b3RhbF9lcG9jaHMsIGdybF9nYW1tYSwgZ3JhZF9jbGlwX25vcm0sCikgLT4gZGljdDoKICAgICIiIk9uZSBwYXNzIG92ZXIgU09VUkNFX1RSQUlOLCBjeWNsaW5nIFRBUkdFVF9BREFQVCBhbG9uZ3NpZGUgaXQuCgogICAgVGhlIEdSTCBsYW1iZGEgaXMgdXBkYXRlZCBQRVIgSVRFUkFUSU9OLCBub3Qgb25jZSBwZXIgZXBvY2ggKHRoaXMgaXMKICAgIHRoZSBvcmlnaW5hbCBEQU5OIHBhcGVyJ3MgZm9ybXVsYXRpb24sIGFuZCB2MSBnb3QgaXQgd3JvbmcgLS0gc2VlCiAgICBET01BSU5fQURBUFRBVElPTl9QTEFOLm1kJ3Mga2VybmVsIHYxL3YyIHN0YXR1cykuIEl0IG1hdHRlcnMgbW9yZSB0aGFuCiAgICBpdCBsb29rczogd2l0aCBhIGZhc3QgcmFtcCAoaGlnaCBncmxfZ2FtbWEpLCBhIHBlci1lcG9jaCBsYW1iZGEgd291bGQKICAgIHNsYW0gdGhlIGRpc2NyaW1pbmF0b3Igd2l0aCBzdHJvbmcgcmV2ZXJzZWQgZ3JhZGllbnRzIGZyb20gdGhlIHZlcnkKICAgIGZpcnN0IGJhdGNoLCBiZWZvcmUgaXQgaGFzIGxlYXJuZWQgYW55dGhpbmcgd29ydGggcmV2ZXJzaW5nIC0tCiAgICBzY3JhbWJsaW5nIGl0IGludG8gcHJvZHVjaW5nIG1lYW5pbmdsZXNzIGdyYWRpZW50cy4gUGVyLWl0ZXJhdGlvbiwKICAgIGxhbWJkYSBzdGlsbCBzdGFydHMgYXQgfjAgZHVyaW5nIHRoZSBmaXJzdCBiYXRjaGVzIG9mIGVwb2NoIDEsIHNvIHRoZQogICAgZGlzY3JpbWluYXRvciBnZXRzIGEgY2xlYW4gaGVhZCBzdGFydCwgdGhlbiB0aGUgcmV2ZXJzYWwgcmFtcHMgdXAKICAgIHNtb290aGx5IHdpdGhpbiB0aGUgZXBvY2guCiAgICAiIiIKICAgIG1vZGVsLnRyYWluKFRydWUpCiAgICBjbHNfY3JpdGVyaW9uID0gdG9yY2gubm4uQ3Jvc3NFbnRyb3B5TG9zcygpCiAgICBkb21haW5fY3JpdGVyaW9uID0gdG9yY2gubm4uQ3Jvc3NFbnRyb3B5TG9zcygpCiAgICB0YXJnZXRfaXRlciA9IF9jeWNsZSh0YXJnZXRfbG9hZGVyKQogICAgbl9iYXRjaGVzID0gbGVuKHNvdXJjZV9sb2FkZXIpCgogICAgdG90YWxzID0gewogICAgICAgICJjbHNfbG9zcyI6IDAuMCwgImRvbWFpbl9sb3NzIjogMC4wLCAiYW5jaG9yX2xvc3MiOiAwLjAsCiAgICAgICAgImNvcnJlY3QiOiAwLCAibiI6IDAsICJsYW1iZGFfc3VtIjogMC4wLAogICAgfQogICAgIyBQZXItZG9tYWluIHRhbGxpZXMsIGZvciBhIEJBTEFOQ0VEIGRvbWFpbiBhY2N1cmFjeS4gUmF3IGFjY3VyYWN5IGlzCiAgICAjIG1pc2xlYWRpbmcgaGVyZTogZXZlcnkgYmF0Y2ggaXMgNjQgc291cmNlICsgNjQgdGFyZ2V0LCBzbyBwdV90YXJnZXQgaXMKICAgICMgNTAlIG9mIGl0IGJ5IGNvbnN0cnVjdGlvbiBhbmQgYSBkaXNjcmltaW5hdG9yIHRoYXQgYmxpbmRseSBhbnN3ZXJzCiAgICAjICJ0YXJnZXQiIHNjb3JlcyAwLjUwIC0tIHdoaWNoIGlzIGV4YWN0bHkgd2hhdCB2MidzIDAuNDg3LTAuNTI0IHdhcy4KICAgICMgTWFjcm8tYXZlcmFnaW5nIHJlY2FsbCBvdmVyIHRoZSAzIGRvbWFpbnMgbWFrZXMgY2hhbmNlIGEgdHJ1ZSAxLzMuCiAgICBkb21haW5fY29ycmVjdCA9IHRvcmNoLnplcm9zKE5fRE9NQUlOUykKICAgIGRvbWFpbl90b3RhbCA9IHRvcmNoLnplcm9zKE5fRE9NQUlOUykKICAgIGZvciBiYXRjaF9pZHgsIChYX3MsIGRvbWFpbl9zLCBsYWJlbF9zKSBpbiBlbnVtZXJhdGUoc291cmNlX2xvYWRlcik6CiAgICAgICAgIyBHbG9iYWwgdHJhaW5pbmcgcHJvZ3Jlc3MgaW4gWzAsIDFdLCBjb3VudGVkIGluIElURVJBVElPTlMuCiAgICAgICAgcHJvZ3Jlc3MgPSAoKGVwb2NoIC0gMSkgKiBuX2JhdGNoZXMgKyBiYXRjaF9pZHgpIC8gbWF4KHRvdGFsX2Vwb2NocyAqIG5fYmF0Y2hlcyAtIDEsIDEpCiAgICAgICAgbGFtYmRhXyA9IGdybF9sYW1iZGFfc2NoZWR1bGUocHJvZ3Jlc3MsIGdybF9nYW1tYSkKICAgICAgICBtb2RlbC5zZXRfZ3JsX2xhbWJkYShsYW1iZGFfKQoKICAgICAgICBYX3QsIGRvbWFpbl90ID0gbmV4dCh0YXJnZXRfaXRlcikKICAgICAgICBYX3MsIGRvbWFpbl9zLCBsYWJlbF9zID0gWF9zLnRvKGRldmljZSksIGRvbWFpbl9zLnRvKGRldmljZSksIGxhYmVsX3MudG8oZGV2aWNlKQogICAgICAgIFhfdCwgZG9tYWluX3QgPSBYX3QudG8oZGV2aWNlKSwgZG9tYWluX3QudG8oZGV2aWNlKQoKICAgICAgICBmZWF0dXJlc19zLCBjbGFzc19sb2dpdHMsIGRvbWFpbl9sb2dpdHNfcyA9IG1vZGVsKFhfcykKICAgICAgICBmZWF0dXJlc190LCBfY2xhc3NfbG9naXRzX3QsIGRvbWFpbl9sb2dpdHNfdCA9IG1vZGVsKFhfdCkKCiAgICAgICAgY2xzX2xvc3MgPSBjbHNfY3JpdGVyaW9uKGNsYXNzX2xvZ2l0cywgbGFiZWxfcykKICAgICAgICBkb21haW5fbG9naXRzID0gdG9yY2guY2F0KFtkb21haW5fbG9naXRzX3MsIGRvbWFpbl9sb2dpdHNfdF0sIGRpbT0wKQogICAgICAgIGRvbWFpbl90YXJnZXRzID0gdG9yY2guY2F0KFtkb21haW5fcywgZG9tYWluX3RdLCBkaW09MCkKICAgICAgICBkb21haW5fbG9zcyA9IGRvbWFpbl9jcml0ZXJpb24oZG9tYWluX2xvZ2l0cywgZG9tYWluX3RhcmdldHMpCiAgICAgICAgYW5jaG9yX2xvc3MgPSBoZWFsdGh5X2FuY2hvcl9sb3NzKGZlYXR1cmVzX3MsIGxhYmVsX3MsIGZlYXR1cmVzX3QpCgogICAgICAgIGxvc3MgPSBjbHNfbG9zcyArIGRvbWFpbl93ZWlnaHQgKiBkb21haW5fbG9zcyArIGFuY2hvcl93ZWlnaHQgKiBhbmNob3JfbG9zcwoKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBpZiBncmFkX2NsaXBfbm9ybSA+IDA6CiAgICAgICAgICAgIHRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIGdyYWRfY2xpcF9ub3JtKQogICAgICAgIG9wdGltaXplci5zdGVwKCkKCiAgICAgICAgbiA9IGxlbihYX3MpCiAgICAgICAgdG90YWxzWyJjbHNfbG9zcyJdICs9IGNsc19sb3NzLml0ZW0oKSAqIG4KICAgICAgICB0b3RhbHNbImRvbWFpbl9sb3NzIl0gKz0gZG9tYWluX2xvc3MuaXRlbSgpICogbgogICAgICAgIHRvdGFsc1siYW5jaG9yX2xvc3MiXSArPSBhbmNob3JfbG9zcy5pdGVtKCkgKiBuCiAgICAgICAgdG90YWxzWyJjb3JyZWN0Il0gKz0gKGNsYXNzX2xvZ2l0cy5hcmdtYXgoZGltPTEpID09IGxhYmVsX3MpLnN1bSgpLml0ZW0oKQogICAgICAgICMgQmFsYW5jZWQgZG9tYWluIGFjY3VyYWN5IChzZWUgZG9tYWluX2NvcnJlY3QvZG9tYWluX3RvdGFsIGFib3ZlKToKICAgICAgICAjIH4wLjMzIG1lYW5zIHRoZSBkaXNjcmltaW5hdG9yIGdlbnVpbmVseSBjYW4ndCB0ZWxsIHRoZSBkb21haW5zCiAgICAgICAgIyBhcGFydDsgaGlnaGVyIG1lYW5zIGl0IHN0aWxsIGNhbi4gTXVjaCBlYXNpZXIgdG8gcmVhZCBsaXZlIHRoYW4gYQogICAgICAgICMgY3Jvc3MtZW50cm9weSBob3ZlcmluZyBuZWFyIGxuKDMpPTEuMDk5IC0tIGFuZCwgdW5saWtlIHJhdwogICAgICAgICMgYWNjdXJhY3ksIG5vdCBpbmZsYXRlZCBieSB0aGUgYmF0Y2gncyA1MCUgdGFyZ2V0IHNoYXJlLgogICAgICAgIGRvbWFpbl9wcmVkcyA9IGRvbWFpbl9sb2dpdHMuYXJnbWF4KGRpbT0xKS5jcHUoKQogICAgICAgIGRvbWFpbl90cnV0aCA9IGRvbWFpbl90YXJnZXRzLmNwdSgpCiAgICAgICAgZm9yIGQgaW4gcmFuZ2UoTl9ET01BSU5TKToKICAgICAgICAgICAgbWFzayA9IGRvbWFpbl90cnV0aCA9PSBkCiAgICAgICAgICAgIGRvbWFpbl90b3RhbFtkXSArPSBtYXNrLnN1bSgpCiAgICAgICAgICAgIGRvbWFpbl9jb3JyZWN0W2RdICs9IChkb21haW5fcHJlZHNbbWFza10gPT0gZCkuc3VtKCkKICAgICAgICB0b3RhbHNbImxhbWJkYV9zdW0iXSArPSBsYW1iZGFfICogbgogICAgICAgIHRvdGFsc1sibiJdICs9IG4KCiAgICBuID0gdG90YWxzLnBvcCgibiIpCiAgICBzZWVuID0gZG9tYWluX3RvdGFsID4gMAogICAgYmFsYW5jZWRfZG9tYWluX2FjYyA9IGZsb2F0KChkb21haW5fY29ycmVjdFtzZWVuXSAvIGRvbWFpbl90b3RhbFtzZWVuXSkubWVhbigpKQogICAgcmV0dXJuIHsKICAgICAgICAiY2xzX2xvc3MiOiB0b3RhbHNbImNsc19sb3NzIl0gLyBuLAogICAgICAgICJkb21haW5fbG9zcyI6IHRvdGFsc1siZG9tYWluX2xvc3MiXSAvIG4sCiAgICAgICAgImFuY2hvcl9sb3NzIjogdG90YWxzWyJhbmNob3JfbG9zcyJdIC8gbiwKICAgICAgICAiY2xzX2FjYyI6IHRvdGFsc1siY29ycmVjdCJdIC8gbiwKICAgICAgICAiZG9tYWluX2FjYyI6IGJhbGFuY2VkX2RvbWFpbl9hY2MsCiAgICAgICAgImxhbWJkYSI6IHRvdGFsc1sibGFtYmRhX3N1bSJdIC8gbiwKICAgIH0KCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBydW5fdmFsX2Vwb2NoKG1vZGVsLCB2YWxfbG9hZGVyLCBkZXZpY2UpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdF06CiAgICBtb2RlbC5ldmFsKCkKICAgIGNyaXRlcmlvbiA9IHRvcmNoLm5uLkNyb3NzRW50cm9weUxvc3MoKQogICAgdG90YWxfbG9zcywgdG90YWxfY29ycmVjdCwgbiA9IDAuMCwgMCwgMAogICAgZm9yIFgsIF9kb21haW5faWQsIGxhYmVsIGluIHZhbF9sb2FkZXI6CiAgICAgICAgWCwgbGFiZWwgPSBYLnRvKGRldmljZSksIGxhYmVsLnRvKGRldmljZSkKICAgICAgICBfZmVhdHVyZXMsIGNsYXNzX2xvZ2l0cywgX2RvbWFpbl9sb2dpdHMgPSBtb2RlbChYKQogICAgICAgIGxvc3MgPSBjcml0ZXJpb24oY2xhc3NfbG9naXRzLCBsYWJlbCkKICAgICAgICB0b3RhbF9sb3NzICs9IGxvc3MuaXRlbSgpICogbGVuKFgpCiAgICAgICAgdG90YWxfY29ycmVjdCArPSAoY2xhc3NfbG9naXRzLmFyZ21heChkaW09MSkgPT0gbGFiZWwpLnN1bSgpLml0ZW0oKQogICAgICAgIG4gKz0gbGVuKFgpCiAgICByZXR1cm4gdG90YWxfbG9zcyAvIG4sIHRvdGFsX2NvcnJlY3QgLyBuCgoKZGVmIHRyYWluKGNvbmZpZzogQ29uZmlnLCBwdV9yb290LCBmb3JjZV9kYXRhOiBib29sID0gRmFsc2UpIC0+IE5vbmU6CiAgICBzZXRfc2VlZChjb25maWcuc2VlZCkKCiAgICBtb2RlbHNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLm1vZGVsc19kaXIpCiAgICByZXBvcnRzX2RpciA9IGNvbmZpZy5yZXNvbHZlKGNvbmZpZy5wYXRocy5yZXBvcnRzX2RpcikKICAgIG1vZGVsc19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmVwb3J0c19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIG5vcm1hbGl6ZXJfcGF0aCA9IG1vZGVsc19kaXIgLyAibm9ybWFsaXplci5ucHoiCiAgICBpZiBub3Qgbm9ybWFsaXplcl9wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmIntub3JtYWxpemVyX3BhdGh9IG5vdCBmb3VuZCAtLSBydW4gYHB5dGhvbiAtbSBkb21haW5fYWRhcHRhdGlvbi50cmFpbl9iYXNlbGluZWAgZmlyc3QgIgogICAgICAgICAgICAiKGl0IGZpdHMgYW5kIHNhdmVzIHRoZSBub3JtYWxpemVyIGJvdGggbW9kZWxzIHNoYXJlKS4iCiAgICAgICAgKQogICAgbm9ybWFsaXplciA9IFNwZWN0cm9ncmFtTm9ybWFsaXplci5sb2FkKG5vcm1hbGl6ZXJfcGF0aCkKCiAgICBwcmludCgiQnVpbGRpbmcgZGF0YXNldHMgKGNhY2hlZCBhZnRlciBmaXJzdCBydW4pLi4uIikKICAgIGRhdGFzZXRzID0gYnVpbGRfZG9tYWluX2FkYXB0YXRpb25fZGF0YXNldHMoY29uZmlnLCBwdV9yb290LCBmb3JjZT1mb3JjZV9kYXRhKQogICAgdHJhaW5fWCwgdHJhaW5fbWV0YSA9IGRhdGFzZXRzWyJzb3VyY2VfdHJhaW4iXQogICAgdmFsX1gsIHZhbF9tZXRhID0gZGF0YXNldHNbInNvdXJjZV92YWwiXQogICAgYWRhcHRfWCwgYWRhcHRfbWV0YSA9IGRhdGFzZXRzWyJ0YXJnZXRfYWRhcHQiXQogICAgcHJpbnQoZiIgIHNvdXJjZV90cmFpbjoge3RyYWluX1guc2hhcGV9ICBzb3VyY2VfdmFsOiB7dmFsX1guc2hhcGV9ICB0YXJnZXRfYWRhcHQ6IHthZGFwdF9YLnNoYXBlfSIpCgogICAgZXhwZWN0ZWQgPSBleHBlY3RlZF9zaGFwZShjb25maWcuY29tbW9uLCBjb25maWcuc3RmdCkKICAgIHRyYWluX1hfbm9ybSA9IG5vcm1hbGl6ZXIudHJhbnNmb3JtKHRyYWluX1gpCiAgICB2YWxfWF9ub3JtID0gbm9ybWFsaXplci50cmFuc2Zvcm0odmFsX1gpCiAgICBhZGFwdF9YX25vcm0gPSBub3JtYWxpemVyLnRyYW5zZm9ybShhZGFwdF9YKQogICAgIyBTZWUgdHJhaW5fYmFzZWxpbmUucHk6IGRyb3AgdGhlIHVuLW5vcm1hbGl6ZWQgY29waWVzLCB0aGV5IGFyZSBjYWNoZWQuCiAgICBkZWwgdHJhaW5fWCwgdmFsX1gsIGFkYXB0X1gsIGRhdGFzZXRzCiAgICBnYy5jb2xsZWN0KCkKCiAgICBkZXZpY2UgPSBzZWxlY3RfZGV2aWNlKGNvbmZpZy50cmFpbl9kYW5uLmRldmljZSkKICAgIHByaW50KGYiVXNpbmcgZGV2aWNlOiB7ZGV2aWNlfSIpCgogICAgc291cmNlX2xvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgTGFiZWxlZERvbWFpbkRhdGFzZXQodHJhaW5fWF9ub3JtLCB0cmFpbl9tZXRhLmRvbWFpbl9pZC50b19udW1weSgpLCB0cmFpbl9tZXRhLnVuaXZlcnNhbF9sYWJlbC50b19udW1weSgpKSwKICAgICAgICBiYXRjaF9zaXplPWNvbmZpZy50cmFpbl9kYW5uLmJhdGNoX3NpemUsCiAgICAgICAgc2h1ZmZsZT1UcnVlLAogICAgICAgIGRyb3BfbGFzdD1UcnVlLCAgIyBrZWVwcyBldmVyeSBzb3VyY2UgYmF0Y2ggZnVsbC1zaXplLCBzbyB0aGUgcGFpcmVkIHRhcmdldCBiYXRjaCBzaXplIGlzIG1lYW5pbmdmdWwgdG9vCiAgICApCiAgICB0YXJnZXRfbG9hZGVyID0gRGF0YUxvYWRlcigKICAgICAgICBVbmxhYmVsZWREb21haW5EYXRhc2V0KGFkYXB0X1hfbm9ybSwgYWRhcHRfbWV0YS5kb21haW5faWQudG9fbnVtcHkoKSksCiAgICAgICAgYmF0Y2hfc2l6ZT1jb25maWcudHJhaW5fZGFubi5iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9VHJ1ZSwKICAgICAgICBkcm9wX2xhc3Q9VHJ1ZSwKICAgICkKICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKAogICAgICAgIExhYmVsZWREb21haW5EYXRhc2V0KHZhbF9YX25vcm0sIHZhbF9tZXRhLmRvbWFpbl9pZC50b19udW1weSgpLCB2YWxfbWV0YS51bml2ZXJzYWxfbGFiZWwudG9fbnVtcHkoKSksCiAgICAgICAgYmF0Y2hfc2l6ZT1jb25maWcudHJhaW5fZGFubi5iYXRjaF9zaXplLAogICAgICAgIHNodWZmbGU9RmFsc2UsCiAgICApCgogICAgbW9kZWwgPSBEQU5OTW9kZWwoZXhwZWN0ZWQsIGNvbmZpZykudG8oZGV2aWNlKQogICAgbl9wYXJhbXMgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIHByaW50KGYiTW9kZWwgcGFyYW1ldGVyczoge25fcGFyYW1zOix9IikKCiAgICAjIFRoZSBkb21haW4gZGlzY3JpbWluYXRvciBnZXRzIGl0cyBvd24gcGFyYW1ldGVyIGdyb3VwIHNvIGl0cyBsZWFybmluZwogICAgIyByYXRlIGNhbiBiZSByYWlzZWQgaW5kZXBlbmRlbnRseSAoY29uZmlnJ3MgZGlzY3JpbWluYXRvcl9scl9tdWx0aXBsaWVyLAogICAgIyBkZWZhdWx0IDEuMCA9IG5vIGNoYW5nZSkuIFdoeSB0aGUga25vYiBleGlzdHM6IGluIHRoZSBtaW5pbWF4IGdhbWUgdGhlCiAgICAjIGRpc2NyaW1pbmF0b3IgaGFzIHRvIHN0YXkgb25lIHN0ZXAgQUhFQUQgb2YgdGhlIGZlYXR1cmUgZXh0cmFjdG9yIHRvCiAgICAjIHByb2R1Y2UgbWVhbmluZ2Z1bCByZXZlcnNlZCBncmFkaWVudHMuIElmIGRvbWFpbl9hY2Mgc2l0cyBhdCBjaGFuY2UKICAgICMgKH4xLzMpIHdoaWxlIHRoZSBwb3N0LWhvYyBwcm9iZSBpbiBldmFsdWF0ZS5weSBzdGlsbCBzZXBhcmF0ZXMgdGhlCiAgICAjIGRvbWFpbnMgZWFzaWx5LCB0aGF0IG1lYW5zIHRoZSBpbi10cmFpbmluZyBkaXNjcmltaW5hdG9yIGlzIHNpbXBseQogICAgIyBsb3NpbmcgdGhlIHJhY2UgLS0gbm90IHRoYXQgdGhlIGZlYXR1cmVzIGFyZSBhbGlnbmVkIC0tIGFuZCB0aGlzIGlzIHRoZQogICAgIyBkaWFsIHRvIHR1cm4uIFNlZSBET01BSU5fQURBUFRBVElPTl9QTEFOLm1kJ3Mga2VybmVsIHYyIG5vdGVzLgogICAgYmFzZV9sciA9IGNvbmZpZy50cmFpbl9kYW5uLmxlYXJuaW5nX3JhdGUKICAgIGRpc2NyaW1pbmF0b3JfbHIgPSBiYXNlX2xyICogY29uZmlnLnRyYWluX2Rhbm4uZGlzY3JpbWluYXRvcl9scl9tdWx0aXBsaWVyCiAgICBkaXNjcmltaW5hdG9yX3BhcmFtcyA9IGxpc3QobW9kZWwuZG9tYWluX2Rpc2NyaW1pbmF0b3IucGFyYW1ldGVycygpKQogICAgZGlzY3JpbWluYXRvcl9wYXJhbV9pZHMgPSB7aWQocCkgZm9yIHAgaW4gZGlzY3JpbWluYXRvcl9wYXJhbXN9CiAgICBiYWNrYm9uZV9wYXJhbXMgPSBbcCBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgaWQocCkgbm90IGluIGRpc2NyaW1pbmF0b3JfcGFyYW1faWRzXQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbSgKICAgICAgICBbCiAgICAgICAgICAgIHsicGFyYW1zIjogYmFja2JvbmVfcGFyYW1zLCAibHIiOiBiYXNlX2xyfSwKICAgICAgICAgICAgeyJwYXJhbXMiOiBkaXNjcmltaW5hdG9yX3BhcmFtcywgImxyIjogZGlzY3JpbWluYXRvcl9scn0sCiAgICAgICAgXSwKICAgICAgICB3ZWlnaHRfZGVjYXk9Y29uZmlnLnRyYWluX2Rhbm4ud2VpZ2h0X2RlY2F5LAogICAgKQogICAgaWYgY29uZmlnLnRyYWluX2Rhbm4uZGlzY3JpbWluYXRvcl9scl9tdWx0aXBsaWVyICE9IDEuMDoKICAgICAgICBwcmludChmIkRpc2NyaW1pbmF0b3IgTFI6IHtkaXNjcmltaW5hdG9yX2xyOi4yZX0gKHtjb25maWcudHJhaW5fZGFubi5kaXNjcmltaW5hdG9yX2xyX211bHRpcGxpZXJ9eCBiYXNlIHtiYXNlX2xyOi4yZX0pIikKCiAgICBiZXN0X3ZhbF9sb3NzID0gZmxvYXQoImluZiIpCiAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA9IDAKICAgIGhpc3RvcnkgPSBbXQogICAgYmVzdF9ja3B0X3BhdGggPSBtb2RlbHNfZGlyIC8gImRhbm5fYmVzdC5wdCIKICAgIHRvdGFsX2Vwb2NocyA9IGNvbmZpZy50cmFpbl9kYW5uLmVwb2NocwoKICAgIGZvciBlcG9jaCBpbiByYW5nZSgxLCB0b3RhbF9lcG9jaHMgKyAxKToKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgIyBOQjogbGFtYmRhIGlzIHNldCBwZXIgSVRFUkFUSU9OIGluc2lkZSBydW5fdHJhaW5fZXBvY2ggKHYxIHNldCBpdAogICAgICAgICMgb25jZSBwZXIgZXBvY2ggaGVyZSAtLSBzZWUgdGhhdCBmdW5jdGlvbidzIGRvY3N0cmluZyBmb3Igd2h5IHRoZQogICAgICAgICMgZGlmZmVyZW5jZSBtYXR0ZXJzIG9uY2UgZ3JsX2dhbW1hIGlzIHJhaXNlZCkuCiAgICAgICAgdHJhaW5fbWV0cmljcyA9IHJ1bl90cmFpbl9lcG9jaCgKICAgICAgICAgICAgbW9kZWwsIHNvdXJjZV9sb2FkZXIsIHRhcmdldF9sb2FkZXIsIG9wdGltaXplciwgZGV2aWNlLAogICAgICAgICAgICBjb25maWcudHJhaW5fZGFubi5kb21haW5fbG9zc193ZWlnaHQsIGNvbmZpZy50cmFpbl9kYW5uLmFuY2hvcl9sb3NzX3dlaWdodCwKICAgICAgICAgICAgZXBvY2gsIHRvdGFsX2Vwb2NocywgY29uZmlnLnRyYWluX2Rhbm4uZ3JsX2dhbW1hLCBjb25maWcudHJhaW5fZGFubi5ncmFkX2NsaXBfbm9ybSwKICAgICAgICApCiAgICAgICAgdmFsX2xvc3MsIHZhbF9hY2MgPSBydW5fdmFsX2Vwb2NoKG1vZGVsLCB2YWxfbG9hZGVyLCBkZXZpY2UpCiAgICAgICAgZHQgPSB0aW1lLnRpbWUoKSAtIHQwCgogICAgICAgIGltcHJvdmVkID0gdmFsX2xvc3MgPCBiZXN0X3ZhbF9sb3NzCiAgICAgICAgaWYgaW1wcm92ZWQ6CiAgICAgICAgICAgIGJlc3RfdmFsX2xvc3MgPSB2YWxfbG9zcwogICAgICAgICAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA9IDAKICAgICAgICAgICAgdG9yY2guc2F2ZShtb2RlbC5zdGF0ZV9kaWN0KCksIGJlc3RfY2twdF9wYXRoKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGVwb2Noc193aXRob3V0X2ltcHJvdmVtZW50ICs9IDEKCiAgICAgICAgbWFya2VyID0gIiAqIiBpZiBpbXByb3ZlZCBlbHNlICIiCiAgICAgICAgcHJpbnQoCiAgICAgICAgICAgIGYiZXBvY2gge2Vwb2NoOjNkfS97dG90YWxfZXBvY2hzfSAgbGFtYmRhfnt0cmFpbl9tZXRyaWNzWydsYW1iZGEnXTouM2Z9ICAiCiAgICAgICAgICAgIGYiY2xzX2xvc3M9e3RyYWluX21ldHJpY3NbJ2Nsc19sb3NzJ106LjRmfSBjbHNfYWNjPXt0cmFpbl9tZXRyaWNzWydjbHNfYWNjJ106LjNmfSAgIgogICAgICAgICAgICBmImRvbWFpbl9sb3NzPXt0cmFpbl9tZXRyaWNzWydkb21haW5fbG9zcyddOi40Zn0gZG9tYWluX2FjYz17dHJhaW5fbWV0cmljc1snZG9tYWluX2FjYyddOi4zZn0gIgogICAgICAgICAgICBmImFuY2hvcl9sb3NzPXt0cmFpbl9tZXRyaWNzWydhbmNob3JfbG9zcyddOi41Zn0gICIKICAgICAgICAgICAgZiJ2YWxfbG9zcz17dmFsX2xvc3M6LjRmfSB2YWxfYWNjPXt2YWxfYWNjOi4zZn0gICh7ZHQ6LjFmfXMpe21hcmtlcn0iCiAgICAgICAgKQogICAgICAgIGhpc3RvcnkuYXBwZW5kKHsiZXBvY2giOiBlcG9jaCwgInZhbF9sb3NzIjogdmFsX2xvc3MsICJ2YWxfYWNjIjogdmFsX2FjYywgKip0cmFpbl9tZXRyaWNzfSkKCiAgICAgICAgaWYgZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgPj0gY29uZmlnLnRyYWluX2Rhbm4uZWFybHlfc3RvcHBpbmdfcGF0aWVuY2U6CiAgICAgICAgICAgIHByaW50KGYiRWFybHkgc3RvcHBpbmc6IG5vIHZhbCBpbXByb3ZlbWVudCBpbiB7Y29uZmlnLnRyYWluX2Rhbm4uZWFybHlfc3RvcHBpbmdfcGF0aWVuY2V9IGVwb2Nocy4iKQogICAgICAgICAgICBicmVhawoKICAgIHBkLkRhdGFGcmFtZShoaXN0b3J5KS50b19jc3YocmVwb3J0c19kaXIgLyAiZGFubl90cmFpbmluZ19sb2cuY3N2IiwgaW5kZXg9RmFsc2UpCiAgICBwcmludChmIlxuQmVzdCB2YWwgbG9zczoge2Jlc3RfdmFsX2xvc3M6LjRmfSIpCiAgICBwcmludChmIkNoZWNrcG9pbnQgc2F2ZWQgdG86IHtiZXN0X2NrcHRfcGF0aH0iKQogICAgcHJpbnQoIk5leHQ6IHB5dGhvbiAtbSBkb21haW5fYWRhcHRhdGlvbi5ldmFsdWF0ZSAtLWNvbmZpZyA8c2FtZSBjb25maWc+IikKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PSJjb25maWdzL2RvbWFpbl9hZGFwdGF0aW9uLnlhbWwiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1wdS1yb290IiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJSb290IGRpciBvZiB0aGUgUFUgZGF0YXNldCAoZGVmYXVsdDogYXV0by1kZXRlY3QgL2thZ2dsZS9pbnB1dCwgZWxzZSBjb25maWcncyBwdV9yb290X2xvY2FsKS4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mb3JjZS1kYXRhIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iUmVjb21wdXRlIGNhY2hlZCBzcGVjdHJvZ3JhbXMgZXZlbiBpZiBwcmVzZW50LiIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgIGNvbmZpZyA9IGxvYWRfY29uZmlnKGFyZ3MuY29uZmlnKQogICAgcHVfcm9vdCA9IGFyZ3MucHVfcm9vdCBvciBmaW5kX3B1X3Jvb3QoY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnB1X3Jvb3RfbG9jYWwpKQogICAgdHJhaW4oY29uZmlnLCBwdV9yb290PXB1X3Jvb3QsIGZvcmNlX2RhdGE9YXJncy5mb3JjZV9kYXRhKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'
    ),
    'src/domain_adaptation/evaluate.py': (
        'IiIiUGhhc2UgNDogdGhlIGFjdHVhbCB6ZXJvLXNob3QgY2hlY2suIFJ1bnMgdGhlIFBoYXNlIDIgYmFzZWxpbmUgYW5kIChpZgp0cmFpbmVkKSB0aGUgUGhhc2UgMyBEQU5OIG1vZGVsIG9uIFRBUkdFVF9URVNUIC0tIHRoZSBoZWxkLW91dCBvcGVyYXRpbmcKY29uZGl0aW9uJ3MgcmVhbC1kYW1hZ2UgYmVhcmluZ3MsIHdob3NlIGZhdWx0IGxhYmVscyBuZWl0aGVyIG1vZGVsIGhhcwpldmVyIHNlZW4gLS0gYW5kIGNvbXBhcmVzIHRoZW0uCgpUQVJHRVRfVEVTVCBtaXhlcyBoZWxkLW91dCBoZWFsdGh5IHJvd3MgKGxhYmVsPTApIHdpdGggZXZlcnkgcmVhbC1kYW1hZ2UKYmVhcmluZydzIHJvd3MgKGxhYmVsPTEgaW5uZXJfcmFjZSwgMiBvdXRlcl9yYWNlLCBzZWUgZGF0YS9kYXRhc2V0LnB5KSAtLQpzbyB0aGlzIGlzIGEgZ2VudWluZSAzLWNsYXNzIGV2YWx1YXRpb24sIG5vdCBqdXN0ICJkb2VzIGl0IGZsYWcgZmF1bHRzLiIKCkFsc28gcHJvZHVjZXMgdGhlIGZlYXR1cmUtc3BhY2UgdmlzdWFsaXphdGlvbiBET01BSU5fQURBUFRBVElPTl9QTEFOLm1kJ3MKUGhhc2UgNCBhc2tzIGZvcjogYSAyRCBwcm9qZWN0aW9uICh0LVNORSkgb2YgdGhlIGZlYXR1cmUgZXh0cmFjdG9yJ3MKZW1iZWRkaW5ncyBhY3Jvc3MgU09VUkNFX1ZBTCArIFRBUkdFVF9URVNULCBjb2xvcmVkIGJ5IGRvbWFpbiAoZG8gdGhlCmRvbWFpbnMgb3ZlcmxhcD8pIGFuZCBieSBmYXVsdCBjbGFzcyAoZG8gdGFyZ2V0IHJlYWwgZmF1bHRzIGxhbmQgaW5zaWRlCnRoZSBzb3VyY2UgZmF1bHQgY2x1c3RlcnM/KSAtLSB1c2luZyB0aGUgREFOTiBmZWF0dXJlIGV4dHJhY3RvciBpZiBpdApleGlzdHMsIGVsc2UgdGhlIGJhc2VsaW5lJ3MsIHNvIHRoZXJlJ3MgYWx3YXlzIHNvbWV0aGluZyB0byBsb29rIGF0IGV2ZW4KYmVmb3JlIFBoYXNlIDMgaGFzIGJlZW4gcnVuLgoKVXNhZ2U6CiAgICBweXRob24gLW0gZG9tYWluX2FkYXB0YXRpb24uZXZhbHVhdGUgLS1jb25maWcgY29uZmlncy9kb21haW5fYWRhcHRhdGlvbi55YW1sIFstLXB1LXJvb3QgRElSXQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gc2tsZWFybi5tYW5pZm9sZCBpbXBvcnQgVFNORQpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgYWNjdXJhY3lfc2NvcmUsIGNvbmZ1c2lvbl9tYXRyaXgsIGYxX3Njb3JlCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgoKZnJvbSB2aWJyYXRpb25fYWQuZmVhdHVyZXMuc3BlY3Ryb2dyYW0gaW1wb3J0IFNwZWN0cm9ncmFtTm9ybWFsaXplciwgZXhwZWN0ZWRfc2hhcGUKCmZyb20gZG9tYWluX2FkYXB0YXRpb24gaW1wb3J0IHZpegpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmFuYWx5c2lzIGltcG9ydCBiaW5hcnlfZGV0ZWN0aW9uX21ldHJpY3MsIGRvbWFpbl9zZXBhcmFiaWxpdHlfcHJvYmUKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5jb25maWcgaW1wb3J0IENvbmZpZywgbG9hZF9jb25maWcKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5kYXRhLmRhdGFzZXQgaW1wb3J0IGJ1aWxkX2RvbWFpbl9hZGFwdGF0aW9uX2RhdGFzZXRzCmZyb20gZG9tYWluX2FkYXB0YXRpb24uZGF0YS5wdV9sb2FkZXIgaW1wb3J0IGZpbmRfcHVfcm9vdApmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEudG9yY2hfZGF0YXNldCBpbXBvcnQgTGFiZWxlZERvbWFpbkRhdGFzZXQKZnJvbSBkb21haW5fYWRhcHRhdGlvbi5tb2RlbHMuZGFubiBpbXBvcnQgREFOTk1vZGVsLCBGZWF0dXJlQ2xhc3NpZmllcgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLm1vZGVscy5mZWF0dXJlX2V4dHJhY3RvciBpbXBvcnQgc2VsZWN0X2RldmljZQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIHJ1bl9pbmZlcmVuY2UobW9kZWwsIFhfbm9ybTogbnAubmRhcnJheSwgZG9tYWluX2lkczogbnAubmRhcnJheSwgbGFiZWxzOiBucC5uZGFycmF5LCBkZXZpY2UsIGJhdGNoX3NpemU6IGludCA9IDY0KToKICAgIGxvYWRlciA9IERhdGFMb2FkZXIoTGFiZWxlZERvbWFpbkRhdGFzZXQoWF9ub3JtLCBkb21haW5faWRzLCBsYWJlbHMpLCBiYXRjaF9zaXplPWJhdGNoX3NpemUsIHNodWZmbGU9RmFsc2UpCiAgICBtb2RlbC5ldmFsKCkKICAgIGFsbF9mZWF0dXJlcywgYWxsX3ByZWRzLCBhbGxfbGFiZWxzID0gW10sIFtdLCBbXQogICAgZm9yIFgsIF9kb21haW5faWQsIGxhYmVsIGluIGxvYWRlcjoKICAgICAgICBYID0gWC50byhkZXZpY2UpCiAgICAgICAgb3V0cHV0cyA9IG1vZGVsKFgpICAjIChmZWF0dXJlcywgY2xhc3NfbG9naXRzWywgZG9tYWluX2xvZ2l0c10pCiAgICAgICAgZmVhdHVyZXMsIGNsYXNzX2xvZ2l0cyA9IG91dHB1dHNbMF0sIG91dHB1dHNbMV0KICAgICAgICBhbGxfZmVhdHVyZXMuYXBwZW5kKGZlYXR1cmVzLmNwdSgpLm51bXB5KCkpCiAgICAgICAgYWxsX3ByZWRzLmFwcGVuZChjbGFzc19sb2dpdHMuYXJnbWF4KGRpbT0xKS5jcHUoKS5udW1weSgpKQogICAgICAgIGFsbF9sYWJlbHMuYXBwZW5kKGxhYmVsLm51bXB5KCkpCiAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUoYWxsX2ZlYXR1cmVzKSwgbnAuY29uY2F0ZW5hdGUoYWxsX3ByZWRzKSwgbnAuY29uY2F0ZW5hdGUoYWxsX2xhYmVscykKCgpkZWYgY29tcHV0ZV9tZXRyaWNzKHlfdHJ1ZTogbnAubmRhcnJheSwgeV9wcmVkOiBucC5uZGFycmF5KSAtPiBkaWN0OgogICAgY20gPSBjb25mdXNpb25fbWF0cml4KHlfdHJ1ZSwgeV9wcmVkLCBsYWJlbHM9WzAsIDEsIDJdKQogICAgcmV0dXJuIHsKICAgICAgICAiYWNjdXJhY3kiOiBmbG9hdChhY2N1cmFjeV9zY29yZSh5X3RydWUsIHlfcHJlZCkpLAogICAgICAgICJtYWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtYWNybyIsIGxhYmVscz1bMCwgMSwgMl0sIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJjb25mdXNpb25fbWF0cml4IjogY20udG9saXN0KCksCiAgICAgICAgIm4iOiBpbnQobGVuKHlfdHJ1ZSkpLAogICAgICAgICMgUmVwb3J0ZWQgYWxvbmdzaWRlIHRoZSAzLWNsYXNzIGZpZ3VyZXMsIG5vdCBpbnN0ZWFkIG9mIHRoZW06IGFjcm9zcwogICAgICAgICMga2VybmVscyB2MS12NSBmYXVsdCBUWVBJTkcgbmV2ZXIgdHJhbnNmZXJyZWQgd2hpbGUgZmF1bHQgREVURUNUSU9OCiAgICAgICAgIyBwYXJ0bHkgZGlkLCBhbmQgZGV0ZWN0aW9uIGlzIHRoZSBxdWVzdGlvbiB0aGUgZGVwbG95bWVudCBhY3R1YWxseQogICAgICAgICMgcG9zZXMuIFNlZSBhbmFseXNpcy5iaW5hcnlfZGV0ZWN0aW9uX21ldHJpY3MuCiAgICAgICAgImJpbmFyeV9kZXRlY3Rpb24iOiBiaW5hcnlfZGV0ZWN0aW9uX21ldHJpY3MoeV90cnVlLCB5X3ByZWQpLAogICAgfQoKCmRlZiBfcHJpbnRfc3VtbWFyeShuYW1lOiBzdHIsIG1ldHJpY3M6IGRpY3QpIC0+IE5vbmU6CiAgICBiID0gbWV0cmljc1siYmluYXJ5X2RldGVjdGlvbiJdCiAgICBwcmludChmIiAge25hbWV9OiAzLWNsYXNzIGFjY3VyYWN5PXttZXRyaWNzWydhY2N1cmFjeSddOi4zZn0gbWFjcm9fZjE9e21ldHJpY3NbJ21hY3JvX2YxJ106LjNmfSIpCiAgICBwcmludCgKICAgICAgICBmIiAge25hbWV9OiBCSU5BUlkgaGVhbHRoeS12cy1mYXVsdHkgLS0gcHJlY2lzaW9uPXtiWydwcmVjaXNpb24nXTouM2Z9ICIKICAgICAgICBmInJlY2FsbD17YlsncmVjYWxsJ106LjNmfSBmMT17YlsnZjEnXTouM2Z9IGZhbHNlX2FsYXJtX3JhdGU9e2JbJ2ZhbHNlX2FsYXJtX3JhdGUnXTouMSV9ICIKICAgICAgICBmIihhbHdheXMtZmF1bHR5IGFjY3VyYWN5IHdvdWxkIGJlIHtiWydhbHdheXNfZmF1bHR5X2FjY3VyYWN5J106LjNmfSkiCiAgICApCiAgICBwcmludChqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yKSkKCgpkZWYgZXZhbHVhdGUoY29uZmlnOiBDb25maWcsIHB1X3Jvb3QpIC0+IGRpY3Q6CiAgICBtb2RlbHNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLm1vZGVsc19kaXIpCiAgICByZXBvcnRzX2RpciA9IGNvbmZpZy5yZXNvbHZlKGNvbmZpZy5wYXRocy5yZXBvcnRzX2RpcikKICAgIGZpZ3VyZXNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLmZpZ3VyZXNfZGlyKQogICAgZm9yIGQgaW4gKHJlcG9ydHNfZGlyLCBmaWd1cmVzX2Rpcik6CiAgICAgICAgZC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgYmFzZWxpbmVfY2twdCA9IG1vZGVsc19kaXIgLyAiYmFzZWxpbmVfYmVzdC5wdCIKICAgIGRhbm5fY2twdCA9IG1vZGVsc19kaXIgLyAiZGFubl9iZXN0LnB0IgogICAgaWYgbm90IGJhc2VsaW5lX2NrcHQuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJ7YmFzZWxpbmVfY2twdH0gbm90IGZvdW5kIC0tIHJ1biB0cmFpbl9iYXNlbGluZS5weSBmaXJzdC4iKQoKICAgIG5vcm1hbGl6ZXIgPSBTcGVjdHJvZ3JhbU5vcm1hbGl6ZXIubG9hZChtb2RlbHNfZGlyIC8gIm5vcm1hbGl6ZXIubnB6IikKICAgIGV4cGVjdGVkID0gZXhwZWN0ZWRfc2hhcGUoY29uZmlnLmNvbW1vbiwgY29uZmlnLnN0ZnQpCiAgICBkZXZpY2UgPSBzZWxlY3RfZGV2aWNlKGNvbmZpZy50cmFpbl9iYXNlbGluZS5kZXZpY2UpCgogICAgcHJpbnQoIkxvYWRpbmcgY2FjaGVkIGRhdGFzZXRzLi4uIikKICAgIGRhdGFzZXRzID0gYnVpbGRfZG9tYWluX2FkYXB0YXRpb25fZGF0YXNldHMoY29uZmlnLCBwdV9yb290KQogICAgdGVzdF9YLCB0ZXN0X21ldGEgPSBkYXRhc2V0c1sidGFyZ2V0X3Rlc3QiXQogICAgdmFsX1gsIHZhbF9tZXRhID0gZGF0YXNldHNbInNvdXJjZV92YWwiXQogICAgdGVzdF9YX25vcm0gPSBub3JtYWxpemVyLnRyYW5zZm9ybSh0ZXN0X1gpCiAgICB2YWxfWF9ub3JtID0gbm9ybWFsaXplci50cmFuc2Zvcm0odmFsX1gpCgogICAgcmVzdWx0czogZGljdFtzdHIsIGRpY3RdID0ge30KICAgIGZlYXR1cmVfc291cmNlID0gTm9uZSAgIyB3aGljaCBtb2RlbCdzIGZlYXR1cmVzIGZlZWQgdGhlIHQtU05FIHBsb3Q6IHByZWZlciBEQU5OLCBlbHNlIGJhc2VsaW5lCgogICAgcHJpbnQoIkV2YWx1YXRpbmcgYmFzZWxpbmUgKG5vIGFkYXB0YXRpb24pIG9uIFRBUkdFVF9URVNULi4uIikKICAgIGJhc2VsaW5lID0gRmVhdHVyZUNsYXNzaWZpZXIoZXhwZWN0ZWQsIGNvbmZpZykudG8oZGV2aWNlKQogICAgYmFzZWxpbmUubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQoYmFzZWxpbmVfY2twdCwgbWFwX2xvY2F0aW9uPWRldmljZSkpCiAgICB0ZXN0X2RvbWFpbl9pZHMgPSB0ZXN0X21ldGEuZG9tYWluX2lkLnRvX251bXB5KCkKICAgIHRlc3RfbGFiZWxzID0gdGVzdF9tZXRhLnVuaXZlcnNhbF9sYWJlbC50b19udW1weSgpCiAgICBiYXNlbGluZV9mZWF0dXJlcywgYmFzZWxpbmVfcHJlZHMsIF8gPSBydW5faW5mZXJlbmNlKGJhc2VsaW5lLCB0ZXN0X1hfbm9ybSwgdGVzdF9kb21haW5faWRzLCB0ZXN0X2xhYmVscywgZGV2aWNlKQogICAgcmVzdWx0c1siYmFzZWxpbmUiXSA9IGNvbXB1dGVfbWV0cmljcyh0ZXN0X2xhYmVscywgYmFzZWxpbmVfcHJlZHMpCiAgICBmZWF0dXJlX3NvdXJjZSA9ICgiYmFzZWxpbmUiLCBiYXNlbGluZSkKICAgIF9wcmludF9zdW1tYXJ5KCJiYXNlbGluZSIsIHJlc3VsdHNbImJhc2VsaW5lIl0pCgogICAgaWYgZGFubl9ja3B0LmV4aXN0cygpOgogICAgICAgIHByaW50KCJFdmFsdWF0aW5nIERBTk4gKGFkYXB0ZWQpIG9uIFRBUkdFVF9URVNULi4uIikKICAgICAgICBkYW5uID0gREFOTk1vZGVsKGV4cGVjdGVkLCBjb25maWcpLnRvKGRldmljZSkKICAgICAgICBkYW5uLmxvYWRfc3RhdGVfZGljdCh0b3JjaC5sb2FkKGRhbm5fY2twdCwgbWFwX2xvY2F0aW9uPWRldmljZSkpCiAgICAgICAgZGFubl9mZWF0dXJlcywgZGFubl9wcmVkcywgXyA9IHJ1bl9pbmZlcmVuY2UoZGFubiwgdGVzdF9YX25vcm0sIHRlc3RfZG9tYWluX2lkcywgdGVzdF9sYWJlbHMsIGRldmljZSkKICAgICAgICByZXN1bHRzWyJkYW5uIl0gPSBjb21wdXRlX21ldHJpY3ModGVzdF9sYWJlbHMsIGRhbm5fcHJlZHMpCiAgICAgICAgZmVhdHVyZV9zb3VyY2UgPSAoImRhbm4iLCBkYW5uKQogICAgICAgIF9wcmludF9zdW1tYXJ5KCJkYW5uIiwgcmVzdWx0c1siZGFubiJdKQogICAgZWxzZToKICAgICAgICBwcmludChmIntkYW5uX2NrcHR9IG5vdCBmb3VuZCAtLSBza2lwcGluZyBEQU5OIGV2YWx1YXRpb24gKHJ1biB0cmFpbl9kYW5uLnB5IHRvIGFkZCB0aGUgY29tcGFyaXNvbikuIikKCiAgICAjIC0tLSBzYXZlIG1ldHJpY3MgKyBwZXItd2luZG93IHByZWRpY3Rpb25zIC0tLQogICAgd2l0aCBvcGVuKHJlcG9ydHNfZGlyIC8gImV2YWx1YXRpb25fbWV0cmljcy5qc29uIiwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChyZXN1bHRzLCBmLCBpbmRlbnQ9MikKICAgIHRlc3RfbWV0YSA9IHRlc3RfbWV0YS5jb3B5KCkKICAgIHRlc3RfbWV0YVsiYmFzZWxpbmVfcHJlZCJdID0gYmFzZWxpbmVfcHJlZHMKICAgIGlmICJkYW5uIiBpbiByZXN1bHRzOgogICAgICAgIHRlc3RfbWV0YVsiZGFubl9wcmVkIl0gPSBkYW5uX3ByZWRzCiAgICB0ZXN0X21ldGEudG9fY3N2KHJlcG9ydHNfZGlyIC8gInRhcmdldF90ZXN0X3ByZWRpY3Rpb25zLmNzdiIsIGluZGV4PUZhbHNlKQoKICAgICMgLS0tIGZpZ3VyZXMgLS0tCiAgICB2aXoucGxvdF9jb25mdXNpb25fbWF0cml4X211bHRpY2xhc3MoCiAgICAgICAgbnAuYXJyYXkocmVzdWx0c1siYmFzZWxpbmUiXVsiY29uZnVzaW9uX21hdHJpeCJdKSwKICAgICAgICBmaWd1cmVzX2RpciAvICJjb25mdXNpb25fbWF0cml4X2Jhc2VsaW5lLnBuZyIsCiAgICAgICAgIkJhc2VsaW5lIChubyBhZGFwdGF0aW9uKSAtLSBUQVJHRVRfVEVTVCIsCiAgICApCiAgICBpZiAiZGFubiIgaW4gcmVzdWx0czoKICAgICAgICB2aXoucGxvdF9jb25mdXNpb25fbWF0cml4X211bHRpY2xhc3MoCiAgICAgICAgICAgIG5wLmFycmF5KHJlc3VsdHNbImRhbm4iXVsiY29uZnVzaW9uX21hdHJpeCJdKSwKICAgICAgICAgICAgZmlndXJlc19kaXIgLyAiY29uZnVzaW9uX21hdHJpeF9kYW5uLnBuZyIsCiAgICAgICAgICAgICJEQU5OIChhZGFwdGVkKSAtLSBUQVJHRVRfVEVTVCIsCiAgICAgICAgKQogICAgdml6LnBsb3RfYmFzZWxpbmVfdnNfZGFubl9iYXJzKHJlc3VsdHNbImJhc2VsaW5lIl0sIHJlc3VsdHMuZ2V0KCJkYW5uIiksIGZpZ3VyZXNfZGlyIC8gImJhc2VsaW5lX3ZzX2Rhbm4ucG5nIikKCiAgICBwcmludCgiQ29tcHV0aW5nIGZlYXR1cmUtc3BhY2UgZW1iZWRkaW5ncyAoU09VUkNFX1ZBTCArIFRBUkdFVF9URVNUKS4uLiIpCiAgICBtb2RlbF9uYW1lLCBtb2RlbCA9IGZlYXR1cmVfc291cmNlCiAgICB2YWxfZG9tYWluX2lkcyA9IHZhbF9tZXRhLmRvbWFpbl9pZC50b19udW1weSgpCiAgICB2YWxfbGFiZWxzID0gdmFsX21ldGEudW5pdmVyc2FsX2xhYmVsLnRvX251bXB5KCkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhjb25maWcuc2VlZCkKICAgIHZhbF9zYW1wbGUgPSBybmcuY2hvaWNlKGxlbih2YWxfWF9ub3JtKSwgc2l6ZT1taW4oMjAwMCwgbGVuKHZhbF9YX25vcm0pKSwgcmVwbGFjZT1GYWxzZSkKCiAgICAjIC0tLSBEaWQgYWRhcHRhdGlvbiBhY3R1YWxseSBhbGlnbiB0aGUgZG9tYWlucz8gLS0tCiAgICAjIFRyYWluIGEgRlJFU0ggcHJvYmUgb24gZWFjaCBtb2RlbCdzIGZyb3plbiBmZWF0dXJlcyAoc2VlCiAgICAjIGFuYWx5c2lzLmRvbWFpbl9zZXBhcmFiaWxpdHlfcHJvYmUpLiBUaGlzIGlzIHRoZSBjaGVjayB2MSBsYWNrZWQ6IGl0cwogICAgIyBkb21haW5fbG9zcyBzYXQgYXQgY2hhbmNlIHRoZSB3aG9sZSBydW4sIHdoaWNoIGxvb2tlZCBsaWtlIGFsaWdubWVudAogICAgIyBidXQgLS0gcGVyIHRoZSB0LVNORSBwbG90IC0tIHdhc24ndC4gVGhlIHByb2JlIGNhbid0IGJlIGZvb2xlZCB0aGF0CiAgICAjIHdheSwgYW5kIGNvbXBhcmluZyBiYXNlbGluZSB2cy4gREFOTiBzYXlzIHdoZXRoZXIgYWRhcHRhdGlvbiBtb3ZlZAogICAgIyB0aGUgbmVlZGxlIEFUIEFMTCwgaW5kZXBlbmRlbnQgb2YgYW55IHRhcmdldCBsYWJlbC4KICAgIGZvciBwcm9iZV9uYW1lLCBwcm9iZV9tb2RlbCwgcHJvYmVfdGVzdF9mZWF0dXJlcyBpbiBbCiAgICAgICAgKCJiYXNlbGluZSIsIGJhc2VsaW5lLCBiYXNlbGluZV9mZWF0dXJlcyksCiAgICAgICAgKCJkYW5uIiwgZGFubiBpZiAiZGFubiIgaW4gcmVzdWx0cyBlbHNlIE5vbmUsIGRhbm5fZmVhdHVyZXMgaWYgImRhbm4iIGluIHJlc3VsdHMgZWxzZSBOb25lKSwKICAgIF06CiAgICAgICAgaWYgcHJvYmVfbW9kZWwgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcm9iZV92YWxfZmVhdHVyZXMsIF8sIF8gPSBydW5faW5mZXJlbmNlKAogICAgICAgICAgICBwcm9iZV9tb2RlbCwgdmFsX1hfbm9ybVt2YWxfc2FtcGxlXSwgdmFsX2RvbWFpbl9pZHNbdmFsX3NhbXBsZV0sIHZhbF9sYWJlbHNbdmFsX3NhbXBsZV0sIGRldmljZQogICAgICAgICkKICAgICAgICBwcm9iZV9yZXN1bHQgPSBkb21haW5fc2VwYXJhYmlsaXR5X3Byb2JlKHByb2JlX3ZhbF9mZWF0dXJlcywgcHJvYmVfdGVzdF9mZWF0dXJlcywgc2VlZD1jb25maWcuc2VlZCkKICAgICAgICByZXN1bHRzW3Byb2JlX25hbWVdWyJkb21haW5fc2VwYXJhYmlsaXR5Il0gPSBwcm9iZV9yZXN1bHQKICAgICAgICBwcmludCgKICAgICAgICAgICAgZiIgIHtwcm9iZV9uYW1lfTogYSBmcmVzaCBwcm9iZSBzZXBhcmF0ZXMgc291cmNlIGZyb20gdGFyZ2V0IHdpdGggIgogICAgICAgICAgICBmIntwcm9iZV9yZXN1bHRbJ3NlcGFyYWJpbGl0eV9hY2N1cmFjeSddOi4zZn0gYWNjdXJhY3kgIgogICAgICAgICAgICBmIigwLjUgPSBhbGlnbmVkL2luZGlzdGluZ3Vpc2hhYmxlLCAxLjAgPSBub3QgYWxpZ25lZCBhdCBhbGw7ICIKICAgICAgICAgICAgZiJwcm94eSBBLWRpc3RhbmNlIHtwcm9iZV9yZXN1bHRbJ3Byb3h5X2FfZGlzdGFuY2UnXTouMmZ9KSIKICAgICAgICApCiAgICAgICAgaWYgcHJvYmVfbmFtZSA9PSBtb2RlbF9uYW1lOgogICAgICAgICAgICB2YWxfZmVhdHVyZXMgPSBwcm9iZV92YWxfZmVhdHVyZXMKCiAgICAjIFJlLXNhdmUgbWV0cmljcyBub3cgdGhhdCB0aGUgcHJvYmUgcmVzdWx0cyBhcmUgYXR0YWNoZWQuCiAgICB3aXRoIG9wZW4ocmVwb3J0c19kaXIgLyAiZXZhbHVhdGlvbl9tZXRyaWNzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKHJlc3VsdHMsIGYsIGluZGVudD0yKQoKICAgIHByaW50KCJGaXR0aW5nIHQtU05FLi4uIikKICAgIHRlc3RfZmVhdHVyZXMgPSBkYW5uX2ZlYXR1cmVzIGlmIG1vZGVsX25hbWUgPT0gImRhbm4iIGVsc2UgYmFzZWxpbmVfZmVhdHVyZXMKCiAgICBjb21iaW5lZF9mZWF0dXJlcyA9IG5wLmNvbmNhdGVuYXRlKFt2YWxfZmVhdHVyZXMsIHRlc3RfZmVhdHVyZXNdLCBheGlzPTApCiAgICBjb21iaW5lZF9kb21haW5faWRzID0gbnAuY29uY2F0ZW5hdGUoW3ZhbF9kb21haW5faWRzW3ZhbF9zYW1wbGVdLCB0ZXN0X2RvbWFpbl9pZHNdKQogICAgY29tYmluZWRfbGFiZWxzID0gbnAuY29uY2F0ZW5hdGUoW3ZhbF9sYWJlbHNbdmFsX3NhbXBsZV0sIHRlc3RfbGFiZWxzXSkKCiAgICB0c25lID0gVFNORShuX2NvbXBvbmVudHM9MiwgcGVycGxleGl0eT1jb25maWcuZXZhbHVhdGlvbi50c25lX3BlcnBsZXhpdHksIHJhbmRvbV9zdGF0ZT1jb25maWcuc2VlZCwgaW5pdD0icGNhIikKICAgIGNvb3JkcyA9IHRzbmUuZml0X3RyYW5zZm9ybShjb21iaW5lZF9mZWF0dXJlcykKICAgIHZpei5wbG90X2ZlYXR1cmVfc3BhY2UoCiAgICAgICAgY29vcmRzLCBjb21iaW5lZF9kb21haW5faWRzLCBjb21iaW5lZF9sYWJlbHMsCiAgICAgICAgZmlndXJlc19kaXIgLyBmImZlYXR1cmVfc3BhY2VfdHNuZV97bW9kZWxfbmFtZX0ucG5nIiwKICAgICAgICBtZXRob2RfbmFtZT1mInQtU05FICh7bW9kZWxfbmFtZX0gZmVhdHVyZXMpIiwKICAgICkKCiAgICBwcmludChmIlxuU2F2ZWQgbWV0cmljcy9wbG90cyB0byB7cmVwb3J0c19kaXJ9IGFuZCB7ZmlndXJlc19kaXJ9IikKICAgIHJldHVybiByZXN1bHRzCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD0iY29uZmlncy9kb21haW5fYWRhcHRhdGlvbi55YW1sIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHUtcm9vdCIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iUm9vdCBkaXIgb2YgdGhlIFBVIGRhdGFzZXQgKGRlZmF1bHQ6IGF1dG8tZGV0ZWN0IC9rYWdnbGUvaW5wdXQsIGVsc2UgY29uZmlnJ3MgcHVfcm9vdF9sb2NhbCkuIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgY29uZmlnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBwdV9yb290ID0gYXJncy5wdV9yb290IG9yIGZpbmRfcHVfcm9vdChjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMucHVfcm9vdF9sb2NhbCkpCiAgICBldmFsdWF0ZShjb25maWcsIHB1X3Jvb3QpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
    ),
    'src/domain_adaptation/analysis.py': (
        'IiIiUHVyZS1udW1weS9za2xlYXJuIGFuYWx5c2lzIGhlbHBlcnMgKG5vIHRvcmNoKSwgc28gdGhleSBjYW4gYmUgdW5pdAp0ZXN0ZWQgbG9jYWxseSAtLSBzYW1lIHNwbGl0IGFzIHZpYnJhdGlvbl9hZC5hbmFseXNpcyAvIGh5ZHJhdWxpY19mdXNpb25fYWQuYW5hbHlzaXMsCndoZXJlIGFuYWx5c2lzLnB5IGhvbGRzIHRoZSBjb21wdXRhdGlvbiBhbmQgZXZhbHVhdGUucHkvYW5hbHl6ZS5weSBqdXN0CmNhbGwgaXQuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTG9naXN0aWNSZWdyZXNzaW9uCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXQKCgpkZWYgZG9tYWluX3NlcGFyYWJpbGl0eV9wcm9iZSgKICAgIHNvdXJjZV9mZWF0dXJlczogbnAubmRhcnJheSwgdGFyZ2V0X2ZlYXR1cmVzOiBucC5uZGFycmF5LCBzZWVkOiBpbnQgPSA0MiwgdGVzdF9zaXplOiBmbG9hdCA9IDAuMwopIC0+IGRpY3Q6CiAgICAiIiJIb3cgZWFzaWx5IGNhbiBhIEZSRVNIIGNsYXNzaWZpZXIgdGVsbCBzb3VyY2UgZmVhdHVyZXMgZnJvbSB0YXJnZXQKICAgIGZlYXR1cmVzPyBUaGlzIGlzIHRoZSBzdGFuZGFyZCBwcm94eS1BLWRpc3RhbmNlIG1lYXN1cmVtZW50LCBhbmQgaXQncwogICAgdGhlIGRpYWdub3N0aWMgdGhhdCBkaXNhbWJpZ3VhdGVzIHRoZSBvbmUgdGhpbmcgdGhlIHRyYWluaW5nIGxvZyBjYW5ub3QuCgogICAgRHVyaW5nIERBTk4gdHJhaW5pbmcgdGhlIGRvbWFpbiBkaXNjcmltaW5hdG9yJ3MgbG9zcyBzaXR0aW5nIGF0IGNoYW5jZQogICAgKGxuKDMpID0gMS4wOTkgZm9yIDMgZG9tYWlucykgaGFzIHR3byBjb21wbGV0ZWx5IGRpZmZlcmVudCBwb3NzaWJsZQogICAgbWVhbmluZ3M6CiAgICAgIDEuIFRoZSBmZWF0dXJlcyByZWFsbHkgYXJlIGRvbWFpbi1pbnZhcmlhbnQgLS0gYWRhcHRhdGlvbiB3b3JrZWQuCiAgICAgIDIuIFRoZSBkaXNjcmltaW5hdG9yIGlzIGp1c3QgbG9zaW5nIHRoZSBtaW5pbWF4IHJhY2UgYW5kIGhhcyBsZWFybmVkCiAgICAgICAgIG5vdGhpbmcgLS0gYWRhcHRhdGlvbiBkaWQgbm90aGluZywgYW5kIHRoZSBmbGF0IGxvc3MgaXMgYW4gaWxsdXNpb24uCiAgICBBIGZyZXNoIHByb2JlIHRyYWluZWQgb24gRlJPWkVOIGZlYXR1cmVzLCB3aXRoIG5vIGdyYWRpZW50IHJldmVyc2FsCiAgICBmaWdodGluZyBpdCwgY2FuJ3QgYmUgZm9vbGVkIHRoYXQgd2F5OiBpZiBpdCBzZXBhcmF0ZXMgdGhlIGRvbWFpbnMKICAgIGVhc2lseSwgdGhlIGZlYXR1cmVzIHdlcmUgbmV2ZXIgYWxpZ25lZCwgd2hhdGV2ZXIgdGhlIHRyYWluaW5nIGxvZyBzYWlkLgoKICAgIFJldHVybnMgYWNjdXJhY3kgb24gYSBoZWxkLW91dCBzcGxpdCBwbHVzIHRoZSBjaGFuY2UgYmFzZWxpbmUuIENsYXNzZXMKICAgIGFyZSBiYWxhbmNlZCBieSBzdWJzYW1wbGluZyB0aGUgbGFyZ2VyIHNpZGUgZmlyc3QsIHNvIGNoYW5jZSBpcyBhbHdheXMKICAgIDAuNSBhbmQgdGhlIG51bWJlciBpcyBkaXJlY3RseSByZWFkYWJsZToKICAgICAgICB+MC41IC0+IGRvbWFpbnMgZ2VudWluZWx5IGluZGlzdGluZ3Vpc2hhYmxlIChhbGlnbmVkKQogICAgICAgIH4xLjAgLT4gdHJpdmlhbGx5IHNlcGFyYWJsZSAobm90IGFsaWduZWQgYXQgYWxsKQogICAgIiIiCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZCkKICAgIG4gPSBtaW4obGVuKHNvdXJjZV9mZWF0dXJlcyksIGxlbih0YXJnZXRfZmVhdHVyZXMpKQogICAgaWYgbiA8IDEwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJOZWVkID49MTAgc2FtcGxlcyBwZXIgZG9tYWluIGZvciBhIG1lYW5pbmdmdWwgcHJvYmUsIGdvdCB7bn0iKQoKICAgIHNvdXJjZV9zYW1wbGUgPSBzb3VyY2VfZmVhdHVyZXNbcm5nLmNob2ljZShsZW4oc291cmNlX2ZlYXR1cmVzKSwgc2l6ZT1uLCByZXBsYWNlPUZhbHNlKV0KICAgIHRhcmdldF9zYW1wbGUgPSB0YXJnZXRfZmVhdHVyZXNbcm5nLmNob2ljZShsZW4odGFyZ2V0X2ZlYXR1cmVzKSwgc2l6ZT1uLCByZXBsYWNlPUZhbHNlKV0KCiAgICBYID0gbnAuY29uY2F0ZW5hdGUoW3NvdXJjZV9zYW1wbGUsIHRhcmdldF9zYW1wbGVdLCBheGlzPTApCiAgICB5ID0gbnAuY29uY2F0ZW5hdGUoW25wLnplcm9zKG4sIGR0eXBlPWludCksIG5wLm9uZXMobiwgZHR5cGU9aW50KV0pCgogICAgWF90cmFpbiwgWF90ZXN0LCB5X3RyYWluLCB5X3Rlc3QgPSB0cmFpbl90ZXN0X3NwbGl0KAogICAgICAgIFgsIHksIHRlc3Rfc2l6ZT10ZXN0X3NpemUsIHJhbmRvbV9zdGF0ZT1zZWVkLCBzdHJhdGlmeT15CiAgICApCiAgICBwcm9iZSA9IExvZ2lzdGljUmVncmVzc2lvbihtYXhfaXRlcj0yMDAwKQogICAgcHJvYmUuZml0KFhfdHJhaW4sIHlfdHJhaW4pCiAgICBhY2N1cmFjeSA9IGZsb2F0KHByb2JlLnNjb3JlKFhfdGVzdCwgeV90ZXN0KSkKCiAgICByZXR1cm4gewogICAgICAgICJzZXBhcmFiaWxpdHlfYWNjdXJhY3kiOiBhY2N1cmFjeSwKICAgICAgICAiY2hhbmNlIjogMC41LAogICAgICAgICMgUHJveHkgQS1kaXN0YW5jZSwgdGhlIHVzdWFsIHN1bW1hcnkgb2YgdGhpcyBtZWFzdXJlbWVudDoKICAgICAgICAjIDIgKiAoMSAtIDIqZXJyb3IpIC0tIDAgbWVhbnMgaW5kaXN0aW5ndWlzaGFibGUsIDIgbWVhbnMgdHJpdmlhbGx5IHNlcGFyYWJsZS4KICAgICAgICAicHJveHlfYV9kaXN0YW5jZSI6IGZsb2F0KDIuMCAqICgxLjAgLSAyLjAgKiAoMS4wIC0gYWNjdXJhY3kpKSksCiAgICAgICAgIm5fcGVyX2RvbWFpbiI6IGludChuKSwKICAgIH0KCgpkZWYgYmluYXJ5X2RldGVjdGlvbl9tZXRyaWNzKHlfdHJ1ZTogbnAubmRhcnJheSwgeV9wcmVkOiBucC5uZGFycmF5KSAtPiBkaWN0OgogICAgIiIiQ29sbGFwc2UgdGhlIDMtY2xhc3MgcHJlZGljdGlvbiB0byBoZWFsdGh5ICgwKSB2cyBmYXVsdHkgKDEgb3IgMikgYW5kCiAgICBzY29yZSBpdCBhcyBhIGRldGVjdG9yLgoKICAgIFdoeSB0aGlzIGJlbG9uZ3MgYWxvbmdzaWRlIHRoZSAzLWNsYXNzIG51bWJlcnMgcmF0aGVyIHRoYW4gaW4gYQogICAgdGhyb3dhd2F5IHNjcmlwdDogdGhlIGRlcGxveW1lbnQgcXVlc3Rpb24gdGhpcyB3aG9sZSBwcm9qZWN0IGV4aXN0cyB0bwogICAgYW5zd2VyIGlzICJ0aGUgbWFjaGluZSBvbmx5IGV2ZXIgZ2F2ZSB1cyBoZWFsdGh5IGRhdGEgLS0gd2lsbCB3ZSBub3RpY2UKICAgIHdoZW4gaXQgYnJlYWtzPyIsIHdoaWNoIGlzIGJpbmFyeS4gRmF1bHQgKnR5cGluZyogaXMgYSBib251cy4gQWNyb3NzCiAgICBrZXJuZWxzIHYxLXY1IHRoZSAzLWNsYXNzIHRhc2sgbmV2ZXIgdHJhbnNmZXJyZWQgd2hpbGUgZGV0ZWN0aW9uIHBhcnRseQogICAgZGlkLCBzbyByZXBvcnRpbmcgb25seSB0aGUgMy1jbGFzcyBmaWd1cmUgdW5kZXJzdGF0ZXMgdGhlIHJlc3VsdCBhbmQKICAgIHJlcG9ydGluZyBvbmx5IGFjY3VyYWN5IG92ZXJzdGF0ZXMgaXQuCgogICAgYGZhbHNlX2FsYXJtX3JhdGVgIGlzIHRoZSBudW1iZXIgdG8gbGVhZCB3aXRoLCBub3QgYWNjdXJhY3k6IFRBUkdFVF9URVNUCiAgICBpcyB+OTAlIGZhdWx0eSwgc28gImFsd2F5cyBzYXkgZmF1bHR5IiBzY29yZXMgMC45MCBhY2N1cmFjeSB3aGlsZSBiZWluZwogICAgdXNlbGVzcy4gRmFsc2UtYWxhcm0gcmF0ZSBpcyBtZWFzdXJlZCBvbiBoZWFsdGh5IHdpbmRvd3Mgb25seSwgc28gdGhlCiAgICB0ZXN0IHNldCdzIHNrZXcgY2Fubm90IGZsYXR0ZXIgaXQsIGFuZCBpbiBjb25kaXRpb24gbW9uaXRvcmluZyBpdCBpcyB0aGUKICAgIGZpZ3VyZSB0aGF0IGRlY2lkZXMgd2hldGhlciBhbnlvbmUga2VlcHMgbGlzdGVuaW5nIHRvIHRoZSBhbGFybS4KICAgICIiIgogICAgeV90cnVlX2JpbiA9IChucC5hc2FycmF5KHlfdHJ1ZSkgIT0gMCkuYXN0eXBlKGludCkKICAgIHlfcHJlZF9iaW4gPSAobnAuYXNhcnJheSh5X3ByZWQpICE9IDApLmFzdHlwZShpbnQpCgogICAgdHAgPSBpbnQoKCh5X3ByZWRfYmluID09IDEpICYgKHlfdHJ1ZV9iaW4gPT0gMSkpLnN1bSgpKQogICAgZnAgPSBpbnQoKCh5X3ByZWRfYmluID09IDEpICYgKHlfdHJ1ZV9iaW4gPT0gMCkpLnN1bSgpKQogICAgdG4gPSBpbnQoKCh5X3ByZWRfYmluID09IDApICYgKHlfdHJ1ZV9iaW4gPT0gMCkpLnN1bSgpKQogICAgZm4gPSBpbnQoKCh5X3ByZWRfYmluID09IDApICYgKHlfdHJ1ZV9iaW4gPT0gMSkpLnN1bSgpKQoKICAgIHByZWNpc2lvbiA9IHRwIC8gKHRwICsgZnApIGlmICh0cCArIGZwKSBlbHNlIDAuMAogICAgcmVjYWxsID0gdHAgLyAodHAgKyBmbikgaWYgKHRwICsgZm4pIGVsc2UgMC4wCiAgICBmMSA9IDIgKiBwcmVjaXNpb24gKiByZWNhbGwgLyAocHJlY2lzaW9uICsgcmVjYWxsKSBpZiAocHJlY2lzaW9uICsgcmVjYWxsKSBlbHNlIDAuMAogICAgbl9oZWFsdGh5LCBuX2ZhdWx0eSA9IHRuICsgZnAsIHRwICsgZm4KCiAgICByZXR1cm4gewogICAgICAgICJwcmVjaXNpb24iOiBmbG9hdChwcmVjaXNpb24pLAogICAgICAgICJyZWNhbGwiOiBmbG9hdChyZWNhbGwpLAogICAgICAgICJmMSI6IGZsb2F0KGYxKSwKICAgICAgICAiYWNjdXJhY3kiOiBmbG9hdCgodHAgKyB0bikgLyBsZW4oeV90cnVlX2JpbikpIGlmIGxlbih5X3RydWVfYmluKSBlbHNlIDAuMCwKICAgICAgICAjIE1lYXN1cmVkIG9uIGhlYWx0aHkgd2luZG93cyBvbmx5IC0tIGltbXVuZSB0byB0aGUgdGVzdCBzZXQncyBza2V3LgogICAgICAgICJmYWxzZV9hbGFybV9yYXRlIjogZmxvYXQoZnAgLyBuX2hlYWx0aHkpIGlmIG5faGVhbHRoeSBlbHNlIDAuMCwKICAgICAgICAiY29uZnVzaW9uIjogeyJ0biI6IHRuLCAiZnAiOiBmcCwgImZuIjogZm4sICJ0cCI6IHRwfSwKICAgICAgICAibl9oZWFsdGh5Ijogbl9oZWFsdGh5LAogICAgICAgICJuX2ZhdWx0eSI6IG5fZmF1bHR5LAogICAgICAgICMgV2hhdCBhIG1vZGVsIHRoYXQgYWx3YXlzIHNob3V0cyAiZmF1bHR5IiB3b3VsZCBzY29yZSwgc28gdGhlCiAgICAgICAgIyBoZWFkbGluZSBudW1iZXIgY2FuIG5ldmVyIGJlIHJlYWQgd2l0aG91dCBpdHMgbnVsbCBjb21wYXJpc29uLgogICAgICAgICJhbHdheXNfZmF1bHR5X2FjY3VyYWN5IjogZmxvYXQobl9mYXVsdHkgLyBsZW4oeV90cnVlX2JpbikpIGlmIGxlbih5X3RydWVfYmluKSBlbHNlIDAuMCwKICAgIH0K'
    ),
    'src/domain_adaptation/infer.py': (
        'IiIiUGhhc2UgNTogcnVuIGEgdHJhaW5lZCBkZXRlY3RvciBvbiBhIE5FVyBtYWNoaW5lJ3MgdmlicmF0aW9uIGRhdGEuCgpUaGlzIGlzIHRoZSBkZXBsb3ltZW50IHBhdGggdGhlIG9yaWdpbmFsIHBsYW4gYXNrcyBmb3IsIGFuZCBpdCBpcwpkZWxpYmVyYXRlbHkgYnVpbHQgYXJvdW5kIHdoYXQga2VybmVscyB2MS12NSBhY3R1YWxseSBlc3RhYmxpc2hlZCByYXRoZXIKdGhhbiB3aGF0IHRoZXkgc2V0IG91dCB0byBwcm92ZToKCiogSXQgbG9hZHMgdGhlICoqYmFzZWxpbmUqKiBtb2RlbCAoYHRyYWluX2Jhc2VsaW5lLnB5YCksIG5vdCB0aGUgYWRhcHRlZAogIG9uZS4gQWNyb3NzIGZpdmUgcnVucyB0aGUgREFOTiBtb2RlbCB3YXMgbmV2ZXIgYmV0dGVyIGFuZCB3YXMgdXN1YWxseQogIGZhciB3b3JzZSAtLSB2NSdzIGFkYXB0ZWQgbW9kZWwgY2FsbGVkIDkxJSBvZiBldmVyeXRoaW5nIGhlYWx0aHkuIFRoZXJlCiAgaXMgbm8gYWRhcHRhdGlvbiBzdGVwIHRvIHBlcmZvcm0gaGVyZSwgd2hpY2ggYWxzbyBtYWtlcyBkZXBsb3ltZW50CiAgc2ltcGxlcjogbm8gZGlzY3JpbWluYXRvciwgbm8gR1JMLCBubyB0YXJnZXQtZG9tYWluIHRyYWluaW5nLgoqIEl0IHJlcG9ydHMgKipoZWFsdGh5IHZzLiBmYXVsdHkqKiwgbm90IHRoZSBmYXVsdCB0eXBlLiBGYXVsdCB0eXBpbmcKICBuZXZlciB0cmFuc2ZlcnJlZCB0byBhbiB1bnNlZW4gbWFjaGluZS9zcGVlZCBpbiBhbnkgdmVyc2lvbjsgZGV0ZWN0aW9uCiAgZGlkLiBTZWUgRE9NQUlOX0FEQVBUQVRJT05fUExBTi5tZCdzIHZlcmRpY3Qgc2VjdGlvbi4KKiBJdCBoYXMgYSAqKmNhbGlicmF0aW9uIG1vZGUqKiwgYmVjYXVzZSB0aGUgZG9taW5hbnQgZXJyb3IgbW9kZSBpcwogIHBlci11bml0LCBub3QgcmFuZG9tOiBvbiB0aGUgdGFyZ2V0IGNvbmRpdGlvbiwgZXZlcnkgZmFsc2UgYWxhcm0gY2FtZQogIGZyb20gT05FIG9mIHNpeCBoZWFsdGh5IGJlYXJpbmcgdW5pdHMgKEswMDI6IDgvOCByZWNvcmRpbmdzLCAxMDAlIG9mCiAgaXRzIHdpbmRvd3MpLCB3aGlsZSB0aGUgb3RoZXIgZml2ZSB3ZXJlIGVzc2VudGlhbGx5IGNsZWFuLiBBIG1hY2hpbmUKICB3aG9zZSBoZWFsdGh5IGJhc2VsaW5lIHJlYWRzIGFzIGZhdWx0eSB3aWxsIGRvIHNvIGNvbnNpc3RlbnRseSAtLSB3aGljaAogIG1lYW5zIHlvdSBjYW4gZGV0ZWN0IHRoYXQgd2l0aCBoZWFsdGh5IGRhdGEgYWxvbmUsIGJlZm9yZSBldmVyIHNlZWluZyBhCiAgZmF1bHQsIGFuZCBjb3JyZWN0IGZvciBpdC4KClR5cGljYWwgdXNlIG9uIGEgcmVhbCBtYWNoaW5lOgoKICAgICMgMS4gQ2FsaWJyYXRlIG9uIGEgcmVjb3JkaW5nIHlvdSBLTk9XIGlzIGhlYWx0aHkuCiAgICBweXRob24gLW0gZG9tYWluX2FkYXB0YXRpb24uaW5mZXIgLS1pbnB1dCBiYXNlbGluZV9oZWFsdGh5Lm5weSAtLWZzIDI1NjAwIC0tY2FsaWJyYXRlCgogICAgIyAyLiBNb25pdG9yLiBDb21wYXJlIGVhY2ggbmV3IHJlY29yZGluZydzIGZsYWdnZWQgZnJhY3Rpb24gYWdhaW5zdAogICAgIyAgICB0aGUgY2FsaWJyYXRpb24gZmlndXJlIHRoZSBwcmV2aW91cyBzdGVwIHByaW50ZWQuCiAgICBweXRob24gLW0gZG9tYWluX2FkYXB0YXRpb24uaW5mZXIgLS1pbnB1dCB0b2RheS5ucHkgLS1mcyAyNTYwMAoKQWNjZXB0cyBgLm5weWAsIGAubnB6YCAodmlhIC0tc2lnbmFsLWtleSwgZWxzZSB0aGUgZmlyc3QgYXJyYXkpIGFuZCBgLmNzdmAKKHNpbmdsZSBjb2x1bW4pLiBgLS1mc2AgaXMgUkVRVUlSRUQgYW5kIG11c3QgYmUgeW91ciB0cnVlIHNhbXBsaW5nIHJhdGU6CnRoZSBzaWduYWwgaXMgcmVzYW1wbGVkIHRvIHRoZSBtb2RlbCdzIGNvbW1vbiByYXRlLCBhbmQgZ2V0dGluZyBpdCB3cm9uZwpzaGlmdHMgZXZlcnkgZmF1bHQgZnJlcXVlbmN5IGFuZCBpbnZhbGlkYXRlcyB0aGUgcmVzdWx0LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCgpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLndpbmRvd3MgaW1wb3J0IG1ha2Vfd2luZG93cwpmcm9tIHZpYnJhdGlvbl9hZC5mZWF0dXJlcy5zcGVjdHJvZ3JhbSBpbXBvcnQgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyLCBleHBlY3RlZF9zaGFwZSwgd2luZG93c190b19sb2dfc3BlY3Ryb2dyYW1zCgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmNvbmZpZyBpbXBvcnQgQ29uZmlnLCBsb2FkX2NvbmZpZwpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLmRhdGEudHJhbnNmb3JtcyBpbXBvcnQgcmVzYW1wbGVfdG9fcmF0ZQpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLm1vZGVscy5kYW5uIGltcG9ydCBGZWF0dXJlQ2xhc3NpZmllcgpmcm9tIGRvbWFpbl9hZGFwdGF0aW9uLm1vZGVscy5mZWF0dXJlX2V4dHJhY3RvciBpbXBvcnQgc2VsZWN0X2RldmljZQoKIyBGcmFjdGlvbiBvZiBhIHJlY29yZGluZydzIHdpbmRvd3MgdGhhdCBtdXN0IGJlIGZsYWdnZWQgYmVmb3JlIHRoZSB3aG9sZQojIHJlY29yZGluZyBpcyBjYWxsZWQgZmF1bHR5LiAwLjEwIGlzIHdoZXJlIHBlci1yZWNvcmRpbmcgZGV0ZWN0aW9uIHBlYWtlZCBvbgojIHRoZSB0YXJnZXQgY29uZGl0aW9uICg5MS44JSwgdnMgNzYuMSUgcGVyLXdpbmRvdykgd2l0aCBubyBpbmNyZWFzZSBpbiBmYWxzZQojIGFsYXJtcyAtLSBhZ2dyZWdhdGluZyBoZWxwcyBiZWNhdXNlIGEgcmVhbCBmYXVsdCBtYXJrcyBFVkVSWSB3aW5kb3cgb2YgYQojIHJlY29yZGluZywgd2hpbGUgdGhlIG1vZGVsJ3MgbWlzdGFrZXMgYXJlIGNvcnJlbGF0ZWQgd2l0aGluIGEgdW5pdCBhbnl3YXkuCkRFRkFVTFRfUkVDT1JESU5HX1RIUkVTSE9MRCA9IDAuMTAKCgpkZWYgbG9hZF9zaWduYWxfZmlsZShwYXRoOiBQYXRoLCBzaWduYWxfa2V5OiBzdHIgfCBOb25lID0gTm9uZSkgLT4gbnAubmRhcnJheToKICAgICIiIlJlYWQgYSAxRCB2aWJyYXRpb24gc2lnbmFsIGZyb20gLm5weSAvIC5ucHogLyAuY3N2LiIiIgogICAgc3VmZml4ID0gcGF0aC5zdWZmaXgubG93ZXIoKQogICAgaWYgc3VmZml4ID09ICIubnB5IjoKICAgICAgICByZXR1cm4gbnAubG9hZChwYXRoKS5hc3R5cGUobnAuZmxvYXQ2NCkucmF2ZWwoKQogICAgaWYgc3VmZml4ID09ICIubnB6IjoKICAgICAgICB3aXRoIG5wLmxvYWQocGF0aCkgYXMgZGF0YToKICAgICAgICAgICAga2V5cyA9IGxpc3QoZGF0YS5rZXlzKCkpCiAgICAgICAgICAgIGlmIHNpZ25hbF9rZXkgaXMgTm9uZToKICAgICAgICAgICAgICAgIGlmIGxlbihrZXlzKSAhPSAxOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7cGF0aH0gaG9sZHMge2tleXN9OyBwYXNzIC0tc2lnbmFsLWtleSB0byBjaG9vc2Ugb25lIikKICAgICAgICAgICAgICAgIHNpZ25hbF9rZXkgPSBrZXlzWzBdCiAgICAgICAgICAgIGlmIHNpZ25hbF9rZXkgbm90IGluIGRhdGE6CiAgICAgICAgICAgICAgICByYWlzZSBLZXlFcnJvcihmIntwYXRofTogbm8gYXJyYXkge3NpZ25hbF9rZXkhcn0gKGhhcyB7a2V5c30pIikKICAgICAgICAgICAgcmV0dXJuIGRhdGFbc2lnbmFsX2tleV0uYXN0eXBlKG5wLmZsb2F0NjQpLnJhdmVsKCkKICAgIGlmIHN1ZmZpeCA9PSAiLmNzdiI6CiAgICAgICAgcmV0dXJuIHBkLnJlYWRfY3N2KHBhdGgsIGhlYWRlcj1Ob25lKS50b19udW1weShkdHlwZT1ucC5mbG9hdDY0KS5yYXZlbCgpCiAgICByYWlzZSBWYWx1ZUVycm9yKGYie3BhdGh9OiB1bnN1cHBvcnRlZCBmb3JtYXQge3N1ZmZpeCFyfSAodXNlIC5ucHksIC5ucHogb3IgLmNzdikiKQoKCmRlZiBzY29yZV9zaWduYWwoc2lnbmFsOiBucC5uZGFycmF5LCBmc19oejogaW50LCBjb25maWc6IENvbmZpZywgbW9kZWwsIG5vcm1hbGl6ZXIsIGRldmljZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiT25lIHJvdyBwZXIgd2luZG93OiBwcmVkaWN0ZWQgY2xhc3MgYW5kIHdoZXRoZXIgaXQgY291bnRzIGFzIGZhdWx0eS4iIiIKICAgIHNpZ25hbCA9IHJlc2FtcGxlX3RvX3JhdGUoc2lnbmFsLCBmc19oeiwgY29uZmlnLmNvbW1vbi5mc19oeikKICAgIHdpbmRvd3MgPSBtYWtlX3dpbmRvd3Moc2lnbmFsLCBjb25maWcuY29tbW9uLnNlZ21lbnRfbGVuZ3RoLCBjb25maWcuY29tbW9uLmV2YWxfc3RyaWRlKQogICAgaWYgbGVuKHdpbmRvd3MpID09IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJTaWduYWwgdG9vIHNob3J0OiB7bGVuKHNpZ25hbCl9IHNhbXBsZXMgYXQge2NvbmZpZy5jb21tb24uZnNfaHp9SHogYWZ0ZXIgcmVzYW1wbGluZywgIgogICAgICAgICAgICBmImJ1dCBvbmUgd2luZG93IG5lZWRzIHtjb25maWcuY29tbW9uLnNlZ21lbnRfbGVuZ3RofSAiCiAgICAgICAgICAgIGYiKH57Y29uZmlnLmNvbW1vbi5zZWdtZW50X2xlbmd0aCAvIGNvbmZpZy5jb21tb24uZnNfaHo6LjJmfXMpLiBSZWNvcmQgYSBsb25nZXIgc2FtcGxlLiIKICAgICAgICApCiAgICBzcGVjdHJvZ3JhbXMgPSBub3JtYWxpemVyLnRyYW5zZm9ybSh3aW5kb3dzX3RvX2xvZ19zcGVjdHJvZ3JhbXMod2luZG93cywgY29uZmlnLmNvbW1vbi5mc19oeiwgY29uZmlnLnN0ZnQpKQoKICAgIFggPSB0b3JjaC5mcm9tX251bXB5KHNwZWN0cm9ncmFtcykudW5zcXVlZXplKDEpLmZsb2F0KCkudG8oZGV2aWNlKQogICAgbW9kZWwuZXZhbCgpCiAgICBwcmVkcyA9IFtdCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCwgbGVuKFgpLCA2NCk6CiAgICAgICAgICAgIF9mZWF0dXJlcywgbG9naXRzID0gbW9kZWwoWFtzdGFydCA6IHN0YXJ0ICsgNjRdKQogICAgICAgICAgICBwcmVkcy5hcHBlbmQobG9naXRzLmFyZ21heChkaW09MSkuY3B1KCkubnVtcHkoKSkKICAgIHByZWRzID0gbnAuY29uY2F0ZW5hdGUocHJlZHMpCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKAogICAgICAgIHsKICAgICAgICAgICAgIndpbmRvdyI6IG5wLmFyYW5nZShsZW4ocHJlZHMpKSwKICAgICAgICAgICAgInByZWRpY3RlZF9jbGFzcyI6IHByZWRzLAogICAgICAgICAgICAjIEZhdWx0IFRZUEUgaXMgZGVsaWJlcmF0ZWx5IG5vdCBzdXJmYWNlZCBhcyBhIHZlcmRpY3Q6IGl0IGRpZCBub3QKICAgICAgICAgICAgIyB0cmFuc2ZlciB0byB1bnNlZW4gbWFjaGluZXMgaW4gYW55IHJ1bi4gS2VwdCBvbmx5IGZvciBpbnNwZWN0aW9uLgogICAgICAgICAgICAiZmxhZ2dlZF9mYXVsdHkiOiAocHJlZHMgIT0gMCkuYXN0eXBlKGludCksCiAgICAgICAgfQogICAgKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1hcmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9ImNvbmZpZ3MvZG9tYWluX2FkYXB0YXRpb24ueWFtbCIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWlucHV0IiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iQSBzaWduYWwgZmlsZSwgb3IgYSBkaXJlY3Rvcnkgb2YgdGhlbS4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mcyIsIHR5cGU9aW50LCByZXF1aXJlZD1UcnVlLCBoZWxwPSJTYW1wbGluZyByYXRlIG9mIFlPVVIgZGF0YSwgaW4gSHouIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2lnbmFsLWtleSIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iQXJyYXkgbmFtZSBpbnNpZGUgYSBtdWx0aS1hcnJheSAubnB6LiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRocmVzaG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9REVGQVVMVF9SRUNPUkRJTkdfVEhSRVNIT0xELAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPWYiRmxhZ2dlZC13aW5kb3cgZnJhY3Rpb24gYWJvdmUgd2hpY2ggYSByZWNvcmRpbmcgaXMgY2FsbGVkIGZhdWx0eSAoZGVmYXVsdCB7REVGQVVMVF9SRUNPUkRJTkdfVEhSRVNIT0xEfSkuIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY2FsaWJyYXRlIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iVHJlYXQgZXZlcnkgaW5wdXQgYXMgS05PV04tSEVBTFRIWSBhbmQgcmVwb3J0IHRoZSBmYWxzZS1hbGFybSByYXRlIG9uIGl0LiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iT3B0aW9uYWwgcGF0aCB0byB3cml0ZSBwZXItd2luZG93IHByZWRpY3Rpb25zIGFzIENTVi4iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBjb25maWcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIG1vZGVsc19kaXIgPSBjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMubW9kZWxzX2RpcikKICAgIGNrcHQgPSBtb2RlbHNfZGlyIC8gImJhc2VsaW5lX2Jlc3QucHQiCiAgICBpZiBub3QgY2twdC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIntja3B0fSBub3QgZm91bmQgLS0gdHJhaW4gaXQgZmlyc3QgKHNlZSBSRUFETUUncyBkb21haW4tYWRhcHRhdGlvbiBzZWN0aW9uKS4iKQoKICAgIGRldmljZSA9IHNlbGVjdF9kZXZpY2UoY29uZmlnLnRyYWluX2Jhc2VsaW5lLmRldmljZSkKICAgIG1vZGVsID0gRmVhdHVyZUNsYXNzaWZpZXIoZXhwZWN0ZWRfc2hhcGUoY29uZmlnLmNvbW1vbiwgY29uZmlnLnN0ZnQpLCBjb25maWcpLnRvKGRldmljZSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdCh0b3JjaC5sb2FkKGNrcHQsIG1hcF9sb2NhdGlvbj1kZXZpY2UpKQogICAgbm9ybWFsaXplciA9IFNwZWN0cm9ncmFtTm9ybWFsaXplci5sb2FkKG1vZGVsc19kaXIgLyAibm9ybWFsaXplci5ucHoiKQoKICAgIHBhdGggPSBQYXRoKGFyZ3MuaW5wdXQpCiAgICBmaWxlcyA9IHNvcnRlZChwIGZvciBwIGluIHBhdGguaXRlcmRpcigpIGlmIHAuc3VmZml4Lmxvd2VyKCkgaW4geyIubnB5IiwgIi5ucHoiLCAiLmNzdiJ9KSBpZiBwYXRoLmlzX2RpcigpIGVsc2UgW3BhdGhdCiAgICBpZiBub3QgZmlsZXM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJObyAubnB5Ly5ucHovLmNzdiBmaWxlcyBmb3VuZCBpbiB7cGF0aH0iKQoKICAgIGFsbF9yb3dzLCBzdW1tYXJ5ID0gW10sIFtdCiAgICBmb3IgZiBpbiBmaWxlczoKICAgICAgICBzaWduYWwgPSBsb2FkX3NpZ25hbF9maWxlKGYsIGFyZ3Muc2lnbmFsX2tleSkKICAgICAgICBwZXJfd2luZG93ID0gc2NvcmVfc2lnbmFsKHNpZ25hbCwgYXJncy5mcywgY29uZmlnLCBtb2RlbCwgbm9ybWFsaXplciwgZGV2aWNlKQogICAgICAgIHBlcl93aW5kb3cuaW5zZXJ0KDAsICJzb3VyY2VfZmlsZSIsIGYubmFtZSkKICAgICAgICBhbGxfcm93cy5hcHBlbmQocGVyX3dpbmRvdykKICAgICAgICBmcmFjID0gZmxvYXQocGVyX3dpbmRvdy5mbGFnZ2VkX2ZhdWx0eS5tZWFuKCkpCiAgICAgICAgc3VtbWFyeS5hcHBlbmQoeyJzb3VyY2VfZmlsZSI6IGYubmFtZSwgIm5fd2luZG93cyI6IGxlbihwZXJfd2luZG93KSwKICAgICAgICAgICAgICAgICAgICAgICAgImZsYWdnZWRfZnJhY3Rpb24iOiBmcmFjLCAidmVyZGljdCI6ICJGQVVMVFkiIGlmIGZyYWMgPiBhcmdzLnRocmVzaG9sZCBlbHNlICJoZWFsdGh5In0pCgogICAgc3VtbWFyeV9kZiA9IHBkLkRhdGFGcmFtZShzdW1tYXJ5KQogICAgcHJpbnQoZiJcblNjb3JlZCB7bGVuKGZpbGVzKX0gcmVjb3JkaW5nKHMpIGF0IHthcmdzLmZzfUh6IC0+IHtjb25maWcuY29tbW9uLmZzX2h6fUh6LCAiCiAgICAgICAgICBmIntjb25maWcuY29tbW9uLnNlZ21lbnRfbGVuZ3RoIC8gY29uZmlnLmNvbW1vbi5mc19oejouMmZ9cyB3aW5kb3dzXG4iKQogICAgcHJpbnQoc3VtbWFyeV9kZi50b19zdHJpbmcoaW5kZXg9RmFsc2UpKQoKICAgIGlmIGFyZ3MuY2FsaWJyYXRlOgogICAgICAgIGZhbHNlX2FsYXJtID0gZmxvYXQoKHN1bW1hcnlfZGYuZmxhZ2dlZF9mcmFjdGlvbiA+IGFyZ3MudGhyZXNob2xkKS5tZWFuKCkpCiAgICAgICAgbWVhbl9mcmFjID0gZmxvYXQoc3VtbWFyeV9kZi5mbGFnZ2VkX2ZyYWN0aW9uLm1lYW4oKSkKICAgICAgICBwcmludCgiXG4tLS0gQ0FMSUJSQVRJT04gKGlucHV0cyBhc3N1bWVkIGhlYWx0aHkpIC0tLSIpCiAgICAgICAgcHJpbnQoZiIgIG1lYW4gZmxhZ2dlZCBmcmFjdGlvbiBvbiBoZWFsdGh5IGRhdGE6IHttZWFuX2ZyYWM6LjElfSIpCiAgICAgICAgcHJpbnQoZiIgIHJlY29yZGluZ3MgdGhhdCB3b3VsZCBmYWxzZS1hbGFybTogICAgIHtmYWxzZV9hbGFybTouMSV9IikKICAgICAgICBpZiBtZWFuX2ZyYWMgPD0gMC4xMDoKICAgICAgICAgICAgcHJpbnQoIiAgVkVSRElDVDogdGhpcyBtYWNoaW5lIHJlYWRzIGNsZWFuLiBNb25pdG9yIHdpdGggdGhlIGRlZmF1bHQgdGhyZXNob2xkLiIpCiAgICAgICAgZWxpZiBtZWFuX2ZyYWMgPj0gMC41MDoKICAgICAgICAgICAgcHJpbnQoIiAgVkVSRElDVDogdGhpcyBtYWNoaW5lJ3MgSEVBTFRIWSBiYXNlbGluZSBhbHJlYWR5IHJlYWRzIGFzIGZhdWx0eS5cbiIKICAgICAgICAgICAgICAgICAgIiAgVGhpcyBpcyB0aGUgSzAwMiBmYWlsdXJlIG1vZGUgKHNlZSBtb2R1bGUgZG9jc3RyaW5nKTogYSBwZXItdW5pdCBvZmZzZXQsXG4iCiAgICAgICAgICAgICAgICAgICIgIG5vdCBub2lzZS4gRG8gTk9UIGRlcGxveSBhcy1pcyAtLSB0aGUgZGV0ZWN0b3Igd291bGQgYWxhcm0gY29uc3RhbnRseS5cbiIKICAgICAgICAgICAgICAgICAgZiIgIEVpdGhlciByYWlzZSAtLXRocmVzaG9sZCBhYm92ZSB7bWVhbl9mcmFjOi4yZn0sIG9yIChiZXR0ZXIpIHJldHJhaW4gd2l0aFxuIgogICAgICAgICAgICAgICAgICAiICByZWNvcmRpbmdzIGZyb20gdGhpcyBtYWNoaW5lIGFkZGVkIHRvIHRoZSBzb3VyY2UgcG9vbDogYnJlYWR0aCB3YXMgdGhlXG4iCiAgICAgICAgICAgICAgICAgICIgIHNpbmdsZSBtb3N0IGVmZmVjdGl2ZSBsZXZlciBmb3VuZCBpbiB0aGlzIHN0dWR5LiIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJpbnQoZiIgIFZFUkRJQ1Q6IGJvcmRlcmxpbmUuIFNldCAtLXRocmVzaG9sZCBjb21mb3J0YWJseSBhYm92ZSB7bWVhbl9mcmFjOi4yZn1cbiIKICAgICAgICAgICAgICAgICAgIiAgYW5kIGNvbmZpcm0gYWdhaW5zdCBhIHNlY29uZCBrbm93bi1oZWFsdGh5IHJlY29yZGluZyBiZWZvcmUgdHJ1c3RpbmcgaXQuIikKICAgIGVsc2U6CiAgICAgICAgbl9mYXVsdHkgPSBpbnQoKHN1bW1hcnlfZGYudmVyZGljdCA9PSAiRkFVTFRZIikuc3VtKCkpCiAgICAgICAgcHJpbnQoZiJcbntuX2ZhdWx0eX0gb2Yge2xlbihzdW1tYXJ5X2RmKX0gcmVjb3JkaW5nKHMpIGZsYWdnZWQgYXMgRkFVTFRZICIKICAgICAgICAgICAgICBmIih0aHJlc2hvbGQ6ID57YXJncy50aHJlc2hvbGQ6LjAlfSBvZiB3aW5kb3dzKS4iKQogICAgICAgIHByaW50KCJDYWxpYnJhdGUgb24ga25vd24taGVhbHRoeSBkYXRhIGZyb20gVEhJUyBtYWNoaW5lIGZpcnN0IGlmIHlvdSBoYXZlIG5vdDogIgogICAgICAgICAgICAgICJweXRob24gLW0gZG9tYWluX2FkYXB0YXRpb24uaW5mZXIgLS1pbnB1dCA8aGVhbHRoeT4gLS1mcyA8cmF0ZT4gLS1jYWxpYnJhdGUiKQoKICAgIGlmIGFyZ3Mub3V0OgogICAgICAgIG91dCA9IFBhdGgoYXJncy5vdXQpCiAgICAgICAgcGQuY29uY2F0KGFsbF9yb3dzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkudG9fY3N2KG91dCwgaW5kZXg9RmFsc2UpCiAgICAgICAgc3VtbWFyeV9kZi50b19jc3Yob3V0LndpdGhfbmFtZShvdXQuc3RlbSArICJfc3VtbWFyeS5jc3YiKSwgaW5kZXg9RmFsc2UpCiAgICAgICAgcHJpbnQoZiJcbldyb3RlIHBlci13aW5kb3cgcHJlZGljdGlvbnMgdG8ge291dH0iKQogICAgcHJpbnQoKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'
    ),
    'src/domain_adaptation/viz.py': (
        'IiIiTWF0cGxvdGxpYiBwbG90dGluZyBoZWxwZXJzIGZvciBldmFsdWF0ZS5weS4gU2FtZSBjb252ZW50aW9ucyBhcwp2aWJyYXRpb25fYWQudml6IC8gaHlkcmF1bGljX2Z1c2lvbl9hZC52aXo6IGV2ZXJ5IGZ1bmN0aW9uIHNhdmVzIGEgUE5HIGFuZApyZXR1cm5zIHRoZSBwYXRoLCBoZWFkbGVzcyAoQWdnIGJhY2tlbmQpLCBubyBwbHQuc2hvdygpLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbWF0cGxvdGxpYgoKbWF0cGxvdGxpYi51c2UoIkFnZyIpCmltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKaW1wb3J0IG51bXB5IGFzIG5wCgpDTEFTU19OQU1FUyA9IFsiaGVhbHRoeSIsICJpbm5lcl9yYWNlIiwgIm91dGVyX3JhY2UiXQojIFNhbWUgYmx1ZS9yZWQgc2VtYW50aWMgcGFpciB2YWxpZGF0ZWQgKGRhdGF2aXotc2tpbGwgcGFsZXR0ZSBjaGVjaykgYW5kCiMgcmV1c2VkIGFjcm9zcyBldmVyeSBleHBlcmltZW50IGluIHRoaXMgcmVwbyBmb3IgaGVhbHRoeS9mYXVsdHk7IG91dGVyX3JhY2UKIyBnZXRzIGEgdGhpcmQsIGRpc3Rpbmd1aXNoYWJsZSBodWUgc2luY2UgdGhpcyBpcyBhIDMtY2xhc3MgdGFzaywgbm90IGJpbmFyeS4KQ0xBU1NfQ09MT1JTID0geyJoZWFsdGh5IjogIiMzYjgyZjYiLCAiaW5uZXJfcmFjZSI6ICIjZWY0NDQ0IiwgIm91dGVyX3JhY2UiOiAiI2Y1OWUwYiJ9CkRPTUFJTl9OQU1FUyA9IHswOiAiY3dydSIsIDE6ICJwdV9hcnRpZmljaWFsIiwgMjogInB1X3RhcmdldCJ9CkRPTUFJTl9DT0xPUlMgPSB7ImN3cnUiOiAiIzNiODJmNiIsICJwdV9hcnRpZmljaWFsIjogIiM4YjVjZjYiLCAicHVfdGFyZ2V0IjogIiNlZjQ0NDQifQoKCmRlZiBwbG90X2NvbmZ1c2lvbl9tYXRyaXhfbXVsdGljbGFzcyhjbTogbnAubmRhcnJheSwgc2F2ZV9wYXRoOiBQYXRoLCB0aXRsZTogc3RyKSAtPiBQYXRoOgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg1LjUsIDUpKQogICAgaW0gPSBheC5pbXNob3coY20sIGNtYXA9IkJsdWVzIikKICAgIGZvciBpIGluIHJhbmdlKGNtLnNoYXBlWzBdKToKICAgICAgICBmb3IgaiBpbiByYW5nZShjbS5zaGFwZVsxXSk6CiAgICAgICAgICAgIGF4LnRleHQoCiAgICAgICAgICAgICAgICBqLCBpLCBzdHIoY21baSwgal0pLCBoYT0iY2VudGVyIiwgdmE9ImNlbnRlciIsIGZvbnRzaXplPTEzLAogICAgICAgICAgICAgICAgY29sb3I9IndoaXRlIiBpZiBjbVtpLCBqXSA+IGNtLm1heCgpIC8gMiBlbHNlICJibGFjayIsCiAgICAgICAgICAgICkKICAgIGF4LnNldF94dGlja3MocmFuZ2UobGVuKENMQVNTX05BTUVTKSkpCiAgICBheC5zZXRfeHRpY2tsYWJlbHMoW2YicHJlZDoge259IiBmb3IgbiBpbiBDTEFTU19OQU1FU10sIHJvdGF0aW9uPTMwLCBoYT0icmlnaHQiKQogICAgYXguc2V0X3l0aWNrcyhyYW5nZShsZW4oQ0xBU1NfTkFNRVMpKSkKICAgIGF4LnNldF95dGlja2xhYmVscyhbZiJ0cnVlOiB7bn0iIGZvciBuIGluIENMQVNTX05BTUVTXSkKICAgIGF4LnNldF90aXRsZSh0aXRsZSkKICAgIGZpZy5jb2xvcmJhcihpbSwgYXg9YXgsIGZyYWN0aW9uPTAuMDQ2LCBwYWQ9MC4wNCkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZmlnLnNhdmVmaWcoc2F2ZV9wYXRoLCBkcGk9MTUwKQogICAgcGx0LmNsb3NlKGZpZykKICAgIHJldHVybiBzYXZlX3BhdGgKCgpkZWYgcGxvdF9iYXNlbGluZV92c19kYW5uX2JhcnMobWV0cmljc19iYXNlbGluZTogZGljdCwgbWV0cmljc19kYW5uOiBkaWN0IHwgTm9uZSwgc2F2ZV9wYXRoOiBQYXRoKSAtPiBQYXRoOgogICAgIiIiU2lkZS1ieS1zaWRlIGFjY3VyYWN5L21hY3JvLUYxIG9uIHRoZSBUQVJHRVRfVEVTVCB6ZXJvLXNob3Qgc2V0IC0tCiAgICB0aGUgaGVhZGxpbmUgImRpZCBhZGFwdGF0aW9uIGhlbHAiIGNvbXBhcmlzb24uIiIiCiAgICBtZXRyaWNfbmFtZXMgPSBbImFjY3VyYWN5IiwgIm1hY3JvX2YxIl0KICAgIG1vZGVscyA9IFsoImJhc2VsaW5lIChubyBhZGFwdGF0aW9uKSIsIG1ldHJpY3NfYmFzZWxpbmUsICIjOWNhM2FmIildCiAgICBpZiBtZXRyaWNzX2Rhbm4gaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWxzLmFwcGVuZCgoIkRBTk4gKGFkYXB0ZWQpIiwgbWV0cmljc19kYW5uLCAiIzNiODJmNiIpKQoKICAgIHggPSBucC5hcmFuZ2UobGVuKG1ldHJpY19uYW1lcykpCiAgICB3aWR0aCA9IDAuOCAvIGxlbihtb2RlbHMpCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KDcsIDUpKQogICAgZm9yIGksIChuYW1lLCBtZXRyaWNzLCBjb2xvcikgaW4gZW51bWVyYXRlKG1vZGVscyk6CiAgICAgICAgdmFsdWVzID0gW21ldHJpY3NbbV0gZm9yIG0gaW4gbWV0cmljX25hbWVzXQogICAgICAgIGJhcnMgPSBheC5iYXIoeCArIGkgKiB3aWR0aCwgdmFsdWVzLCB3aWR0aCwgbGFiZWw9bmFtZSwgY29sb3I9Y29sb3IpCiAgICAgICAgYXguYmFyX2xhYmVsKGJhcnMsIGZtdD0iJS4zZiIsIGZvbnRzaXplPTksIHBhZGRpbmc9MikKICAgIGF4LnNldF94dGlja3MoeCArIHdpZHRoICogKGxlbihtb2RlbHMpIC0gMSkgLyAyKQogICAgYXguc2V0X3h0aWNrbGFiZWxzKFsiYWNjdXJhY3kiLCAibWFjcm8gRjEiXSkKICAgIGF4LnNldF95bGltKDAsIDEuMDgpCiAgICBheC5zZXRfeWxhYmVsKCJTY29yZSBvbiBUQVJHRVRfVEVTVCAoemVyby1zaG90LCBoZWxkLW91dCBjb25kaXRpb24pIikKICAgIGF4LnNldF90aXRsZSgiRG9tYWluLXNoaWZ0IHBlcmZvcm1hbmNlOiBiZWZvcmUgdnMuIGFmdGVyIGFkYXB0YXRpb24iKQogICAgYXgubGVnZW5kKCkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZmlnLnNhdmVmaWcoc2F2ZV9wYXRoLCBkcGk9MTUwKQogICAgcGx0LmNsb3NlKGZpZykKICAgIHJldHVybiBzYXZlX3BhdGgKCgpkZWYgcGxvdF9mZWF0dXJlX3NwYWNlKAogICAgY29vcmRzOiBucC5uZGFycmF5LCBkb21haW5faWRzOiBucC5uZGFycmF5LCBsYWJlbHM6IG5wLm5kYXJyYXksIHNhdmVfcGF0aDogUGF0aCwgbWV0aG9kX25hbWU6IHN0ciA9ICJ0LVNORSIKKSAtPiBQYXRoOgogICAgIiIiVHdvLXBhbmVsIDJEIHByb2plY3Rpb24gb2YgZmVhdHVyZS1leHRyYWN0b3IgZW1iZWRkaW5nczogbGVmdAogICAgY29sb3JlZCBieSBkb21haW4gKGRvZXMgQ1dSVSAvIFBVX0FydGlmaWNpYWwgLyBQVV9UYXJnZXQgb3ZlcmxhcD8pLAogICAgcmlnaHQgY29sb3JlZCBieSBmYXVsdCBjbGFzcyAoZG8gdGFyZ2V0IHJlYWwgZmF1bHRzIGZhbGwgaW5zaWRlIHRoZQogICAgc291cmNlIGZhdWx0IGNsdXN0ZXJzPykgLS0gdGhlIHR3byBoeXBvdGhlc2VzIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQKICAgIFBoYXNlIDQgYXNrcyB0byBjaGVjayB2aXN1YWxseS4KICAgICIiIgogICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEzLCA2KSkKCiAgICBheCA9IGF4ZXNbMF0KICAgIGZvciBkb21haW5faWQsIG5hbWUgaW4gRE9NQUlOX05BTUVTLml0ZW1zKCk6CiAgICAgICAgbWFzayA9IGRvbWFpbl9pZHMgPT0gZG9tYWluX2lkCiAgICAgICAgaWYgbWFzay5hbnkoKToKICAgICAgICAgICAgYXguc2NhdHRlcihjb29yZHNbbWFzaywgMF0sIGNvb3Jkc1ttYXNrLCAxXSwgcz04LCBhbHBoYT0wLjUsIGNvbG9yPURPTUFJTl9DT0xPUlNbbmFtZV0sIGxhYmVsPWYie25hbWV9IChuPXttYXNrLnN1bSgpfSkiKQogICAgYXguc2V0X3RpdGxlKGYie21ldGhvZF9uYW1lfTogY29sb3JlZCBieSBkb21haW4iKQogICAgYXgubGVnZW5kKGZvbnRzaXplPTgpCiAgICBheC5zZXRfeHRpY2tzKFtdKQogICAgYXguc2V0X3l0aWNrcyhbXSkKCiAgICBheCA9IGF4ZXNbMV0KICAgIGZvciBsYWJlbF9pZCwgbmFtZSBpbiBlbnVtZXJhdGUoQ0xBU1NfTkFNRVMpOgogICAgICAgIG1hc2sgPSBsYWJlbHMgPT0gbGFiZWxfaWQKICAgICAgICBpZiBtYXNrLmFueSgpOgogICAgICAgICAgICBheC5zY2F0dGVyKGNvb3Jkc1ttYXNrLCAwXSwgY29vcmRzW21hc2ssIDFdLCBzPTgsIGFscGhhPTAuNSwgY29sb3I9Q0xBU1NfQ09MT1JTW25hbWVdLCBsYWJlbD1mIntuYW1lfSAobj17bWFzay5zdW0oKX0pIikKICAgIGF4LnNldF90aXRsZShmInttZXRob2RfbmFtZX06IGNvbG9yZWQgYnkgZmF1bHQgY2xhc3MiKQogICAgYXgubGVnZW5kKGZvbnRzaXplPTgpCiAgICBheC5zZXRfeHRpY2tzKFtdKQogICAgYXguc2V0X3l0aWNrcyhbXSkKCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIGZpZy5zYXZlZmlnKHNhdmVfcGF0aCwgZHBpPTE1MCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gc2F2ZV9wYXRoCg=='
    ),
}
for relpath, b64 in SRC_FILES.items():
    p = WORKDIR / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print(f'wrote {len(SRC_FILES)} source files under {WORKDIR}/src')

In [ ]:
CONFIG_FILES = {
    'configs/domain_adaptation.yaml': (
        'IyBDb25maWcgZm9yIHRoZSBkb21haW4tYWRhcHRhdGlvbiAvIHplcm8tc2hvdCBmYXVsdCBkaWFnbm9zaXMgZXhwZXJpbWVudAojIChzcmMvZG9tYWluX2FkYXB0YXRpb24vKS4gU2VlIERPTUFJTl9BREFQVEFUSU9OX1BMQU4ubWQgZm9yIHRoZSBmdWxsIHBsYW4KIyBhbmQgaXRzICJSZXZpc2lvbnMgJiBEZWNpc2lvbnMiIHNlY3Rpb24gZm9yIHdoeSB0aGVzZSBzcGVjaWZpYyBudW1iZXJzLgoKc2VlZDogNDIKCnBhdGhzOgogIGN3cnVfcmF3X2RpcjogZGF0YS9yYXcgICAgICAgICAgICAgICAgICAgICAgICAjIHJldXNlcyB0aGUgZXhpc3RpbmcgQ1dSVSBucHogZmlsZXMsIHVudG91Y2hlZAogIHByb2Nlc3NlZF9kaXI6IGRhdGEvZGFfcHJvY2Vzc2VkCiAgbW9kZWxzX2Rpcjogb3V0cHV0cy9kYV9tb2RlbHMKICBmaWd1cmVzX2Rpcjogb3V0cHV0cy9kYV9maWd1cmVzCiAgcmVwb3J0c19kaXI6IG91dHB1dHMvZGFfcmVwb3J0cwogICMgTG9jYWwtZGV2LW9ubHk6IHBvaW50IHRoaXMgYXQgYSBzbWFsbCBmb2xkZXIgbGFpZCBvdXQgbGlrZSB0aGUgcmVhbCBQVQogICMgZGF0YXNldCAoYmVhcmluZy1jb2RlIHN1YmZvbGRlcnMgb2YgLm1hdCBmaWxlcykgaWYgeW91IGV2ZXIgd2FudCB0byB0ZXN0CiAgIyBhZ2FpbnN0IHJlYWwgZmlsZXMgd2l0aG91dCBLYWdnbGUuIE5vdCByZXF1aXJlZCAtLSBzZWUKICAjIGRhdGEvcHVfbG9hZGVyLmZpbmRfcHVfcm9vdCgpIGFuZCB0ZXN0cy90ZXN0X2RvbWFpbl9hZGFwdGF0aW9uX3Ntb2tlLnB5LAogICMgd2hpY2ggdXNlIHN5bnRoZXRpYyAubWF0IGZpbGVzIGluc3RlYWQgc28gbm8gZG93bmxvYWQgaXMgbmVlZGVkIHRvIHJ1bgogICMgdGhlIHRlc3Qgc3VpdGUuCiAgcHVfcm9vdF9sb2NhbDogZGF0YS9wdV9kZXZfc2FtcGxlCgojIC0tLSBDb21tb24gY3Jvc3MtZG9tYWluIGFsaWdubWVudCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIEV2ZXJ5IHNpZ25hbCAoQ1dSVSBhbHJlYWR5IG5hdGl2ZSwgUFUgcmVzYW1wbGVkIGRvd24gZnJvbSA2NGtIeikgaXMKIyBicm91Z2h0IHRvIHRoaXMgb25lIHJhdGUgYmVmb3JlIHdpbmRvd2luZy9TVEZULCBzbyAic3BlY3Ryb2dyYW0gYmluIDQwIgojIG1lYW5zIHRoZSBzYW1lIHBoeXNpY2FsIGZyZXF1ZW5jeSBpbiBldmVyeSBkb21haW4uIHNlZ21lbnRfbGVuZ3RoL1NURlQKIyBwYXJhbXMgYmVsb3cgYXJlIElERU5USUNBTCB0byB2aWJyYXRpb25fYWQvY29uZmlncy9kZWZhdWx0LnlhbWwgb24KIyBwdXJwb3NlIC0tIHNhbWUgMTI4eDEyOCBvdXRwdXQgc2hhcGUsIHNvIHZpYnJhdGlvbl9hZCdzIG93bgojIHdpbmRvd2luZy9zcGVjdHJvZ3JhbSBjb2RlIGlzIGltcG9ydGVkIGRpcmVjdGx5LCBub3QgcmVpbXBsZW1lbnRlZC4KY29tbW9uOgogIGZzX2h6OiAxMjAwMAogIHNlZ21lbnRfbGVuZ3RoOiA4Mzg0CiAgdHJhaW5fc3RyaWRlOiA1MTIKICBldmFsX3N0cmlkZTogMjA5NgoKc3RmdDoKICBucGVyc2VnOiAyNTYKICBob3BfbGVuZ3RoOiA2NAogIGxvZ19lcHNpbG9uOiAxLjBlLTEwCgojIC0tLSBDV1JVIChTb3VyY2UgQSkgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KY3dydToKICBzZW5zb3I6IERFCiAgdHJhaW5fcnBtczogWzE3MzAsIDE3NTAsIDE3NzJdCiAgdmFsX3JwbTogMTc5NwogICMgdjQgKGtlcm5lbCB2MyAtPiB2NCk6IENXUlUgZHJvcHBlZCBmcm9tIHRoZSBzb3VyY2UgcG9vbCBlbnRpcmVseS4KICAjIHYxLXYzIGFsbCB0cmFpbmVkIG9uIENXUlUgKyBQVV9hcnRpZmljaWFsIGFuZCBub25lIG9mIHRoZW0gYWNoaWV2ZWQKICAjIGFsaWdubWVudC4gVGhlIGh5cG90aGVzaXMgYmVpbmcgdGVzdGVkOiBmb3JjaW5nIHRoZSBmZWF0dXJlIGV4dHJhY3RvciB0bwogICMgYmUgaW52YXJpYW50IGFjcm9zcyB0d28gRU5USVJFTFkgZGlmZmVyZW50IHJpZ3MgbWF5IGJlIGRlc3Ryb3lpbmcgdGhlCiAgIyB2ZXJ5IHN0cnVjdHVyZSB0aGUgY2xhc3NpZmllciBuZWVkcywgbWFraW5nIENXUlUgYWN0aXZlbHkgaGFybWZ1bCByYXRoZXIKICAjIHRoYW4gZXh0cmEgZGF0YS4gV2l0aCB0aGlzIGZhbHNlLCB0aGUgb25seSBzaGlmdCBsZWZ0IGJldHdlZW4gc291cmNlIGFuZAogICMgdGFyZ2V0IGlzIFBVJ3Mgb3duIE4xNSAtPiBOMDkgc3BlZWQgY2hhbmdlIChwbHVzIGFydGlmaWNpYWwgLT4gcmVhbAogICMgZGFtYWdlKSAtLSBhIGZhciBzbWFsbGVyIGdhcCwgb24gb25lIHJpZyB3aXRoIG9uZSBzZW5zb3IuCiAgIyB2NCBSRVNVTFQ6IGh5cG90aGVzaXMgcmVmdXRlZCAtLSBDV1JVIHdhcyBIRUxQSU5HLCBub3QgaHVydGluZy4gRHJvcHBpbmcKICAjIGl0IGNvc3QgfjExIGFjY3VyYWN5IHBvaW50cyBvbiB0aGUgMy1jbGFzcyB0YXNrICgwLjI4MiAtPiAwLjE3MSBmb3IgdGhlCiAgIyBiYXNlbGluZSkgYW5kIHdyZWNrZWQgYmluYXJ5IGRldGVjdGlvbiAoREFOTiBGMSAwLjY2NiAtPiAwLjM3MSwgZmFsc2UKICAjIGFsYXJtcyAyLjYlIC0+IDM1LjMlKS4gU291cmNlIERJVkVSU0lUWSBtYXR0ZXJzIG1vcmUgdGhhbiBzb3VyY2UgcHVyaXR5OgogICMgYSBuYXJyb3dlciwgc2luZ2xlLXJpZyBzb3VyY2Ugb3ZlcmZpdHMgYW5kIGdlbmVyYWxpemVzIHdvcnNlLCBldmVuCiAgIyB0aG91Z2ggaXQgaXMgImNsb3NlciIgdG8gdGhlIHRhcmdldC4gUmVzdG9yZWQgdG8gdHJ1ZS4KICBpbmNsdWRlX2luX3NvdXJjZTogdHJ1ZQoKIyAtLS0gUGFkZXJib3JuIC8gS0F0LURhdGFDZW50ZXIgKFNvdXJjZSBCICsgVGFyZ2V0KSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgQmVhcmluZy1jb2RlIG1lYW5pbmdzOiBzcmMvZG9tYWluX2FkYXB0YXRpb24vZGF0YS9wdV9iZWFyaW5nX2NvZGVzLnB5LgojIEtlcHQgZGVsaWJlcmF0ZWx5IHNtYWxsIGZvciBhIGZpcnN0IEthZ2dsZSBydW4gLS0gd2lkZW4gdGhlc2UgbGlzdHMgdG8gdXNlCiMgbW9yZSBvZiB0aGUgZGF0YXNldCBvbmNlIHRoZSBwaXBlbGluZSBpcyBwcm92ZW47IG5vdGhpbmcgZG93bnN0cmVhbSBpcwojIGhhcmRjb2RlZCB0byB0aGlzIGV4YWN0IHN1YnNldC4KcHU6CiAgZGF0YXNldF9zbHVnOiBkaXBwYXRlbDAzL3BhZGVyYm9ybi1kYiAgICAgIyBhdHRhY2hlZCB2aWEgbm90ZWJvb2tzL2RvbWFpbl9hZGFwdGF0aW9uL2tlcm5lbC1tZXRhZGF0YS5qc29uCiAgbmF0aXZlX2ZzX2h6OiA2NDAwMAogIHZpYnJhdGlvbl9jaGFubmVsOiB2aWJyYXRpb25fMQogICMgdjUgKGtlcm5lbCB2NCAtPiB2NSk6IHVzZSBFVkVSWSBhdmFpbGFibGUgc291cmNlIGJlYXJpbmcgY29kZSwgNyAtPiAxOC4KICAjIHYxLXY0IHJhbiBvbiBhIGRlbGliZXJhdGVseSB0cmltbWVkIHN1YnNldCBjaG9zZW4gdG8ga2VlcCB0aGUgZmlyc3QgcnVuCiAgIyBjaGVhcC4gdjQgdGhlbiBzaG93ZWQgc291cmNlIGRpdmVyc2l0eSBpcyB0aGUgbGV2ZXIgdGhhdCBhY3R1YWxseSBtb3ZlcwogICMgdGFyZ2V0IHBlcmZvcm1hbmNlIChyZW1vdmluZyBDV1JVIGNvc3QgfjExIGFjY3VyYWN5IHBvaW50cyksIHNvIHNwZW5kaW5nCiAgIyBjb21wdXRlIG9uIGJyZWFkdGggaXMgdGhlIGJlc3QtZXZpZGVuY2VkIG1vdmUgYXZhaWxhYmxlLiAyLjA5eCB0aGUKICAjIHNvdXJjZSB3aW5kb3dzLCB+MTAzIG1pbiBwcm9qZWN0ZWQgdnMgdjIncyB+NTAuCiAgaGVhbHRoeV9jb2RlczogW0swMDEsIEswMDIsIEswMDMsIEswMDQsIEswMDUsIEswMDZdCiAgYXJ0aWZpY2lhbF9vdXRlcl9jb2RlczogW0tBMDEsIEtBMDMsIEtBMDUsIEtBMDYsIEtBMDcsIEtBMDgsIEtBMDldCiAgYXJ0aWZpY2lhbF9pbm5lcl9jb2RlczogW0tJMDEsIEtJMDMsIEtJMDUsIEtJMDcsIEtJMDhdCiAgIyBBTEwgcmVhbC1kYW1hZ2UgY29kZXMgYXJlIHVzZWQgZm9yIHRoZSB0YXJnZXQgdGVzdCBzZXQgLS0gdGhhdCdzIHRoZQogICMgYWN0dWFsIHJlc3VsdCBiZWluZyBtZWFzdXJlZCwgbm90IGEgcGxhY2UgdG8gdHJpbSBmb3Igc3BlZWQuCiAgcmVhbF9vdXRlcl9jb2RlczogW0tBMDQsIEtBMTUsIEtBMTYsIEtBMjIsIEtBMzBdCiAgcmVhbF9pbm5lcl9jb2RlczogW0tJMDQsIEtJMTQsIEtJMTYsIEtJMTcsIEtJMTgsIEtJMjFdCiAgIyBUaGUgb3RoZXIgMyBjb25kaXRpb25zIHZhcnkgbG9hZCB0b3JxdWUgLyByYWRpYWwgZm9yY2UgYXQgdGhlIFNBTUUKICAjIDE1MDBycG0gKE4xNSkgc3BlZWQ7IE4wOSAoOTAwcnBtKSBpcyBoZWxkIG91dCBhcyB0aGUgdGFyZ2V0IGJlY2F1c2UKICAjIHNwZWVkIHNoaWZ0cyBldmVyeSBiZWFyaW5nIGNoYXJhY3RlcmlzdGljIGZyZXF1ZW5jeSBtdWx0aXBsaWNhdGl2ZWx5IC0tCiAgIyB0aGUgbW9zdCBwaHlzaWNhbGx5IGRpc3RpbmN0IGRvbWFpbiBnYXAgYXZhaWxhYmxlIGFtb25nIHRoZSA0LgogIHNvdXJjZV9jb25kaXRpb25zOiBbTjE1X00wN19GMTAsIE4xNV9NMDFfRjEwLCBOMTVfTTA3X0YwNF0KICB0YXJnZXRfY29uZGl0aW9uOiBOMDlfTTA3X0YxMAogIHNvdXJjZV90cmFpbl9mcmFjdGlvbjogMC43CiAgdGFyZ2V0X25vcm1hbF9hZGFwdF9mcmFjdGlvbjogMC42CgojIC0tLSBNb2RlbCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU21hbGwgY3VzdG9tIENOTiAoc2FtZSA0LWxheWVyIHN0cmlkZS0yIGNvbnYgc2hhcGUgYXMgdmlicmF0aW9uX2FkJ3MKIyBhdXRvZW5jb2RlciBlbmNvZGVyKSwgbm90IGxpdGVyYWxseSBSZXNOZXQtMTggLS0gc2VlIHBsYW4gcmV2aXNpb25zIGZvciB3aHkuCm1vZGVsOgogIGJhc2VfY2hhbm5lbHM6IDE2CiAgZmVhdHVyZV9kaW06IDI1NgogIGNsYXNzaWZpZXJfaGlkZGVuOiAxMjgKICBkb21haW5faGlkZGVuOiAxMjgKICBkcm9wb3V0OiAwLjMKCiMgLS0tIFBoYXNlIDI6IHN1cGVydmlzZWQgc291cmNlLW9ubHkgYmFzZWxpbmUgKG5vIGFkYXB0YXRpb24pIC0tLS0tLS0tLS0tLS0tCnRyYWluX2Jhc2VsaW5lOgogIGJhdGNoX3NpemU6IDY0CiAgZXBvY2hzOiAzMAogIGxlYXJuaW5nX3JhdGU6IDEuMGUtMwogIHdlaWdodF9kZWNheTogMS4wZS01CiAgZWFybHlfc3RvcHBpbmdfcGF0aWVuY2U6IDYKICBkZXZpY2U6IGF1dG8KCiMgLS0tIFBoYXNlIDM6IERBTk4gKGFkZHMgZG9tYWluIGRpc2NyaW1pbmF0b3IgKyBHUkwgKyBoZWFsdGh5LWFuY2hvciBsb3NzKSAtCnRyYWluX2Rhbm46CiAgYmF0Y2hfc2l6ZTogNjQKICBlcG9jaHM6IDMwCiAgbGVhcm5pbmdfcmF0ZTogMS4wZS0zCiAgd2VpZ2h0X2RlY2F5OiAxLjBlLTUKICBlYXJseV9zdG9wcGluZ19wYXRpZW5jZTogNgogICMgdjIgKGtlcm5lbCB2MSAtPiB2Mik6IGFkdmVyc2FyaWFsIHByZXNzdXJlIHJhaXNlZCBhbmQgYXBwbGllZCBFQVJMSUVSLgogICMgdjEncyBkb21haW4gYWxpZ25tZW50IHNpbXBseSBuZXZlciBoYXBwZW5lZCAtLSB0YXJnZXQgZmVhdHVyZXMgZW5kZWQgdXAKICAjIGluIHJlZ2lvbnMgb2YgZmVhdHVyZSBzcGFjZSBubyBzb3VyY2UgZGF0YSBvY2N1cGllcyAoc2VlIHRoZSBwbGFuJ3MKICAjIFN0YXR1cyBzZWN0aW9uKS4gUm9vdCBjYXVzZTogdGhlIGNsYXNzaWZpZXIgY29udmVyZ2VzIG9uIHNvdXJjZSBhbG1vc3QKICAjIGltbWVkaWF0ZWx5ICg4MSUgYWNjdXJhY3kgaW4gZXBvY2ggMSkgd2hpbGUgbGFtYmRhIHdhcyBzdGlsbCBuZWFyIGl0cwogICMgZmxvb3IsIHNvIGJ5IHRoZSB0aW1lIHRoZSBhZHZlcnNhcmlhbCBzaWduYWwgYXJyaXZlZCB0aGUgZmVhdHVyZQogICMgZXh0cmFjdG9yIGhhZCBhbHJlYWR5IGNvbW1pdHRlZCB0byBzb3VyY2Utb25seSBzaG9ydGN1dHMuCiAgZG9tYWluX2xvc3Nfd2VpZ2h0OiAzLjAgICAgIyB2MTogMS4wIC0tIGxvdWRlciBhZHZlcnNhcmlhbCBzaWduYWwgdnMuIHRoZSBjbGFzc2lmaWNhdGlvbiBncmFkaWVudAogIGFuY2hvcl9sb3NzX3dlaWdodDogMC41ICAgICMgdW5jaGFuZ2VkIHNpbmNlIHYxLCBkZWxpYmVyYXRlbHk6IGlzb2xhdGUgY2hhbmdlcwogIGdybF9nYW1tYTogMjUuMCAgICAgICAgICAgICMgdjE6IDEwLjAgLS0gbGFtYmRhIGhpdHMgfjAuNDYgYnkgZW5kIG9mIGVwb2NoIDEgaW5zdGVhZCBvZiB+MC4xNwogICMgdjMgKGtlcm5lbCB2MiAtPiB2Myk6IDEuMCAtPiA1LjAuIHYyIHByb2R1Y2VkIGV4YWN0bHkgdGhlIHNpZ25hdHVyZSB0aGlzCiAgIyBrbm9iIGV4aXN0cyBmb3I6IHRoZSBwb3N0LWhvYyBwcm9iZSBzdGlsbCBzZXBhcmF0ZWQgc291cmNlIGZyb20gdGFyZ2V0IGF0CiAgIyAwLjk2OSAoZmVhdHVyZXMgTk9UIGFsaWduZWQpLCB3aGlsZSB0aGUgaW4tdHJhaW5pbmcgZGlzY3JpbWluYXRvciBzY29yZWQKICAjIDAuNDg3LTAuNTI0IGJhbGFuY2VkLWVxdWl2YWxlbnQgLS0gaS5lLiB0aGUgbWFqb3JpdHktY2xhc3MgcmF0ZSwgc2luY2UKICAjIHB1X3RhcmdldCBpcyA1MCUgb2YgZXZlcnkgYmF0Y2guIFRoZSBmZWF0dXJlcyB3ZXJlIHNlcGFyYWJsZSBhbmQgdGhlCiAgIyBkaXNjcmltaW5hdG9yIHNpbXBseSBjb3VsZCBub3QgbGVhcm4gdG8gZXhwbG9pdCB0aGVtOiBpdCB3YXMgYmVpbmcKICAjIHNjcmFtYmxlZCBieSB0aGUgcmV2ZXJzYWwgZmFzdGVyIHRoYW4gaXQgY291bGQgbGVhcm4sIHNvIGl0IGZlZCB0aGUKICAjIGZlYXR1cmUgZXh0cmFjdG9yIG1lYW5pbmdsZXNzIGdyYWRpZW50cy4gTGV0dGluZyBpdCBsZWFybiA1eCBmYXN0ZXIgaXMKICAjIHRoZSBzdGFuZGFyZCBmaXggZm9yIGxvc2luZyB0aGUgbWluaW1heCByYWNlLgogICMgQmFjayB0byAxLjAgZm9yIHY0IC0tIHYyJ3MgdmFsdWUsIHdoaWNoIHJhbiBzdGFibHkuIHYzJ3MgNS4wIGJsZXcgdXAgYXQKICAjIGVwb2NoIDQgYW5kIGZyb3plIHRoZSBuZXR3b3JrLCBzbyBrZWVwaW5nIGl0IHdvdWxkIGNvbmZvdW5kIHRoZSBvbmUKICAjIHZhcmlhYmxlIHRoaXMgcnVuIGlzIG1lYW50IHRvIGlzb2xhdGUgKENXUlUgaW4gdnMuIG91dCkuCiAgZGlzY3JpbWluYXRvcl9scl9tdWx0aXBsaWVyOiAxLjAKICAjIFNhZmV0eSBuZXQgb25seSwgYWRkZWQgYWZ0ZXIgdjMncyBibG93LXVwOyBpbmVydCBvbiBhIGhlYWx0aHkgcnVuLgogIGdyYWRfY2xpcF9ub3JtOiA1LjAKICBkZXZpY2U6IGF1dG8KCmV2YWx1YXRpb246CiAgdHNuZV9wZXJwbGV4aXR5OiAzMAo='
    ),
}
for relpath, b64 in CONFIG_FILES.items():
    p = WORKDIR / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print(f'wrote {len(CONFIG_FILES)} config files')

## 2. Environment setup + confirm PyTorch
Same subprocess-PYTHONPATH pattern as the other two notebooks in this project -- `!python` on the PATH isn't guaranteed to be this kernel's interpreter.

In [ ]:
import sys, os, subprocess
sys.path.insert(0, str(WORKDIR / 'src'))
os.environ['PYTHONPATH'] = str(WORKDIR / 'src')
os.chdir(WORKDIR)
PY = sys.executable
import torch
print('python:', PY)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available(), '(CPU-only run, expected)')

def run(*args):
    subprocess.run([PY, *args], check=True)

## 3. Convert the CWRU dataset from the attached input (.mat -> .npz)
Identical conversion to the CWRU vibration MVP notebook -- same `esraakhaled299/cwru-data` dataset, same `{rpm}_{fault}_{size}_DE12.npz` / `{rpm}_Normal.npz` naming `vibration_ad.data.catalog` expects. See vibration_ad's `notes/07_dvc_and_kaggle.md` for why this dataset/format (not a runtime download) and how the mount path was confirmed.

In [ ]:
import scipy.io, re
import numpy as np

_candidates = list(pathlib.Path('/kaggle/input').rglob('CWRU_organized'))
if not _candidates:
    print('CWRU_organized not found under /kaggle/input. Contents:')
    for p in sorted(pathlib.Path('/kaggle/input').rglob('*')):
        print(' ', p)
    raise FileNotFoundError(
        "cwru-data dataset not found under /kaggle/input -- is "
        "'esraakhaled299/cwru-data' attached to this kernel?"
    )
MAT_ROOT = _candidates[0]
print(f'MAT_ROOT = {MAT_ROOT}')
RAW_DIR = WORKDIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOAD_TO_RPM = {0: 1797, 1: 1772, 2: 1750, 3: 1730}

BALL_IR_RE = re.compile(r'^(B|IR)(\d{3})_(\d)\.mat$')
OR_RE = re.compile(r'^OR(\d{3})@(\d+)_(\d)\.mat$')
NORMAL_RE = re.compile(r'^Normal_(\d)\.mat$')

def de_signal(mat_path):
    d = scipy.io.loadmat(str(mat_path))
    key = next(k for k in d if k.endswith('_DE_time'))
    return d[key].ravel()

n = 0
for mat_path in sorted((MAT_ROOT / '12k_DE').rglob('*.mat')):
    fname = mat_path.name
    m = BALL_IR_RE.match(fname)
    if m:
        fault, size, load = m.group(1), int(m.group(2)), int(m.group(3))
        out_name = f'{LOAD_TO_RPM[load]}_{fault}_{size}_DE12.npz'
    else:
        m = OR_RE.match(fname)
        if not m:
            print(f'skip (unrecognized): {mat_path}')
            continue
        size, pos, load = int(m.group(1)), m.group(2), int(m.group(3))
        out_name = f'{LOAD_TO_RPM[load]}_OR@{pos}_{size}_DE12.npz'
    np.savez(RAW_DIR / out_name, DE=de_signal(mat_path))
    n += 1

for mat_path in sorted((MAT_ROOT / 'Normal').rglob('*.mat')):
    m = NORMAL_RE.match(mat_path.name)
    if not m:
        print(f'skip (unrecognized): {mat_path}')
        continue
    rpm = LOAD_TO_RPM[int(m.group(1))]
    np.savez(RAW_DIR / f'{rpm}_Normal.npz', DE=de_signal(mat_path))
    n += 1

print(f'converted {n} files into {RAW_DIR}')
assert n == 64, f'expected 64 CWRU files, got {n}'

## 4. Locate the attached Paderborn (PU) dataset
No conversion needed -- read directly via `domain_adaptation.data.pu_loader`. `find_pu_root()` searches `/kaggle/input` for a `K001/` subfolder rather than assuming a fixed mount path (same reasoning as the CWRU notebook's `CWRU_organized` search) -- see DOMAIN_ADAPTATION_PLAN.md's 'Revisions' section for how this dataset's structure was verified.

In [ ]:
from domain_adaptation.data.pu_loader import find_pu_root

PU_ROOT = find_pu_root()
print(f'PU_ROOT = {PU_ROOT}')
print('Bearing-code folders found:', sorted(p.name for p in PU_ROOT.iterdir() if p.is_dir())[:10], '...')

## 5. Phase 2: supervised source-only baseline (no adaptation)
Trains on SOURCE_TRAIN (CWRU + PU_Artificial, fully labeled), early-stops on SOURCE_VAL. Fits + saves the shared normalizer.

In [ ]:
run('-m', 'domain_adaptation.train_baseline', '--config', 'configs/domain_adaptation.yaml', '--pu-root', str(PU_ROOT))

## 6. Phase 3: DANN adaptation
Adds the domain discriminator (behind a Gradient Reversal Layer) and the healthy-anchor loss, adapting on TARGET_ADAPT -- the held-out condition's HEALTHY runs only. No fault example, labeled or not, from the target domain appears anywhere in this step.

In [ ]:
run('-m', 'domain_adaptation.train_dann', '--config', 'configs/domain_adaptation.yaml', '--pu-root', str(PU_ROOT))

## 7. Phase 4: the zero-shot check
Evaluates both models on TARGET_TEST -- the held-out condition's REAL damage bearings, whose labels neither model has seen -- and compares baseline vs. DANN. Also renders a t-SNE feature-space plot.

In [ ]:
run('-m', 'domain_adaptation.evaluate', '--config', 'configs/domain_adaptation.yaml', '--pu-root', str(PU_ROOT))

## 8. Results

In [ ]:
import json
metrics = json.loads((WORKDIR / 'outputs/da_reports/evaluation_metrics.json').read_text())
for name, m in metrics.items():
    print(f"{name:10s}  accuracy={m['accuracy']:.3f}  macro_f1={m['macro_f1']:.3f}  n={m['n']}")
    print('  confusion_matrix (rows=true, cols=pred, order=[healthy, inner_race, outer_race]):')
    for row in m['confusion_matrix']:
        print('   ', row)
print()
print(json.dumps(metrics, indent=2))

## 9. Phase 5 smoke test: the deployment path on real data
`infer.py` is what a real machine would actually run -- it loads the BASELINE model (never the adapted one; see the plan's verdict), resamples whatever rate the machine records at to the model's common rate, and reports healthy-vs-faulty per recording rather than fault type. Exercised here against real CWRU files, since it cannot be run on the dev laptop (no torch there). Calibration mode is pointed at a known-HEALTHY file, then detection mode at a known-FAULTY one -- so if either verdict comes out backwards, this cell says so loudly instead of shipping an untested deployment script.

In [ ]:
print('--- calibrate on a known-HEALTHY recording ---')
run('-m', 'domain_adaptation.infer', '--config', 'configs/domain_adaptation.yaml',
    '--input', 'data/raw/1797_Normal.npz', '--signal-key', 'DE', '--fs', '12000', '--calibrate')
print()
print('--- detect on a known-FAULTY recording (inner race, 21 mil) ---')
run('-m', 'domain_adaptation.infer', '--config', 'configs/domain_adaptation.yaml',
    '--input', 'data/raw/1797_IR_21_DE12.npz', '--signal-key', 'DE', '--fs', '12000',
    '--out', 'outputs/da_reports/infer_demo.csv')

## 10. Figures

In [ ]:
from IPython.display import Image, display
figdir = WORKDIR / 'outputs' / 'da_figures'
for name in sorted(p.name for p in figdir.glob('*.png')):
    print(name)
    display(Image(filename=str(figdir / name)))